<a href="https://colab.research.google.com/github/Yash02102/AutoTest/blob/main/AIMO_3_Dedug.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

ai_mathematical_olympiad_progress_prize_3_path = kagglehub.competition_download('ai-mathematical-olympiad-progress-prize-3')
ritwikakancharla_aimo_3_benchmark_path = kagglehub.dataset_download('ritwikakancharla/aimo-3-benchmark')
kalyankkr_aimo_3_utils_path = kagglehub.dataset_download('kalyankkr/aimo-3-utils')
qwen_lm_qwen2_5_math_transformers_7b_instruct_1_path = kagglehub.model_download('qwen-lm/qwen2.5-math/Transformers/7b-instruct/1')
deepseek_ai_deepseek_r1_transformers_deepseek_r1_distill_qwen_32b_2_path = kagglehub.model_download('deepseek-ai/deepseek-r1/Transformers/deepseek-r1-distill-qwen-32b/2')
qwen_lm_qwq_32b_transformers_qwq_32b_1_path = kagglehub.model_download('qwen-lm/qwq-32b/Transformers/qwq-32b/1')
qwen_lm_qwen_3_transformers_30b_a3b_1_path = kagglehub.model_download('qwen-lm/qwen-3/Transformers/30b-a3b/1')

print('Data source import complete.')


100%|██████████| 636k/636k [00:00<00:00, 1.41MB/s]

Extracting files...


100%|██████████| 52.8k/52.8k [00:00<00:00, 2.22MB/s]

Extracting files...


 79%|███████▉  | 3.82G/4.84G [01:49<00:27, 40.1MB/s]

# AIMO3 Frontier TIR General Harness

This notebook is split into explicit setup, server, solver, and experiment cells so you can rerun only the parts you need.

Recommended order:
1. Optional Colab bootstrap
2. Runtime config and setup execution
3. Imports, prompts, and server helpers
4. Worker and solver definitions
5. Start or restart `vLLM`
6. Configure and run a single experiment or an evaluation helper


## 1. Runtime Config
Definitions for environment variables, path discovery, offline wheels, and model/input lookup.

In [ ]:
import os, re, sys, math, time, json, queue, shutil, hashlib, tempfile, subprocess
from dataclasses import dataclass
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

class CFG:
    runtime = os.getenv("AIMO3_RUNTIME", "auto")
    input_roots_env = os.getenv("AIMO3_INPUT_ROOTS", "")
    work_root_env = os.getenv("AIMO3_WORK_ROOT", "")
    competition_path_env = os.getenv("AIMO3_COMPETITION_PATH", "")
    model_path_env = os.getenv("AIMO3_MODEL_PATH", "")
    primary_model_path_env = os.getenv("AIMO3_PRIMARY_MODEL_PATH", "")
    contrarian_model_path_env = os.getenv("AIMO3_CONTRARIAN_MODEL_PATH", "")
    verifier_model_path_env = os.getenv("AIMO3_VERIFIER_MODEL_PATH", "")
    benchmark_path_env = os.getenv("AIMO3_BENCHMARK_PATH", "")
    wheels_dir_env = os.getenv("AIMO3_WHEELS_DIR", "")
    wheels_archive_env = os.getenv("AIMO3_WHEELS_ARCHIVE", "")
    tiktoken_vocab_env = os.getenv("AIMO3_TIKTOKEN_VOCAB", "")
    allow_online_install = os.getenv("AIMO3_ALLOW_ONLINE_INSTALL", "")
    cuda_visible_devices = os.getenv("AIMO3_CUDA_VISIBLE_DEVICES", "0")
    primary_served_model_name = os.getenv("AIMO3_PRIMARY_SERVED_MODEL_NAME", os.getenv("AIMO3_SERVED_MODEL_NAME", "aimo3-primary"))
    contrarian_served_model_name = os.getenv("AIMO3_CONTRARIAN_SERVED_MODEL_NAME", "aimo3-contrarian")
    verifier_served_model_name = os.getenv("AIMO3_VERIFIER_SERVED_MODEL_NAME", "aimo3-verifier")
    primary_model_hint = os.getenv("AIMO3_PRIMARY_MODEL_HINT", os.getenv("AIMO3_MODEL_HINT", "Qwen3-30B-A3B"))
    contrarian_model_hints_env = os.getenv("AIMO3_CONTRARIAN_MODEL_HINTS", "QwQ-32B,DeepSeek-R1-Distill-Qwen-32B")
    verifier_model_hint = os.getenv("AIMO3_VERIFIER_MODEL_HINT", "Qwen2.5-Math-7B-Instruct")
    primary_cuda_visible_devices = os.getenv("AIMO3_PRIMARY_CUDA_VISIBLE_DEVICES", cuda_visible_devices)
    contrarian_cuda_visible_devices = os.getenv("AIMO3_CONTRARIAN_CUDA_VISIBLE_DEVICES", "")
    verifier_cuda_visible_devices = os.getenv("AIMO3_VERIFIER_CUDA_VISIBLE_DEVICES", "")
    served_model_name = primary_served_model_name
    model_hint = primary_model_hint
    prefer_danielhanchen_base = False
    preferred_model_versions = ("9", "20")
    adapter_hint = os.getenv("AIMO3_ADAPTER_HINT", "aimo3_lora_feasibility")
    adapter_alias = os.getenv("AIMO3_ADAPTER_ALIAS", "aimo3-lora")
    enable_lora = os.getenv("AIMO3_ENABLE_LORA", "1") == "1"
    max_lora_rank = int(os.getenv("AIMO3_MAX_LORA_RANK", "64"))
    server_host = "127.0.0.1"
    server_port = 8000
    primary_server_port = int(os.getenv("AIMO3_PRIMARY_SERVER_PORT", str(server_port)))
    contrarian_server_port = int(os.getenv("AIMO3_CONTRARIAN_SERVER_PORT", str(server_port + 1)))
    verifier_server_port = int(os.getenv("AIMO3_VERIFIER_SERVER_PORT", str(server_port + 2)))
    server_timeout_seconds = int(os.getenv("AIMO3_SERVER_TIMEOUT_SECONDS", "1800"))
    request_timeout_seconds = 960
    notebook_budget_seconds = 17400
    hard_deadline_reserve_seconds = 45
    gpu_memory_utilization = 0.90
    aux_gpu_memory_utilization = float(os.getenv("AIMO3_AUX_GPU_MEMORY_UTILIZATION", "0.55"))
    verifier_gpu_memory_utilization = float(os.getenv("AIMO3_VERIFIER_GPU_MEMORY_UTILIZATION", "0.35"))
    max_model_len = int(os.getenv("AIMO3_MAX_MODEL_LEN", "65536"))
    max_num_seqs = int(os.getenv("AIMO3_MAX_NUM_SEQS", "128"))
    kv_cache_dtype = "fp8_e4m3"
    vllm_enforce_eager = os.getenv("AIMO3_VLLM_ENFORCE_EAGER", "1") == "1"
    startup_seed = 42
    attempts = 8
    workers = 8
    early_stop_votes = 5
    temperature = 1.0
    min_p = 0.02
    max_tokens = 24576
    top_logprobs = 5
    tool_rounds = 8
    python_timeout_seconds = 12
    kernel_pool_size = 8
    weak_leader_support_ratio = 0.80
    tool_supported_margin_ratio = 0.86
    benchmark_eval_limit = int(os.getenv("AIMO3_BENCHMARK_LIMIT", "20"))
    use_personas = True
    current_date = "2026-04-07"
    log_jsonl = True
    debug_text_chars = 900
    tool_confusion_weight = 0.25
    approximate_reasoning_penalty = 0.40
    random_search_penalty = 0.30
    weak_exactness_penalty = 0.72
    extremal_certificate_penalty = 0.70
    enable_aux_models = os.getenv("AIMO3_ENABLE_AUX_MODELS", "1") == "1"
    verifier_enabled = os.getenv("AIMO3_ENABLE_VERIFIER", "1") == "1"
    verifier_max_candidates = int(os.getenv("AIMO3_VERIFIER_MAX_CANDIDATES", "5"))
    verifier_candidate_score_ratio = float(os.getenv("AIMO3_VERIFIER_SCORE_RATIO", "0.40"))
    verifier_close_score_ratio = float(os.getenv("AIMO3_VERIFIER_CLOSE_RATIO", "0.70"))
    verifier_temperature = float(os.getenv("AIMO3_VERIFIER_TEMPERATURE", "0.0"))
    verifier_max_tokens = int(os.getenv("AIMO3_VERIFIER_MAX_TOKENS", "6144"))
    verifier_python_timeout_seconds = 20
    verifier_rounds = 3
    challenge_temperature = float(os.getenv("AIMO3_CHALLENGE_TEMPERATURE", "0.65"))
    challenge_confirm_weight = float(os.getenv("AIMO3_CHALLENGE_CONFIRM_WEIGHT", "0.65"))
    challenge_discovery_weight = float(os.getenv("AIMO3_CHALLENGE_DISCOVERY_WEIGHT", "2.4"))
    strong_majority_min_votes = int(os.getenv("AIMO3_STRONG_MAJORITY_MIN_VOTES", "5"))
    strong_majority_vote_margin = int(os.getenv("AIMO3_STRONG_MAJORITY_VOTE_MARGIN", "3"))
    strong_majority_score_ratio = float(os.getenv("AIMO3_STRONG_MAJORITY_SCORE_RATIO", "0.72"))
    challenge_close_ratio = float(os.getenv("AIMO3_CHALLENGE_CLOSE_RATIO", "0.82"))
    enable_exactness_penalties = os.getenv("AIMO3_ENABLE_EXACTNESS_PENALTIES", "1") == "1"
    attempt_temperatures = (0.0, 0.35, 0.55, 0.75, 0.9, 1.0, 1.1, 1.15)
    base_developer_prompt = """# Instructions
Solve the olympiad problem. The final answer is a non-negative integer in [0, 99999].
Use exact reasoning. You may call python for computation, search, enumeration, symbolic checks, or verification.
If you choose Python, emit executable Python to the python tool/recipient. Do not write that tools are unavailable, not set up, or that you cannot see output; the harness will return stdout/stderr as an observation.
When calling python, keep code small, deterministic, and assert-heavy. Use available packages: math, itertools, functools, fractions, collections, decimal, sympy, numpy.
Use Python for exact enumeration, symbolic algebra, discrete counterexamples, or recurrence checks; avoid random search, floating minimization, or approximation-only evidence.
Do not stop at an unverified pattern when a direct check or independent derivation is possible.
In the final channel, output exactly one boxed answer, for example \\boxed{42}.
"""
    personas = (
        "Use tool-integrated reasoning. Generate code when computation reduces risk, then verify independently.",
        "Use proof-first reasoning. Identify invariants, transformations, and exact reductions before brute force.",
        "Use counterexample-first reasoning. Test small cases, edge cases, and traps before committing to a formula.",
        "Use backward-check reasoning. Treat candidate answers skeptically and verify all constraints before finalizing.",
    )
    attempt_profiles = (
        "Start with an exact structural reduction before chasing numbers.",
        "Work small exact cases first, then generalize only after the pattern is justified.",
        "Use Python early for exact enumeration or symbolic algebra if it can prune the search space.",
        "Assume the obvious small answer is wrong and try to refute it before accepting anything.",
        "Look for a constructive witness or equality case that could pin down the final answer.",
        "Search for an invariant, monotonic quantity, or conserved relation that collapses the problem.",
        "Translate the problem into factorization, recurrence, or root-structure language before counting.",
        "Work backwards from the required conclusion and narrow the admissible cases exactly.",
    )


def slug(text):
    return re.sub(r"[^a-z0-9]+", "-", str(text).lower()).strip("-")

def parse_path_list(value):
    if not value:
        return []
    parts = []
    for chunk in str(value).replace("\n", os.pathsep).split(os.pathsep):
        for piece in chunk.split(","):
            piece = piece.strip()
            if piece:
                parts.append(Path(piece).expanduser())
    return parts

def detect_runtime_name():
    explicit = str(CFG.runtime or "").strip().lower()
    if explicit and explicit != "auto":
        return explicit
    if Path("/kaggle/input").exists():
        return "kaggle"
    if "COLAB_GPU" in os.environ or Path("/content").exists():
        return "colab"
    return "local"

def resolve_input_roots():
    configured = [p for p in parse_path_list(CFG.input_roots_env) if p.exists()]
    if configured:
        return configured
    if RUNTIME_NAME == "kaggle":
        return [Path("/kaggle/input")]
    if RUNTIME_NAME == "colab":
        roots = [Path("/content")]
        drive_root = Path("/content/drive/MyDrive")
        if drive_root.exists():
            roots.append(drive_root)
        return [p for p in roots if p.exists()]
    return [Path.cwd()]

def resolve_work_root():
    if CFG.work_root_env:
        root = Path(CFG.work_root_env).expanduser()
    elif RUNTIME_NAME == "kaggle":
        root = Path("/kaggle/working")
    elif RUNTIME_NAME == "colab":
        root = Path("/content/aimo3_work")
    else:
        root = Path.cwd() / "aimo3_work"
    root.mkdir(parents=True, exist_ok=True)
    return root

def allow_online_install():
    token = str(CFG.allow_online_install or "").strip().lower()
    if token:
        return token in ("1", "true", "yes", "on")
    return RUNTIME_NAME != "kaggle"

def iter_input_roots():
    return list(INPUT_ROOTS)

def scan_directories_with_marker(marker_name):
    seen = set()
    for root in iter_input_roots():
        if not root.exists():
            continue
        for path in root.rglob(marker_name):
            parent = path.parent
            key = str(parent.resolve()).lower()
            if key not in seen:
                seen.add(key)
                yield parent


def load_model_config_dict(model_path):
    model_dir = resolve_model_dir(model_path)
    if model_dir is None:
        return {}
    config_path = model_dir / "config.json"
    if not config_path.exists():
        return {}
    try:
        return json.loads(config_path.read_text(encoding="utf-8"))
    except Exception:
        return {}

def derive_model_max_length(model_path):
    config = load_model_config_dict(model_path)
    candidates = []
    for key in (
        "max_position_embeddings",
        "model_max_length",
        "max_sequence_length",
        "max_seq_len",
        "seq_length",
        "n_positions",
    ):
        value = config.get(key)
        if isinstance(value, int) and value > 0:
            candidates.append(value)
    rope_scaling = config.get("rope_scaling")
    if isinstance(rope_scaling, dict):
        for key in ("original_max_position_embeddings", "max_position_embeddings"):
            value = rope_scaling.get(key)
            if isinstance(value, int) and value > 0:
                candidates.append(value)
    return min(candidates) if candidates else None

def resolve_effective_max_model_len(model_path, requested_max_len=None):
    requested = CFG.max_model_len if requested_max_len is None else int(requested_max_len)
    derived = derive_model_max_length(model_path)
    if derived is None:
        return requested
    allow_long = str(os.getenv("VLLM_ALLOW_LONG_MAX_MODEL_LEN", "")).strip() == "1"
    if allow_long:
        return requested
    return min(requested, derived)

RUNTIME_NAME = detect_runtime_name()
INPUT_ROOTS = resolve_input_roots()
WORK_ROOT = resolve_work_root()

def find_competition_path(required=True):
    if CFG.competition_path_env:
        explicit = Path(CFG.competition_path_env).expanduser()
        if (explicit / "kaggle_evaluation" / "aimo_3_gateway.py").exists() and (explicit / "test.csv").exists():
            return explicit
        if required:
            raise FileNotFoundError(f"Configured competition path is invalid: {explicit}")
        return None
    direct = []
    for root in iter_input_roots():
        direct.extend([root / "ai-mathematical-olympiad-progress-prize-3", root / "competitions" / "ai-mathematical-olympiad-progress-prize-3"])
    for c in direct:
        if (c / "kaggle_evaluation" / "aimo_3_gateway.py").exists() and (c / "test.csv").exists(): return c
    for c in scan_directories_with_marker("test.csv"):
        if (c / "kaggle_evaluation" / "aimo_3_gateway.py").exists():
            return c
    if required:
        raise FileNotFoundError(f"AIMO3 input not found under roots: {[str(p) for p in INPUT_ROOTS]}")
    return None

def resolve_model_dir(path_like):
    if not path_like:
        return None
    candidate = Path(str(path_like).strip()).expanduser()
    if candidate.is_file():
        candidate = candidate.parent
    if (candidate / "config.json").exists():
        return candidate
    return None

def normalize_model_hint(text):
    return str(text or "").strip().lower().replace("\\", "/")

def model_hint_aliases(hint):
    raw = normalize_model_hint(hint)
    if not raw:
        return []
    dashed = re.sub(r"[^a-z0-9]+", "-", raw).strip("-")
    compact = re.sub(r"[^a-z0-9]+", "", raw)
    aliases = {
        raw,
        raw.replace("_", "-"),
        raw.replace(" ", "-"),
        dashed,
        compact,
    }
    if "qwen3" in compact and "30ba3b" in compact:
        aliases.update({"qwen-3", "30b-a3b", "qwen-3/transformers/30b-a3b"})
    if "qwen25math" in compact and "72b" in compact and "instruct" in compact:
        aliases.update({"qwen2.5-math", "72b-instruct", "qwen2.5-math/transformers/72b-instruct"})
    if "qwen25math" in compact and "7b" in compact and "instruct" in compact:
        aliases.update({"qwen2.5-math", "7b-instruct", "qwen2.5-math/transformers/7b-instruct"})
    if "qwq32b" in compact:
        aliases.update({"qwq-32b", "qwen-lm/qwq-32b", "transformers/qwq-32b"})
    if "deepseekr1distillqwen32b" in compact:
        aliases.update({"deepseek-r1-distill-qwen-32b", "deepseek-r1/transformers/deepseek-r1-distill-qwen-32b"})
    if "gptoss20b" in compact:
        aliases.update({"gpt-oss-20b", "20b"})
    if "gptoss120b" in compact:
        aliases.update({"gpt-oss-120b", "120b", "160a"})
    return sorted((a for a in aliases if a), key=len, reverse=True)

def known_model_candidates(hint):
    compact = re.sub(r"[^a-z0-9]+", "", str(hint or "").lower())
    candidates = []
    for root in iter_input_roots():
        model_root = root / "models"
        if "qwen3" in compact and "30ba3b" in compact:
            candidates.extend([
                model_root / "qwen-lm" / "qwen-3" / "transformers" / "30b-a3b" / "1",
                root / "qwen-3" / "transformers" / "30b-a3b" / "1",
            ])
        if "qwen25math" in compact and "72b" in compact and "instruct" in compact:
            candidates.extend([
                model_root / "qwen-lm" / "qwen2.5-math" / "transformers" / "72b-instruct" / "1",
                root / "qwen2.5-math" / "transformers" / "72b-instruct" / "1",
            ])
        if "qwen25math" in compact and "7b" in compact and "instruct" in compact:
            candidates.extend([
                model_root / "qwen-lm" / "qwen2.5-math" / "transformers" / "7b-instruct" / "1",
                root / "qwen2.5-math" / "transformers" / "7b-instruct" / "1",
            ])
        if "qwq32b" in compact:
            candidates.extend([
                model_root / "qwen-lm" / "qwq-32b" / "transformers" / "qwq-32b" / "1",
                root / "qwq-32b" / "transformers" / "qwq-32b" / "1",
            ])
        if "deepseekr1distillqwen32b" in compact:
            candidates.extend([
                model_root / "deepseek-ai" / "deepseek-r1" / "transformers" / "deepseek-r1-distill-qwen-32b" / "2",
                root / "deepseek-r1" / "transformers" / "deepseek-r1-distill-qwen-32b" / "2",
            ])
    return candidates

def find_model_path(hint, explicit_path_env="", explicit_env_name="AIMO3_MODEL_PATH"):
    hint_text = str(hint or "").strip()
    env_value = explicit_path_env or CFG.model_path_env
    if env_value:
        resolved = resolve_model_dir(env_value)
        if resolved is not None:
            print({"selected_model_path": str(resolved), "source": explicit_env_name if explicit_path_env else "AIMO3_MODEL_PATH"})
            return resolved
        raise FileNotFoundError(f"Configured model path is invalid: {env_value}")
    resolved_hint_path = resolve_model_dir(hint_text)
    if resolved_hint_path is not None:
        print({"selected_model_path": str(resolved_hint_path), "source": "hint_path"})
        return resolved_hint_path
    hint = normalize_model_hint(hint_text)
    explicit = list(known_model_candidates(hint))
    if "20b" in hint:
        base_candidates = []
        for root in iter_input_roots():
            base_candidates += [
                root / "models" / "danielhanchen" / "gpt-oss-20b" / "transformers" / "default" / "1",
                root / "models" / "openai" / "gpt-oss-20b" / "transformers" / "default" / "1",
                root / "models" / "shelterw" / "gpt" / "transformers" / "gpt-oss-20b" / "1",
                root / "models" / "gpt-oss-20b" / "transformers" / "default" / "1",
                root / "gpt-oss-20b" / "transformers" / "default" / "1",
            ]
    else:
        base_candidates = []
        for root in iter_input_roots():
            base_candidates += [
                root / "models" / "danielhanchen" / "gpt-oss-120b" / "transformers" / "default" / "1",
                root / "models" / "gpt-oss-120b" / "transformers" / "default" / "1",
                root / "gpt-oss-120b" / "transformers" / "default" / "1",
            ]
    if CFG.prefer_danielhanchen_base:
        explicit += base_candidates
    if "120b" in hint:
        for root in iter_input_roots():
            for v in CFG.preferred_model_versions:
                explicit += [
                    root / "models" / "huikang" / "gpt-oss-120b-aimo3" / "transformers" / "160a" / v,
                    root / "huikang" / "gpt-oss-120b-aimo3" / "transformers" / "160a" / v,
                    root / "gpt-oss-120b-aimo3" / "transformers" / "160a" / v,
                ]
    if not CFG.prefer_danielhanchen_base:
        explicit += base_candidates
    seen = set()
    for c in explicit:
        key = str(c)
        if key in seen:
            continue
        seen.add(key)
        if (c / "config.json").exists():
            print({"selected_model_path": str(c)})
            return c
    aliases = model_hint_aliases(hint)
    model_dirs = list(scan_directories_with_marker("config.json"))
    for v in CFG.preferred_model_versions:
        for c in model_dirs:
            s = str(c).lower().replace("\\", "/")
            if f"/160a/{v}" in s and any(alias in s for alias in aliases):
                print({"selected_model_path": str(c)})
                return c
    for c in model_dirs:
        s = str(c).lower().replace("\\", "/")
        if any(alias in s for alias in aliases):
            print({"selected_model_path": str(c), "warning": "using alias-matched model version"})
            return c
    for c in model_dirs:
        compact_dir = re.sub(r"[^a-z0-9]+", "", str(c).lower())
        if any(re.sub(r"[^a-z0-9]+", "", alias) in compact_dir for alias in aliases):
            print({"selected_model_path": str(c), "warning": "falling back to normalized hint-matched model"})
            return c
    for c in model_dirs:
        s = str(c).lower()
        if "gpt-oss-20b" in s and "20b" in hint:
            print({"selected_model_path": str(c), "warning": "falling back to base gpt-oss-20b"})
            return c
    for c in model_dirs:
        s = str(c).lower()
        if "gpt-oss-120b" in s and "120b" in hint:
            print({"selected_model_path": str(c), "warning": "falling back to base gpt-oss-120b"})
            return c
    raise FileNotFoundError(f"Model not found for hint {hint_text}")




def parse_hint_list(value):
    if not value:
        return []
    hints = []
    for chunk in str(value).replace("\n", ",").split(","):
        hint = str(chunk).strip()
        if hint and hint.lower() not in {"none", "null"}:
            hints.append(hint)
    return hints

def find_optional_model_path(hints, explicit_path_env="", explicit_env_name="AIMO3_MODEL_PATH"):
    if isinstance(hints, str):
        hints = [hints]
    for raw_hint in hints or []:
        hint = str(raw_hint).strip()
        if not hint:
            continue
        try:
            return find_model_path(hint, explicit_path_env=explicit_path_env, explicit_env_name=explicit_env_name)
        except FileNotFoundError:
            continue
    return None


def infer_model_family(model_path=None, hint=""):
    text = " ".join(
        x for x in (
            "" if model_path is None else str(model_path),
            "" if hint is None else str(hint),
        )
    ).lower()
    if "gpt-oss" in text or "gpt_oss" in text:
        return "gpt_oss"
    if "deepseek" in text:
        return "deepseek"
    if "qwen" in text or "qwq" in text:
        return "qwen"
    return "generic"

def find_adapter_path(hint):
    if not CFG.enable_lora:
        return None

    def maybe_normalize_adapter_dir(base_dir):
        direct = base_dir if base_dir.is_dir() else base_dir.parent
        if (direct / "adapter_config.json").exists() and any((direct / name).exists() for name in ("adapter_model.safetensors", "adapter_model.bin")):
            return direct

        candidates = [direct]
        if (direct / "adapter").is_dir():
            candidates.append(direct / "adapter")

        config_re = re.compile(r"adapter[_ -]?config.*\.json$", re.IGNORECASE)
        model_re = re.compile(r"adapter[_ -]?model.*\.(safetensors|bin)$", re.IGNORECASE)
        readme_re = re.compile(r"readme.*\.md$", re.IGNORECASE)

        for candidate_dir in candidates:
            config_file = None
            model_file = None
            readme_file = None
            for path in candidate_dir.rglob("*"):
                if not path.is_file():
                    continue
                name = path.name
                if config_file is None and config_re.fullmatch(name):
                    config_file = path
                elif model_file is None and model_re.fullmatch(name):
                    model_file = path
                elif readme_file is None and readme_re.fullmatch(name):
                    readme_file = path
            if config_file is None or model_file is None:
                continue
            normalized_root = WORK_ROOT / "aimo3_adapter_normalized" / slug(str(candidate_dir))
            if normalized_root.exists():
                shutil.rmtree(normalized_root)
            normalized_root.mkdir(parents=True, exist_ok=True)
            shutil.copy2(config_file, normalized_root / "adapter_config.json")
            target_model_name = "adapter_model.safetensors" if model_file.suffix.lower() == ".safetensors" else "adapter_model.bin"
            shutil.copy2(model_file, normalized_root / target_model_name)
            if readme_file is not None:
                shutil.copy2(readme_file, normalized_root / "README.md")
            print({"normalized_adapter_path": str(normalized_root), "config_source": str(config_file), "model_source": str(model_file)})
            return normalized_root
        return None

    explicit = []
    if hint:
        for root in iter_input_roots():
            for candidate in root.rglob("*"):
                lower = str(candidate).lower().replace("\\", "/")
                if hint.lower() in lower:
                    explicit.append(candidate if candidate.is_dir() else candidate.parent)
    for root in iter_input_roots():
        explicit += [path.parent for path in root.rglob("adapter_config.json")]
        explicit += [path.parent for path in root.rglob("adapter_model.safetensors")]
    seen = set()
    for candidate in explicit:
        candidate = candidate.resolve()
        text = str(candidate).lower()
        if text in seen:
            continue
        seen.add(text)
        resolved = maybe_normalize_adapter_dir(candidate)
        if resolved is not None:
            print({"selected_adapter_path": str(resolved)})
            return resolved
    print({"adapter_warning": f"no adapter found for hint={hint}"})
    return None

def find_wheels_archive():
    if CFG.wheels_archive_env:
        explicit = Path(CFG.wheels_archive_env).expanduser()
        if explicit.exists():
            return explicit
    for root in iter_input_roots():
        for a in root.rglob("wheels.tar.gz"): return a
    return None

def find_wheels_dir():
    if CFG.wheels_dir_env:
        explicit = Path(CFG.wheels_dir_env).expanduser()
        if explicit.is_dir():
            return explicit
    for root in iter_input_roots():
        for c in root.rglob("wheels"):
            if c.is_dir(): return c
    return None

def ensure_offline_packages():
    missing = []
    for n in ["openai", "vllm", "openai_harmony", "jupyter_client"]:
        try: __import__(n)
        except Exception: missing.append(n)
    if not missing: return
    archive, wheels_dir = find_wheels_archive(), find_wheels_dir()
    if archive is None and wheels_dir is None and not allow_online_install():
        raise RuntimeError("Missing dependency source. Provide wheels via AIMO3_WHEELS_DIR/AIMO3_WHEELS_ARCHIVE or enable online install.")
    if wheels_dir is None and archive is not None:
        setup = WORK_ROOT / "aimo3_frontier_setup"
        if setup.exists(): shutil.rmtree(setup)
        setup.mkdir(parents=True, exist_ok=True)
        subprocess.run(["tar", "-xzf", str(archive), "-C", str(setup)], check=True)
        wheels_dir = setup / "wheels"
    if wheels_dir is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--find-links", str(wheels_dir), "openai", "vllm", "openai-harmony", "jupyter-client"], check=True)
        return
    subprocess.run([sys.executable, "-m", "pip", "install", "openai>=1.58,<2", "vllm>=0.10,<1", "openai-harmony", "jupyter-client"], check=True)

def configure_harmony_vocab():
    explicit_vocab = Path(CFG.tiktoken_vocab_env).expanduser() if CFG.tiktoken_vocab_env else None
    expected_url = "https://openaipublic.blob.core.windows.net/encodings/o200k_base.tiktoken"
    cache_key = hashlib.sha1(expected_url.encode("utf-8")).hexdigest()
    cache_dir = WORK_ROOT / "tiktoken-rs-cache"
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / cache_key
    named_vocab = explicit_vocab if explicit_vocab and explicit_vocab.is_file() else None
    for root in iter_input_roots():
        if not root.exists():
            continue
        if named_vocab is None:
            for name in ("o200k_base.tiktoken", "o200k_harmony.tiktoken"):
                for c in root.rglob(name):
                    if c.is_file(): named_vocab = c; break
                if named_vocab is not None: break
        if named_vocab is not None:
            break
    hashed_vocab = None
    for root in iter_input_roots():
        if not root.exists():
            continue
        for c in root.rglob(cache_key):
            if c.is_file(): hashed_vocab = c; break
        if named_vocab is None and hashed_vocab is None:
            for c in root.rglob("*"):
                if not c.is_file(): continue
                try: size = c.stat().st_size
                except OSError: continue
                if c.suffix == ".tiktoken" and size > 3_000_000: named_vocab = c; break
                if re.fullmatch(r"[0-9a-f]{40}", c.name) and size > 3_000_000: hashed_vocab = c; break
        if named_vocab is not None or hashed_vocab is not None:
            break
    debug = {"cache_key": cache_key, "runtime": RUNTIME_NAME, "input_roots": [str(p) for p in INPUT_ROOTS]}
    if named_vocab is not None:
        os.environ["TIKTOKEN_ENCODINGS_BASE"] = str(named_vocab.parent)
        if not cache_path.exists(): shutil.copy2(named_vocab, cache_path)
        debug["vocab"] = str(named_vocab)
    if hashed_vocab is not None:
        if not cache_path.exists(): shutil.copy2(hashed_vocab, cache_path)
        debug["hashed_vocab"] = str(hashed_vocab)
    if cache_path.exists():
        os.environ["TIKTOKEN_RS_CACHE_DIR"] = str(cache_dir)
        debug["tiktoken_rs_cache"] = str(cache_path)
    print(debug)



## 2. Setup Execution
Installs missing packages if needed, configures tokenizer assets, and resolves the competition/model paths.

In [ ]:
ensure_offline_packages()
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = CFG.primary_cuda_visible_devices or CFG.cuda_visible_devices
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("PYTHONUNBUFFERED", "1")
configure_harmony_vocab()

COMP_PATH = find_competition_path(required=False)
PRIMARY_MODEL_HINTS = [CFG.primary_model_hint, "gpt-oss-120b", "gpt-oss-20b"]
MODEL_PATH = find_optional_model_path(
    PRIMARY_MODEL_HINTS,
    explicit_path_env=CFG.primary_model_path_env,
    explicit_env_name="AIMO3_PRIMARY_MODEL_PATH",
)
if MODEL_PATH is None:
    raise FileNotFoundError(f"Primary model not found for hints: {PRIMARY_MODEL_HINTS}")
MODEL_FAMILY = infer_model_family(MODEL_PATH, CFG.primary_model_hint)
CONTRARIAN_MODEL_HINTS = parse_hint_list(CFG.contrarian_model_hints_env)
CONTRARIAN_MODEL_PATH = find_optional_model_path(
    CONTRARIAN_MODEL_HINTS,
    explicit_path_env=CFG.contrarian_model_path_env,
    explicit_env_name="AIMO3_CONTRARIAN_MODEL_PATH",
) if CFG.enable_aux_models else None
CONTRARIAN_MODEL_FAMILY = infer_model_family(CONTRARIAN_MODEL_PATH, CONTRARIAN_MODEL_HINTS[0] if CONTRARIAN_MODEL_HINTS else "")
VERIFIER_MODEL_PATH = find_optional_model_path(
    [CFG.verifier_model_hint],
    explicit_path_env=CFG.verifier_model_path_env,
    explicit_env_name="AIMO3_VERIFIER_MODEL_PATH",
) if CFG.enable_aux_models else None
VERIFIER_MODEL_FAMILY = infer_model_family(VERIFIER_MODEL_PATH, CFG.verifier_model_hint)

PRIMARY_MAX_MODEL_LEN = resolve_effective_max_model_len(MODEL_PATH, CFG.max_model_len)
CONTRARIAN_MAX_MODEL_LEN = (
    resolve_effective_max_model_len(CONTRARIAN_MODEL_PATH, CFG.max_model_len)
    if CONTRARIAN_MODEL_PATH is not None else None
)
VERIFIER_MAX_MODEL_LEN = (
    resolve_effective_max_model_len(VERIFIER_MODEL_PATH, CFG.max_model_len)
    if VERIFIER_MODEL_PATH is not None else None
)
CFG.max_model_len = PRIMARY_MAX_MODEL_LEN
ADAPTER_PATH = find_adapter_path(CFG.adapter_hint)
REQUEST_MODEL_NAME = CFG.adapter_alias if ADAPTER_PATH is not None else CFG.primary_served_model_name
PRIMARY_REQUEST_MODEL_NAME = REQUEST_MODEL_NAME
CONTRARIAN_REQUEST_MODEL_NAME = CFG.contrarian_served_model_name
VERIFIER_REQUEST_MODEL_NAME = CFG.verifier_served_model_name
print({
    "primary_model_path": str(MODEL_PATH),
    "primary_model_family": MODEL_FAMILY,
    "contrarian_model_path": None if CONTRARIAN_MODEL_PATH is None else str(CONTRARIAN_MODEL_PATH),
    "contrarian_model_family": CONTRARIAN_MODEL_FAMILY,
    "verifier_model_path": None if VERIFIER_MODEL_PATH is None else str(VERIFIER_MODEL_PATH),
    "verifier_model_family": VERIFIER_MODEL_FAMILY,
    "primary_max_model_len": PRIMARY_MAX_MODEL_LEN,
    "contrarian_max_model_len": CONTRARIAN_MAX_MODEL_LEN,
    "verifier_max_model_len": VERIFIER_MAX_MODEL_LEN,
})
if COMP_PATH is not None and str(COMP_PATH) not in sys.path: sys.path.insert(0, str(COMP_PATH))


## 3. Runtime Imports
Imports packages that depend on the setup cell and prints the resolved runtime state.

In [ ]:
import pandas as pd
from openai import OpenAI
try:
    import kaggle_evaluation.aimo_3_inference_server as aimo3_inference_server
except Exception:
    aimo3_inference_server = None
from openai_harmony import Author, Conversation, DeveloperContent, HarmonyEncodingName, Message, ReasoningEffort, Role, SystemContent, load_harmony_encoding
try:
    from jupyter_client import KernelManager
except Exception:
    KernelManager = None
print({"runtime": RUNTIME_NAME, "competition_path": None if COMP_PATH is None else str(COMP_PATH), "model_path": str(MODEL_PATH), "adapter_path": None if ADAPTER_PATH is None else str(ADAPTER_PATH), "request_model": REQUEST_MODEL_NAME, "work_root": str(WORK_ROOT)})



## 4. Common Notebook Utilities
Small helpers used across parsing, logging, and experiment display.

In [ ]:
def clip_text(text, limit=CFG.debug_text_chars):
    text = (text or "").replace("\x00", "")
    if len(text) <= limit:
        return text
    return text[: limit // 2] + "\n...[truncated]...\n" + text[-limit // 2:]


## 5. Parsing, Prompts, and Routing
Answer parsing, route classification, certificate heuristics, and prompt scaffolding.

In [ ]:

# Robust parsing, schemas, certificates, and routing helpers for the modular harness.

INT_RE = re.compile(r"[-+]?\d+")
BOXED_RE = re.compile(r"\\boxed\s*\{([^{}]+)\}")
JSON_OBJECT_RE = re.compile(r"\{.*\}", re.DOTALL)
LAST_LINE_INTEGER_RE = re.compile(r"^\s*([-+]?\d+)\s*$")
TOOL_CONFUSION_RE = re.compile(
    r"(?:can't|cannot|can not|unable to|not able to)\s+(?:run|execute|see|access)|"
    r"(?:not executing|not showing output|not set up|code execution is not possible)|"
    r"(?:no|without)\s+(?:python|code)\s+(?:output|execution)|"
    r"(?:python|tool|code execution)\s+(?:is|are|seems|seem)\s+not\s+(?:available|enabled|working)",
    re.IGNORECASE,
)
APPROX_REASONING_RE = re.compile(
    r"(approx|approximately|numerical(?:ly)?|manual(?:ly)? approximate|root between|estimate|heuristic|looks like|seems likely|sample(?:d|s| )|simulate|simulation|monte carlo)",
    re.IGNORECASE,
)
RANDOM_SEARCH_RE = re.compile(
    r"(import\s+random|random_search|np\.random|uniform\(|randint\(|sample\(|monte carlo|simulat(?:e|ion)|floating search)",
    re.IGNORECASE,
)
EXACT_REASONING_RE = re.compile(
    r"(therefore|hence|contradiction|for all|if and only if|iff|factor|factorization|symbolic|closed form|exact|equality holds|lower bound|upper bound|construction|induction|recurrence|vieta|discriminant|telescop|bijection|inclusion-exclusion|pigeonhole)",
    re.IGNORECASE,
)
CONSTRUCTION_RE = re.compile(r"(construction|choose|take|set .*?=|equality holds|attained at|achieved by)", re.IGNORECASE)
BOUND_RE = re.compile(r"(lower bound|upper bound|minimality|maximality|cannot be smaller|cannot be larger|at least|at most|thus minimum|thus maximum)", re.IGNORECASE)
ASSUMPTION_RE = re.compile(r"\b(?:assume|suppose|wlog|without loss of generality)\b", re.IGNORECASE)
ENUMERATION_CODE_RE = re.compile(r"(assert\s|all\(|itertools|product\(|permutations\(|combinations\(|combinations_with_replacement\(|for\s+\w+\s+in\s+range\()", re.IGNORECASE)
SYMPY_CODE_RE = re.compile(r"(import\s+sympy|from\s+sympy|sp\.|sympy\.|simplify\(|factor\(|solve\(|expand\()", re.IGNORECASE)
COUNTEREXAMPLE_RE = re.compile(r"(counterexample|no counterexample|checked all cases|violat(?:e|es|ion)|refute)", re.IGNORECASE)
CHECK_EVIDENCE_RE = re.compile(r"(assert|verified|check(ed)?|confirmed|proved|exhaustive)", re.IGNORECASE)

CERTIFICATE_RANK = {
    "none": 0,
    "counterexample_check": 1,
    "optimization_witness": 2,
    "equivalence_check": 3,
    "exhaustive_check": 4,
    "formal_proof": 5,
}
MACHINE_CHECKABLE_CERTIFICATES = {"counterexample_check", "equivalence_check", "exhaustive_check", "formal_proof"}
DISCRETE_TOOL_CERT_ROUTES = {
    "combinatorics_counting",
    "optimization_extremal",
    "subset_sum_threshold",
    "number_theory_exact",
    "recurrence_sequence_structure",
}
STRONG_CHECK_OUTPUT_RE = re.compile(
    r"(checked all cases|all cases checked|verified|verification succeeded|no counterexample|identity holds|all tests passed|assertions passed|proved for all|exhaustive)",
    re.IGNORECASE,
)
DISCRETE_TOOL_CERT_ROUTES = {
    "combinatorics_counting",
    "optimization_extremal",
    "subset_sum_threshold",
    "number_theory_exact",
    "recurrence_sequence_structure",
}
STRONG_CHECK_OUTPUT_RE = re.compile(
    r"(checked all cases|all cases checked|verified|verification succeeded|no counterexample|identity holds|all tests passed|assertions passed|proved for all|exhaustive)",
    re.IGNORECASE,
)
DISCRETE_TOOL_CERT_ROUTES = {
    "combinatorics_counting",
    "optimization_extremal",
    "subset_sum_threshold",
    "number_theory_exact",
    "recurrence_sequence_structure",
}
STRONG_CHECK_OUTPUT_RE = re.compile(
    r"(checked all cases|all cases checked|verified|verification succeeded|no counterexample|identity holds|all tests passed|assertions passed|proved for all|exhaustive)",
    re.IGNORECASE,
)
DISCRETE_TOOL_CERT_ROUTES = {
    "combinatorics_counting",
    "optimization_extremal",
    "subset_sum_threshold",
    "number_theory_exact",
    "recurrence_sequence_structure",
}
STRONG_CHECK_OUTPUT_RE = re.compile(
    r"(checked all cases|all cases checked|verified|verification succeeded|no counterexample|identity holds|all tests passed|assertions passed|proved for all|exhaustive)",
    re.IGNORECASE,
)
DISCRETE_TOOL_CERT_ROUTES = {
    "combinatorics_counting",
    "optimization_extremal",
    "subset_sum_threshold",
    "number_theory_exact",
    "recurrence_sequence_structure",
}
STRONG_CHECK_OUTPUT_RE = re.compile(
    r"(checked all cases|all cases checked|verified|verification succeeded|no counterexample|identity holds|all tests passed|assertions passed|proved for all|exhaustive)",
    re.IGNORECASE,
)
DISCRETE_TOOL_CERT_ROUTES = {
    "combinatorics_counting",
    "optimization_extremal",
    "subset_sum_threshold",
    "number_theory_exact",
    "recurrence_sequence_structure",
}
STRONG_CHECK_OUTPUT_RE = re.compile(
    r"(checked all cases|all cases checked|verified|verification succeeded|no counterexample|identity holds|all tests passed|assertions passed|proved for all|exhaustive)",
    re.IGNORECASE,
)

ROUTE_PROMPTS = {
    "optimization_extremal": "Optimization lens: prove both achievability and optimality. Do not trust a candidate without a matching construction and bound.",
    "functional_equation_structure": "Functional-equation lens: probe identities, injectivity/surjectivity, zero sets, composition, and regularity assumptions before guessing a closed form.",
    "subset_sum_threshold": "Subset/counting lens: describe the reachable set or counted family exactly; derive a recurrence, bijection, invariant, or inclusion-exclusion argument.",
    "recurrence_sequence_structure": "Recurrence lens: isolate the transition rule and search for an invariant, monotonicity, or closed-form pattern that can be proved.",
    "polynomial_root_structure": "Polynomial lens: use factorization, multiplicities, discriminant, derivative, or Vieta relations before numerical root hunting.",
    "geometry_exact": "Geometry lens: identify the invariant configuration first. Use coordinates only as a check unless they materially simplify the proof.",
    "geometry_inequality": "Geometry-extremal lens: identify the equality configuration and prove it is optimal.",
    "number_theory_exact": "Number-theory lens: factor first, use congruences/valuations carefully, and verify all divisibility edge cases exactly.",
    "combinatorics_counting": "Combinatorics lens: define the objects precisely and avoid pattern extrapolation from tiny examples unless you later prove it.",
    "generic": "General lens: small cases may guide the proof, but they do not replace one.",
}

def normalize_answer(value):
    return int(value) % 100000

def extract_json_object(text):
    text = str(text or "")
    if not text.strip():
        return None
    decoder = json.JSONDecoder()
    candidates = []
    for i, ch in enumerate(text):
        if ch != "{":
            continue
        try:
            obj, end = decoder.raw_decode(text[i:])
        except Exception:
            continue
        if isinstance(obj, dict):
            keys = set(obj.keys())
            answerish = len(keys & {"answer", "selected_answer", "final_answer", "answer_mod_100000"})
            schema = len(keys & {"status", "method", "proof_sketch", "certificate_type", "certificate_summary", "assumptions", "self_checks", "final_statement"})
            candidates.append(((answerish, schema, len(keys), i), obj))
    if not candidates:
        return None
    candidates.sort(key=lambda item: item[0], reverse=True)
    return candidates[0][1]

def extract_answer(text):
    text = str(text or "")
    obj = extract_json_object(text)
    if isinstance(obj, dict):
        for key in ("answer", "selected_answer", "final_answer", "answer_mod_100000"):
            if obj.get(key) is None:
                continue
            try:
                return normalize_answer(obj[key])
            except Exception:
                pass
    boxed = BOXED_RE.findall(text)
    if boxed:
        ints = INT_RE.findall(boxed[-1])
        if ints:
            try:
                return normalize_answer(int(ints[-1]))
            except Exception:
                pass
    stripped = text.strip()
    last_line = stripped.splitlines()[-1].strip() if stripped else ""
    for candidate_text in (last_line, stripped[-240:]):
        for pattern in (
            r"(?i)\bfinal answer\s*(?:is|=|:)?\s*(-?\d+)\b",
            r"(?i)\banswer\s*(?:is|=|:)\s*(-?\d+)\b",
            r"^\s*(-?\d+)\s*$",
        ):
            m = re.search(pattern, candidate_text)
            if not m:
                continue
            try:
                return normalize_answer(int(m.group(1)))
            except Exception:
                pass
    return None

def has_tool_confusion(text):
    return bool(text and TOOL_CONFUSION_RE.search(text))

def validate_candidate_payload(obj):
    if not isinstance(obj, dict):
        return False, "candidate_not_dict"
    required = {
        "answer", "status", "method", "proof_sketch",
        "certificate_type", "certificate_summary",
        "assumptions", "self_checks", "final_statement"
    }
    missing = [k for k in required if k not in obj]
    if missing:
        return False, f"candidate_missing:{','.join(missing)}"
    if obj["answer"] is not None:
        try:
            obj["answer"] = normalize_answer(obj["answer"])
        except Exception:
            return False, "candidate_bad_answer"
    if obj["status"] not in {"solved", "partial", "stuck"}:
        return False, "candidate_bad_status"
    if obj["certificate_type"] not in CERTIFICATE_RANK:
        return False, "candidate_bad_certificate_type"
    if not isinstance(obj["assumptions"], list):
        return False, "candidate_bad_assumptions"
    if not isinstance(obj["self_checks"], list):
        return False, "candidate_bad_self_checks"
    return True, None

def validate_verifier_payload(obj, allowed_answers):
    if not isinstance(obj, dict):
        return False, "verifier_not_dict"
    required = {"selected_answer", "status", "reason", "reject_reasons"}
    missing = [k for k in required if k not in obj]
    if missing:
        return False, f"verifier_missing:{','.join(missing)}"
    if obj["status"] not in {"verified", "inconclusive", "rejected_all"}:
        return False, "verifier_bad_status"
    ans = obj["selected_answer"]
    if ans is None:
        return True, None
    try:
        ans = normalize_answer(ans)
    except Exception:
        return False, "verifier_bad_answer"
    if ans not in set(int(x) for x in allowed_answers):
        return False, "verifier_out_of_set"
    obj["selected_answer"] = ans
    return True, None

def tool_status(output):
    text = str(output or "")
    if "status: timeout" in text.lower():
        return "timeout"
    if "traceback" in text.lower() or "stderr:" in text.lower():
        return "error"
    if "status: exit_code=0" in text.lower() or text.strip() == "status: ok":
        return "ok"
    return "ok"

def parse_tool_final_answer(output):
    text = str(output or "").strip()
    if not text:
        return None
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if not lines:
        return None
    for candidate in reversed(lines[-5:]):
        try:
            obj = json.loads(candidate.replace("'", '"'))
        except Exception:
            obj = None
        if isinstance(obj, dict) and obj.get("final_answer") is not None:
            try:
                return normalize_answer(obj["final_answer"])
            except Exception:
                return None
        m = LAST_LINE_INTEGER_RE.match(candidate)
        if m:
            return normalize_answer(int(m.group(1)))
    return None

def build_tool_artifact(code, output):
    output = str(output or "")
    return {
        "status": tool_status(output),
        "code": clip_text(str(code or ""), 900),
        "output": clip_text(output, 1200),
        "final_answer": parse_tool_final_answer(output),
    }

def tool_artifact_has_explicit_check_signal(artifact):
    combined = str(artifact.get("code", "")) + "\n" + str(artifact.get("output", ""))
    return bool(STRONG_CHECK_OUTPUT_RE.search(combined))

def is_machine_checkable_attempt(attempt, answer=None, tool_count=0):
    if attempt is None:
        return False
    certificate_type = str(getattr(attempt, "certificate_type", "none") or "none")
    if certificate_type not in MACHINE_CHECKABLE_CERTIFICATES:
        return False
    if certificate_type != "formal_proof" and int(getattr(attempt, "python_calls", 0) or 0) <= 0:
        return False
    route = str(getattr(attempt, "route", "") or "")
    if route in DISCRETE_TOOL_CERT_ROUTES and int(tool_count or 0) < 2:
        return False
    return True

def tool_artifact_has_explicit_check_signal(artifact):
    combined = str(artifact.get("code", "")) + "\n" + str(artifact.get("output", ""))
    return bool(STRONG_CHECK_OUTPUT_RE.search(combined))

def is_machine_checkable_attempt(attempt, answer=None, tool_count=0):
    if attempt is None:
        return False
    certificate_type = str(getattr(attempt, "certificate_type", "none") or "none")
    if certificate_type not in MACHINE_CHECKABLE_CERTIFICATES:
        return False
    if certificate_type != "formal_proof" and int(getattr(attempt, "python_calls", 0) or 0) <= 0:
        return False
    route = str(getattr(attempt, "route", "") or "")
    if route in DISCRETE_TOOL_CERT_ROUTES and int(tool_count or 0) < 2:
        return False
    return True

def tool_artifact_has_explicit_check_signal(artifact):
    combined = str(artifact.get("code", "")) + "\n" + str(artifact.get("output", ""))
    return bool(STRONG_CHECK_OUTPUT_RE.search(combined))

def is_machine_checkable_attempt(attempt, answer=None, tool_count=0):
    if attempt is None:
        return False
    certificate_type = str(getattr(attempt, "certificate_type", "none") or "none")
    if certificate_type not in MACHINE_CHECKABLE_CERTIFICATES:
        return False
    if certificate_type != "formal_proof" and int(getattr(attempt, "python_calls", 0) or 0) <= 0:
        return False
    route = str(getattr(attempt, "route", "") or "")
    if route in DISCRETE_TOOL_CERT_ROUTES and int(tool_count or 0) < 2:
        return False
    return True

def tool_artifact_has_explicit_check_signal(artifact):
    combined = str(artifact.get("code", "")) + "\n" + str(artifact.get("output", ""))
    return bool(STRONG_CHECK_OUTPUT_RE.search(combined))

def is_machine_checkable_attempt(attempt, answer=None, tool_count=0):
    if attempt is None:
        return False
    certificate_type = str(getattr(attempt, "certificate_type", "none") or "none")
    if certificate_type not in MACHINE_CHECKABLE_CERTIFICATES:
        return False
    if certificate_type != "formal_proof" and int(getattr(attempt, "python_calls", 0) or 0) <= 0:
        return False
    route = str(getattr(attempt, "route", "") or "")
    if route in DISCRETE_TOOL_CERT_ROUTES and int(tool_count or 0) < 2:
        return False
    return True

def tool_artifact_has_explicit_check_signal(artifact):
    combined = str(artifact.get("code", "")) + "\n" + str(artifact.get("output", ""))
    return bool(STRONG_CHECK_OUTPUT_RE.search(combined))

def is_machine_checkable_attempt(attempt, answer=None, tool_count=0):
    if attempt is None:
        return False
    certificate_type = str(getattr(attempt, "certificate_type", "none") or "none")
    if certificate_type not in MACHINE_CHECKABLE_CERTIFICATES:
        return False
    if certificate_type != "formal_proof" and int(getattr(attempt, "python_calls", 0) or 0) <= 0:
        return False
    route = str(getattr(attempt, "route", "") or "")
    if route in DISCRETE_TOOL_CERT_ROUTES and int(tool_count or 0) < 2:
        return False
    return True

def tool_artifact_has_explicit_check_signal(artifact):
    combined = str(artifact.get("code", "")) + "\n" + str(artifact.get("output", ""))
    return bool(STRONG_CHECK_OUTPUT_RE.search(combined))

def is_machine_checkable_attempt(attempt, answer=None, tool_count=0):
    if attempt is None:
        return False
    certificate_type = str(getattr(attempt, "certificate_type", "none") or "none")
    if certificate_type not in MACHINE_CHECKABLE_CERTIFICATES:
        return False
    if certificate_type != "formal_proof" and int(getattr(attempt, "python_calls", 0) or 0) <= 0:
        return False
    route = str(getattr(attempt, "route", "") or "")
    if route in DISCRETE_TOOL_CERT_ROUTES and int(tool_count or 0) < 2:
        return False
    return True

def classify_certificate(final_text, route, tool_artifacts):
    best_type = "none"
    best_summary = "No machine-checkable certificate found."
    for artifact in tool_artifacts or ():
        status = artifact.get("status")
        code = str(artifact.get("code", ""))
        output = str(artifact.get("output", ""))
        combined = code + "\n" + output
        explicit_check = tool_artifact_has_explicit_check_signal(artifact)
        has_assert = "assert" in code
        has_final_answer = artifact.get("final_answer") is not None
        if status == "ok" and SYMPY_CODE_RE.search(code) and has_assert:
            if explicit_check or has_final_answer:
                return "equivalence_check", "SymPy-style symbolic verification completed without tool errors."
        if status == "ok" and ENUMERATION_CODE_RE.search(code) and has_assert:
            if explicit_check:
                return "exhaustive_check", "Deterministic python enumeration/assertion completed without tool errors."
        if status == "ok" and COUNTEREXAMPLE_RE.search(combined) and (explicit_check or has_assert):
            best_type = "counterexample_check"
            best_summary = "Tool-based search reported no counterexample in the checked space."
    if route == "optimization_extremal" and CONSTRUCTION_RE.search(str(final_text or "")) and BOUND_RE.search(str(final_text or "")):
        best_type = "optimization_witness"
        best_summary = "Reasoning contains both an extremal construction and a matching bound."
    return best_type, best_summary

def certificate_rank(certificate_type):
    return CERTIFICATE_RANK.get(str(certificate_type or "none"), 0)

def extract_assumptions(text):
    assumptions = []
    for line in str(text or "").splitlines():
        line = line.strip()
        if line and ASSUMPTION_RE.search(line):
            assumptions.append(clip_text(line, 220))
        if len(assumptions) >= 5:
            break
    return tuple(dict.fromkeys(assumptions))

def summarize_self_checks(final_text, tool_artifacts):
    checks = []
    for artifact in tool_artifacts or ():
        code = str(artifact.get("code", ""))
        output = str(artifact.get("output", ""))
        if "assert" in code:
            checks.append("python_assertions")
        if artifact.get("status") == "ok":
            checks.append("python_completed_cleanly")
        if artifact.get("final_answer") is not None:
            checks.append("tool_produced_final_answer")
        if COUNTEREXAMPLE_RE.search(code + "\n" + output):
            checks.append("counterexample_search")
        if CHECK_EVIDENCE_RE.search(output):
            checks.append("tool_reported_check")
    if CHECK_EVIDENCE_RE.search(str(final_text or "")):
        checks.append("reasoning_mentions_check")
    return tuple(dict.fromkeys(checks))

def classify_problem_routes(problem_text):
    lower = str(problem_text or "").lower()
    scores = Counter()
    extremal = any(k in lower for k in ("maximum", "minimum", "largest", "smallest", "least", "greatest", "minimum possible", "maximum possible"))
    if ("function" in lower or re.search(r"\bf\(", lower) or re.search(r"\bg\(", lower)) and any(k in lower for k in ("for all", "satisfies", "all real", "all integers", "all rational")):
        scores["functional_equation_structure"] += 6
    if any(k in lower for k in ("polynomial", "coefficient", "root", "roots", "discriminant", "vieta")):
        scores["polynomial_root_structure"] += 5
    if any(k in lower for k in ("sequence", "recurrence", "a_n", "x_n", "a_{", "x_{")):
        scores["recurrence_sequence_structure"] += 5
    if any(k in lower for k in ("how many", "number of", "ways", "permutation", "subset", "color", "graph", "path")):
        scores["combinatorics_counting"] += 4
    if any(k in lower for k in ("triangle", "circle", "circumcircle", "incenter", "orthocenter", "centroid", "quadrilateral", "tetrahedron", "sphere", "\\angle", "angle")):
        scores["geometry_exact"] += 4
        if extremal:
            scores["geometry_inequality"] += 3
    if any(k in lower for k in ("mod", "remainder", "divisor", "prime", "valuation", "gcd", "lcm", "congruent", "divisible")):
        scores["number_theory_exact"] += 4
    if extremal:
        scores["optimization_extremal"] += 3
    if any(k in lower for k in ("subset", "reachable", "sum", "coverage", "threshold")):
        scores["subset_sum_threshold"] += 2
    if not scores:
        scores["generic"] = 1
    ranked = [k for k, _ in scores.most_common()]
    if "generic" not in ranked:
        ranked.append("generic")
    return ranked, scores

def classify_problem_route(problem_text):
    return classify_problem_routes(problem_text)[0][0]

def route_prompt(route):
    if isinstance(route, (list, tuple)):
        return "\n".join(dict.fromkeys(ROUTE_PROMPTS.get(item, ROUTE_PROMPTS["generic"]) for item in route))
    return ROUTE_PROMPTS.get(route, ROUTE_PROMPTS["generic"])

def route_attempt_stance(route, attempt_index):
    base = CFG.attempt_profiles[attempt_index % len(CFG.attempt_profiles)]
    if route == "optimization_extremal":
        return base + " Prove both feasibility and optimality; do not accept a pure guess."
    if route == "functional_equation_structure":
        return base + " Treat special values and algebraic identities as the main entry point."
    if route in {"polynomial_root_structure", "number_theory_exact"}:
        return base + " Prefer symbolic manipulation and exact divisibility logic to numeric exploration."
    return base

def attempt_temperature(attempt_index):
    return CFG.attempt_temperatures[attempt_index % len(CFG.attempt_temperatures)]

def route_prefers_verifier(route):
    return route in {"optimization_extremal", "functional_equation_structure", "subset_sum_threshold", "recurrence_sequence_structure", "polynomial_root_structure", "geometry_inequality", "number_theory_exact"}

def exact_heavy_route(route):
    return route in {"optimization_extremal", "functional_equation_structure", "polynomial_root_structure", "recurrence_sequence_structure", "geometry_exact", "geometry_inequality", "number_theory_exact"}

def analyze_reasoning_quality(text, route, python_calls):
    text = str(text or "")
    approx_only = bool(APPROX_REASONING_RE.search(text)) and not bool(EXACT_REASONING_RE.search(text))
    random_search = bool(RANDOM_SEARCH_RE.search(text))
    weak_exactness = exact_heavy_route(route) and not bool(EXACT_REASONING_RE.search(text))
    missing_extremal_certificate = route == "optimization_extremal" and not (CONSTRUCTION_RE.search(text) and BOUND_RE.search(text))
    penalty = 1.0
    if random_search:
        penalty *= CFG.random_search_penalty
    if approx_only:
        penalty *= CFG.approximate_reasoning_penalty
    if weak_exactness and python_calls > 0:
        penalty *= CFG.weak_exactness_penalty
    if missing_extremal_certificate:
        penalty *= CFG.extremal_certificate_penalty
    return penalty, {
        "approx_only": approx_only,
        "random_search": random_search,
        "weak_exactness": weak_exactness,
        "missing_extremal_certificate": missing_extremal_certificate,
    }

def domain_strategy_hint(problem_text):
    lower = str(problem_text or "").lower()
    routes, route_scores = classify_problem_routes(problem_text)
    hints = [route_prompt(routes[:2])]
    if route_scores:
        hints.append("Route scores: " + ", ".join(f"{name}={score}" for name, score in list(route_scores.items())[:4]))
    if any(k in lower for k in ("mod", "remainder", "divisor", "prime", "valuation", "gcd", "lcm")):
        hints.append("Number theory: factor first, keep exact modular/valuation checks, and verify boundary cases.")
    if any(k in lower for k in ("how many", "number of", "ways", "permutation", "sequence", "graph", "path", "color")):
        hints.append("Combinatorics: define the counted objects precisely, test small cases, then derive a recurrence, bijection, or inclusion-exclusion formula.")
    if any(k in lower for k in ("triangle", "circle", "circumcircle", "incenter", "angle", "geometry", "tetrahedron", "sphere")):
        hints.append("Geometry: identify invariant geometry first; use coordinates or symbolic computation only as an independent check.")
    if any(k in lower for k in ("polynomial", "roots", "coefficient", "function", "equation", "matrix")):
        hints.append("Algebra: look for structure-preserving substitutions, factorization, interpolation, or linear recurrence before numerical guessing.")
    if any(k in lower for k in ("maximum", "minimum", "largest", "smallest", "least", "greatest")):
        hints.append("Optimization: prove both construction and upper/lower bound; do not accept a candidate without a matching extremal argument.")
    if any(k in lower for k in ("maximum", "minimum", "largest", "smallest", "least", "greatest", "threshold", "mod", "remainder", "divisor", "prime", "valuation", "gcd", "lcm")):
        hints.append("Boundary discipline: if a candidate looks like a threshold or clean value, explicitly test nearby cases such as n-1 and n+1 before finalizing.")
    return "\n".join(dict.fromkeys(hints))

def problem_requires_full_attempts(problem_text, route=None):
    route = route or classify_problem_route(problem_text)
    routes, scores = classify_problem_routes(problem_text)
    lower = str(problem_text or "").lower()
    mixed = len([k for k,v in scores.items() if v >= 3]) >= 2
    return (
        len(str(problem_text or "")) > 900
        or mixed
        or route in {"optimization_extremal", "functional_equation_structure", "subset_sum_threshold", "recurrence_sequence_structure", "polynomial_root_structure", "geometry_inequality"}
        or any(k in lower for k in ("prove", "show that", "determine all", "smallest possible", "largest possible"))
    )


## 6. Sandboxes and Server Controls
Scratchpad sandboxes plus explicit helpers to start, stop, and reset the local inference runtime.

In [ ]:
def write_debug_event(event):
    if not CFG.log_jsonl or os.getenv("KAGGLE_IS_COMPETITION_RERUN"): return
    with (WORK_ROOT / "aimo3_frontier_debug.jsonl").open("a", encoding="utf-8") as h:
        h.write(json.dumps(event, ensure_ascii=True, default=str) + "\n")

class SubprocessSandbox:
    def __init__(self):
        self.poisoned = False
        self.history = ["import math, itertools, functools, fractions, collections, decimal, statistics", "from fractions import Fraction", "try:\n    import sympy as sp\nexcept Exception:\n    sp = None", "try:\n    import numpy as np\nexcept Exception:\n    np = None"]
    def execute(self, code, timeout=CFG.python_timeout_seconds):
        script = "\n".join(self.history + [code])
        with tempfile.TemporaryDirectory(prefix="aimo3_frontier_tool_") as tmp:
            p = Path(tmp) / "tool.py"; p.write_text(script + "\n", encoding="utf-8")
            try:
                r = subprocess.run([sys.executable, str(p)], capture_output=True, text=True, cwd=tmp, timeout=timeout)
                if r.returncode == 0: self.history.append(code)
                out = [f"status: exit_code={r.returncode}"]
                if r.stdout.strip(): out.append("stdout:\n" + r.stdout.strip())
                if r.stderr.strip(): out.append("stderr:\n" + r.stderr.strip())
                return "\n\n".join(out)[-12000:]
            except subprocess.TimeoutExpired as e:
                so, se = e.stdout or "", e.stderr or ""
                if isinstance(so, bytes): so = so.decode("utf-8", errors="ignore")
                if isinstance(se, bytes): se = se.decode("utf-8", errors="ignore")
                return f"status: timeout after {timeout}s\nstdout:\n{so.strip()}\nstderr:\n{se.strip()}"[-12000:]
    def close(self): pass

class JupyterSandbox:
    def __init__(self):
        if KernelManager is None: raise RuntimeError("jupyter_client is unavailable")
        self.poisoned = False
        self.km = None
        self.kc = None
        self._start_kernel()
        self.execute("import math, itertools, functools, fractions, collections, decimal, statistics\nfrom fractions import Fraction\ntry:\n    import sympy as sp\nexcept Exception:\n    sp = None\ntry:\n    import numpy as np\nexcept Exception:\n    np = None\n", timeout=10)
    def _start_kernel(self):
        self.km = KernelManager(kernel_name="python3")
        self.km.start_kernel()
        self.kc = self.km.client()
        self.kc.start_channels()
        self.kc.wait_for_ready(timeout=30)
    def _interrupt_and_poison(self):
        self.poisoned = True
        try:
            self.km.interrupt_kernel()
        except Exception:
            pass
    def execute(self, code, timeout=CFG.python_timeout_seconds):
        if self.poisoned:
            raise RuntimeError("sandbox_poisoned")
        msg_id = self.kc.execute(code, store_history=True); outputs = []; deadline = time.time() + timeout
        while True:
            if time.time() > deadline:
                outputs.append(f"status: timeout after {timeout}s")
                self._interrupt_and_poison()
                break
            try: msg = self.kc.get_iopub_msg(timeout=min(max(0.1, deadline - time.time()), 1.0))
            except Exception: continue
            if msg.get("parent_header", {}).get("msg_id") != msg_id: continue
            t, c = msg.get("msg_type"), msg.get("content", {})
            if t == "stream": outputs.append(str(c.get("text", "")))
            elif t in ("display_data", "execute_result"): outputs.append(str(c.get("data", {}).get("text/plain", "")))
            elif t == "error": outputs.append("\n".join(str(x) for x in (c.get("traceback") or [c.get("ename", "error"), c.get("evalue", "")])) )
            elif t == "status" and c.get("execution_state") == "idle": break
        return ("".join(outputs).strip() or "status: ok")[-12000:]
    def close(self):
        try: self.kc.stop_channels()
        except Exception: pass
        try: self.km.shutdown_kernel(now=True)
        except Exception: pass

class NoReuseSandboxPool:
    def acquire(self):
        return SubprocessSandbox()
    def release(self, sandbox):
        try:
            sandbox.close()
        except Exception:
            pass
    def close(self):
        return None

SANDBOX_POOL = NoReuseSandboxPool()

def gpu_cleanup():
    for pat in ("vllm", "api_server", "EngineCore"):
        subprocess.run(f"pkill -9 -f {pat}", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(3)

class VLLMServer:
    def __init__(
        self,
        role_name,
        model_path,
        served_model_name,
        port,
        cuda_visible_devices=None,
        adapter_path=None,
        gpu_memory_utilization=None,
        max_model_len=None,
    ):
        self.role_name = str(role_name)
        self.model_path = Path(model_path)
        self.served_model_name = str(served_model_name)
        self.port = int(port)
        self.cuda_visible_devices = None if cuda_visible_devices in (None, "") else str(cuda_visible_devices)
        self.adapter_path = None if adapter_path is None else Path(adapter_path)
        self.gpu_memory_utilization = float(
            CFG.gpu_memory_utilization if gpu_memory_utilization is None else gpu_memory_utilization
        )
        self.max_model_len = int(resolve_effective_max_model_len(
            self.model_path,
            CFG.max_model_len if max_model_len is None else max_model_len,
        ))
        self.base_url = f"http://{CFG.server_host}:{self.port}/v1"
        self.log_path = WORK_ROOT / f"vllm_frontier_server_{self.role_name}.log"
        self.process = None
        self.log_handle = None
        self.client = None

    def start(self, cleanup=False):
        if cleanup:
            gpu_cleanup()
        env = os.environ.copy()
        if self.cuda_visible_devices:
            env["CUDA_VISIBLE_DEVICES"] = self.cuda_visible_devices
        cmd = [
            sys.executable,
            "-m",
            "vllm.entrypoints.openai.api_server",
            "--model",
            str(self.model_path),
            "--served-model-name",
            self.served_model_name,
            "--host",
            CFG.server_host,
            "--port",
            str(self.port),
            "--dtype",
            "auto",
            "--kv-cache-dtype",
            CFG.kv_cache_dtype,
            "--gpu-memory-utilization",
            str(self.gpu_memory_utilization),
            "--max-model-len",
            str(self.max_model_len),
            "--max-num-seqs",
            str(CFG.max_num_seqs),
            "--seed",
            str(CFG.startup_seed),
            "--generation-config",
            "vllm",
            "--enable-prefix-caching",
            "--disable-log-stats",
        ]
        if CFG.vllm_enforce_eager:
            cmd.append("--enforce-eager")
        if self.adapter_path is not None:
            cmd += [
                "--enable-lora",
                "--max-lora-rank",
                str(CFG.max_lora_rank),
                "--lora-modules",
                f"{CFG.adapter_alias}={self.adapter_path}",
            ]
        self.log_handle = self.log_path.open("w", encoding="utf-8")
        self.process = subprocess.Popen(cmd, stdout=self.log_handle, stderr=subprocess.STDOUT, env=env)
        self.client = OpenAI(base_url=self.base_url, api_key="EMPTY", timeout=CFG.request_timeout_seconds)
        t0 = time.time()
        while time.time() - t0 < CFG.server_timeout_seconds:
            if self.process.poll() is not None:
                self.log_handle.flush()
                logs = self.log_path.read_text(encoding="utf-8", errors="ignore")[-12000:]
                raise RuntimeError(f"vLLM exited early for {self.role_name}:\n{logs}")
            try:
                models = self.client.models.list()
                model_ids = [getattr(item, "id", None) for item in getattr(models, "data", [])]
                print({
                    "vllm": "ready",
                    "role": self.role_name,
                    "startup_s": f"{time.time() - t0:.0f}",
                    "url": self.base_url,
                    "models": model_ids,
                    "cuda_visible_devices": self.cuda_visible_devices,
                    "max_model_len": self.max_model_len,
                })
                return
            except Exception:
                time.sleep(2)
        self.log_handle.flush()
        logs = self.log_path.read_text(encoding="utf-8", errors="ignore")[-12000:]
        raise RuntimeError(f"vLLM timeout for {self.role_name}:\n{logs}")

    def stop(self):
        if self.process and self.process.poll() is None:
            self.process.terminate()
            try:
                self.process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                self.process.kill()
        if self.log_handle:
            self.log_handle.close()

SERVER = None
CLIENT = None
ENCODING = None
STOP_TOKENS = None
SERVER_REGISTRY = {}
CLIENT_REGISTRY = {}
MODEL_ROLE_FAILURES = set()
THINK_TAG_RE = re.compile(r"<think>.*?</think>", re.IGNORECASE | re.DOTALL)
NOTEBOOK_START = time.time(); PROBLEMS_REMAINING = 50

def sanitize_model_response_text(text, family):
    text = coerce_text(text)
    family = str(family or "generic")
    if family in {"qwen", "deepseek"}:
        text = THINK_TAG_RE.sub("", text)
    return text.strip()

def build_model_chat_messages(model_spec, purpose, system_prompt, user_content):
    family = str((model_spec or {}).get("family") or "generic")
    system_bits = [str(system_prompt or "").strip()]
    if family in {"qwen", "deepseek"}:
        system_bits.append(
            "Do not emit <think> tags. "
            "When returning the final JSON answer, do not wrap it in markdown fences."
        )
    else:
        system_bits.append("When returning the final JSON answer, do not wrap it in markdown fences.")
    if purpose == "verifier":
        system_bits.append("Never introduce any answer that is not already in the listed candidate set.")
    merged_system = "\n\n".join(part for part in system_bits if part)
    strict_user = str(user_content or "").strip()
    if purpose == "worker":
        strict_user += (
            "\n\nOutput rule: if you need Python execution next, return one ```python``` block only. "
            "If you are finalizing, return only the JSON object with no markdown fences."
        )
    elif purpose in {"planner", "candidate", "verifier"}:
        strict_user += "\n\nStrict output rule: return only the requested JSON object with no markdown fences."
    return [
        {"role": "system", "content": merged_system},
        {"role": "user", "content": strict_user},
    ]

def detect_cuda_runtime():
    state = {
        "runtime": RUNTIME_NAME,
        "cuda_visible_devices": os.getenv("CUDA_VISIBLE_DEVICES"),
        "nvidia_smi": shutil.which("nvidia-smi"),
        "libcuda_path": None,
        "torch_cuda": False,
        "torch_device_count": 0,
        "torch_error": None,
    }
    for candidate in (
        Path("/usr/lib/x86_64-linux-gnu/libcuda.so.1"),
        Path("/usr/lib/wsl/lib/libcuda.so.1"),
        Path("/usr/lib64/libcuda.so.1"),
    ):
        if candidate.exists():
            state["libcuda_path"] = str(candidate)
            break
    try:
        import torch

        state["torch_cuda"] = bool(torch.cuda.is_available())
        state["torch_device_count"] = int(torch.cuda.device_count()) if state["torch_cuda"] else 0
    except Exception as exc:
        state["torch_error"] = f"{type(exc).__name__}: {str(exc)[:160]}"
    return state

def build_model_runtime_specs():
    specs = {
        "primary": {
            "role": "primary",
            "model_path": MODEL_PATH,
            "family": MODEL_FAMILY,
            "served_model_name": PRIMARY_REQUEST_MODEL_NAME,
            "request_model_name": PRIMARY_REQUEST_MODEL_NAME,
            "adapter_path": ADAPTER_PATH,
            "hint": CFG.primary_model_hint,
            "port": CFG.primary_server_port,
            "cuda_visible_devices": CFG.primary_cuda_visible_devices or CFG.cuda_visible_devices,
            "gpu_memory_utilization": CFG.gpu_memory_utilization,
            "max_model_len": PRIMARY_MAX_MODEL_LEN,
        }
    }
    if CFG.enable_aux_models and CONTRARIAN_MODEL_PATH is not None:
        specs["contrarian"] = {
            "role": "contrarian",
            "model_path": CONTRARIAN_MODEL_PATH,
            "family": CONTRARIAN_MODEL_FAMILY,
            "served_model_name": CONTRARIAN_REQUEST_MODEL_NAME,
            "request_model_name": CONTRARIAN_REQUEST_MODEL_NAME,
            "adapter_path": None,
            "hint": CONTRARIAN_MODEL_HINTS[0] if CONTRARIAN_MODEL_HINTS else "contrarian",
            "port": CFG.contrarian_server_port,
            "cuda_visible_devices": CFG.contrarian_cuda_visible_devices or None,
            "gpu_memory_utilization": CFG.aux_gpu_memory_utilization,
            "max_model_len": CONTRARIAN_MAX_MODEL_LEN,
        }
    if CFG.enable_aux_models and VERIFIER_MODEL_PATH is not None:
        specs["verifier"] = {
            "role": "verifier",
            "model_path": VERIFIER_MODEL_PATH,
            "family": VERIFIER_MODEL_FAMILY,
            "served_model_name": VERIFIER_REQUEST_MODEL_NAME,
            "request_model_name": VERIFIER_REQUEST_MODEL_NAME,
            "adapter_path": None,
            "hint": CFG.verifier_model_hint,
            "port": CFG.verifier_server_port,
            "cuda_visible_devices": CFG.verifier_cuda_visible_devices or None,
            "gpu_memory_utilization": CFG.verifier_gpu_memory_utilization,
            "max_model_len": VERIFIER_MAX_MODEL_LEN,
        }
    return specs

MODEL_RUNTIME_SPECS = build_model_runtime_specs()

def summarize_model_routing():
    return {
        role: {
            "family": spec["family"],
            "port": spec["port"],
            "request_model_name": spec["request_model_name"],
            "cuda_visible_devices": spec["cuda_visible_devices"],
            "model_path": str(spec["model_path"]),
            "max_model_len": spec.get("max_model_len"),
        }
        for role, spec in MODEL_RUNTIME_SPECS.items()
    }

def resolve_model_role(role="primary"):
    role = str(role or "primary")
    if role in MODEL_ROLE_FAILURES or role not in MODEL_RUNTIME_SPECS:
        return "primary"
    return role

def get_model_runtime_spec(role="primary"):
    return MODEL_RUNTIME_SPECS[resolve_model_role(role)]

def require_cuda_runtime_for_vllm(role="primary"):
    state = detect_cuda_runtime()
    if state["torch_cuda"] or state["nvidia_smi"] or state["libcuda_path"]:
        return state
    fit_hint = ""
    spec = get_model_runtime_spec(role)
    if "120b" in str(spec.get("hint") or "").lower():
        fit_hint = " Current model hint is a 120b-scale model, which is also unlikely to fit standard Kaggle notebook GPU memory."
    raise RuntimeError(
        "CUDA runtime is unavailable for vLLM. This harness requires a GPU-enabled runtime. "
        "On Kaggle, set Notebook Accelerator=GPU and save a new notebook version before submitting."
        + fit_hint
        + " Debug="
        + json.dumps(state, ensure_ascii=True)
    )

def maybe_load_harmony_state(spec):
    global ENCODING, STOP_TOKENS
    if str(spec.get("family") or "") == "gpt_oss":
        ENCODING = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        STOP_TOKENS = ENCODING.stop_tokens_for_assistant_actions()
    else:
        ENCODING = None
        STOP_TOKENS = []

def ensure_inference_ready(role="primary"):
    server = start_frontier_server(force_restart=False, role=role)
    resolved_role = getattr(server, "role_name", resolve_model_role(role))
    client = CLIENT_REGISTRY.get(resolved_role) or CLIENT_REGISTRY.get("primary")
    spec = get_model_runtime_spec(resolved_role)
    if client is None:
        raise RuntimeError(f"Inference runtime is not ready for role={resolved_role}.")
    if resolved_role == "primary" or client is CLIENT_REGISTRY.get("primary"):
        global CLIENT, SERVER
        CLIENT = CLIENT_REGISTRY.get("primary", client)
        SERVER = SERVER_REGISTRY.get("primary", SERVER)
    return client, spec

def start_frontier_server(force_restart=False, role="primary"):
    global SERVER, CLIENT, NOTEBOOK_START
    resolved_role = resolve_model_role(role)
    spec = get_model_runtime_spec(resolved_role)
    cuda_state = require_cuda_runtime_for_vllm(resolved_role)
    if resolved_role == "primary" and "120b" in str(spec.get("hint") or "").lower():
        print({
            "model_warning": "120b-scale models may exceed standard Kaggle notebook GPU memory; if startup later fails with OOM, use a smaller or quantized model.",
            "cuda_state": cuda_state,
        })
    existing = SERVER_REGISTRY.get(resolved_role)
    if (
        not force_restart
        and existing is not None
        and existing.process is not None
        and existing.process.poll() is None
        and CLIENT_REGISTRY.get(resolved_role) is not None
    ):
        if resolved_role == "primary":
            SERVER = existing
            CLIENT = CLIENT_REGISTRY.get("primary")
            maybe_load_harmony_state(spec)
        print({"vllm": "already_running", "role": resolved_role, "url": existing.base_url, "model_path": str(spec["model_path"])})
        return existing
    if existing is not None:
        stop_frontier_server(role=resolved_role)
    cleanup = not SERVER_REGISTRY
    server = VLLMServer(
        resolved_role,
        spec["model_path"],
        spec["served_model_name"],
        spec["port"],
        cuda_visible_devices=spec.get("cuda_visible_devices"),
        adapter_path=spec.get("adapter_path"),
        gpu_memory_utilization=spec.get("gpu_memory_utilization"),
        max_model_len=spec.get("max_model_len"),
    )
    try:
        server.start(cleanup=cleanup)
    except Exception as exc:
        if resolved_role != "primary":
            MODEL_ROLE_FAILURES.add(resolved_role)
            print({
                "model_role": resolved_role,
                "status": "fallback_to_primary",
                "error": f"{type(exc).__name__}: {str(exc)[:240]}",
            })
            return start_frontier_server(force_restart=False, role="primary")
        raise
    SERVER_REGISTRY[resolved_role] = server
    CLIENT_REGISTRY[resolved_role] = server.client
    if resolved_role == "primary":
        SERVER = server
        CLIENT = server.client
        maybe_load_harmony_state(spec)
        NOTEBOOK_START = time.time()
    return server

def stop_frontier_server(role=None):
    global SERVER, CLIENT, ENCODING, STOP_TOKENS
    if role is None:
        roles = list(SERVER_REGISTRY.keys())
    else:
        roles = [resolve_model_role(role)]
    for current_role in roles:
        server = SERVER_REGISTRY.pop(current_role, None)
        CLIENT_REGISTRY.pop(current_role, None)
        if server is not None:
            try:
                server.stop()
            except Exception:
                pass
    SERVER = SERVER_REGISTRY.get("primary")
    CLIENT = CLIENT_REGISTRY.get("primary")
    if SERVER is None:
        ENCODING = None
        STOP_TOKENS = None
    print({"vllm": "stopped", "role": role or "all"})

def reset_notebook_budget(problem_count=50):
    global NOTEBOOK_START, PROBLEMS_REMAINING
    NOTEBOOK_START = time.time()
    PROBLEMS_REMAINING = int(problem_count)
    summary = {"note": "budget_reset", "problems_remaining": PROBLEMS_REMAINING, "started_at": NOTEBOOK_START}
    print(summary)
    return summary



## 7. Attempt Result Model
Structured per-attempt state including certificates, assumptions, and tool artifacts.

In [ ]:
@dataclass
class AttemptResult:
    answer: int | None
    weight: float
    entropy: float
    attempt_index: int
    seed: int
    python_calls: int
    final_text: str
    error: str | None = None
    tool_confused: bool = False
    approx_only: bool = False
    random_search: bool = False
    weak_exactness: bool = False
    missing_extremal_certificate: bool = False
    route: str = "generic"
    method: str = "proof"
    certificate_type: str = "none"
    certificate_summary: str = ""
    certificate_payload: tuple = ()
    assumptions: tuple = ()
    self_checks: tuple = ()
    resource_use: dict | None = None

def make_attempt_result(answer, weight, entropy, attempt_index, seed, python_calls, final_text, route, error=None, tool_artifacts=None):
    tool_confused = has_tool_confusion(final_text)
    exactness_penalty, exactness_flags = analyze_reasoning_quality(final_text, route, python_calls)
    tool_artifacts = tuple(tool_artifacts or ())
    certificate_type, certificate_summary = classify_certificate(final_text, route, tool_artifacts)
    method = "hybrid" if python_calls else "proof"
    if python_calls and certificate_type in MACHINE_CHECKABLE_CERTIFICATES:
        weight *= 1.10
    if answer is not None and tool_confused:
        weight *= CFG.tool_confusion_weight
    if answer is not None and CFG.enable_exactness_penalties:
        weight *= exactness_penalty
    return AttemptResult(
        answer,
        weight,
        entropy,
        attempt_index,
        seed,
        python_calls,
        final_text,
        error,
        tool_confused,
        exactness_flags["approx_only"],
        exactness_flags["random_search"],
        exactness_flags["weak_exactness"],
        exactness_flags["missing_extremal_certificate"],
        route,
        method,
        certificate_type,
        certificate_summary,
        tool_artifacts,
        extract_assumptions(final_text),
        summarize_self_checks(final_text, tool_artifacts),
        {
            "python_calls": int(python_calls),
            "mean_entropy": round(float(entropy), 4),
            "tool_artifacts": len(tool_artifacts),
        },
    )


## 8. Chat Worker
Alternative chat-style solver path and tool loop primitives.

In [ ]:

# JSON-first chat fallback. This path is intentionally simpler than the Harmony path.

CHAT_CANDIDATE_SYSTEM_PROMPT = """Reasoning: high

You are an olympiad-math solver.
Return ONLY one JSON object with these keys:
{
  "answer": <integer 0..99999 or null>,
  "status": "solved" | "partial" | "stuck",
  "method": "proof_first" | "pot_python" | "hybrid",
  "proof_sketch": "<concise but rigorous reasoning>",
  "certificate_type": "none" | "counterexample_check" | "optimization_witness" | "equivalence_check" | "exhaustive_check" | "formal_proof",
  "certificate_summary": "<what was verified>",
  "assumptions": ["..."],
  "self_checks": ["..."],
  "final_statement": "<one sentence>"
}

Rules:
- Do not use markdown.
- Do not include prose before or after the JSON object.
- If unsure, set answer to null and status to "partial" or "stuck".
- No guessing.
"""

def coerce_text(content):
    if isinstance(content, str):
        return content
    if content is None:
        return ""
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and item.get("type") == "text":
                parts.append(str(item.get("text", "")))
            elif hasattr(item, "text"):
                parts.append(str(item.text))
            else:
                parts.append(str(item))
        return "\n".join(p for p in parts if p)
    return str(content)

def chat_attempt(problem_text, attempt_index, route):
    ensure_inference_ready()
    seed = CFG.startup_seed + attempt_index * 100
    temp = max(0.2, attempt_temperature(attempt_index) - 0.15)
    attempt_style = route_attempt_stance(route, attempt_index)
    prompt = (
        problem_text.strip()
        + "\n\n"
        + domain_strategy_hint(problem_text)
        + "\n\nAttempt style:\n"
        + attempt_style
        + "\n\nReturn JSON only."
    )
    try:
        client, model_spec = ensure_inference_ready(role="primary")
        response = client.chat.completions.create(
            model=model_spec["request_model_name"],
            messages=build_model_chat_messages(model_spec, "candidate", CHAT_CANDIDATE_SYSTEM_PROMPT, prompt),
            temperature=temp,
            top_p=0.95,
            max_tokens=6144,
            seed=seed,
        )
        if not response.choices:
            return AttemptResult(None, 0.0, 1.0, attempt_index, seed, 0, "", "chat_no_choices")
        msg = response.choices[0].message
        text = sanitize_model_response_text(msg.content, model_spec["family"])
        obj = extract_json_object(text)
        if obj is None:
            ans = extract_answer(text)
            if ans is not None:
                payload = {
                    "answer": ans,
                    "status": "partial",
                    "method": "proof_first",
                    "proof_sketch": clip_text(text, 1200),
                    "certificate_type": "none",
                    "certificate_summary": "Recovered from non-JSON chat response.",
                    "assumptions": [],
                    "self_checks": [],
                    "final_statement": "Recovered fallback answer from raw text.",
                }
                ok, err = validate_candidate_payload(payload)
                if ok:
                    return make_attempt_result(payload["answer"], 0.45, 1.0, attempt_index, seed, 0, text, route, tool_artifacts=[])
            return AttemptResult(None, 0.0, 1.0, attempt_index, seed, 0, text, "chat_bad_json")
        ok, err = validate_candidate_payload(obj)
        if not ok:
            return AttemptResult(None, 0.0, 1.0, attempt_index, seed, 0, text, f"chat_schema:{err}")
        answer = obj["answer"]
        weight = 0.75 if answer is not None else 0.0
        return make_attempt_result(answer, weight, 1.0, attempt_index, seed, 0, text, route, tool_artifacts=[])
    except Exception as exc:
        return AttemptResult(None, 0.0, 1.0, attempt_index, seed, 0, "", f"chat:{type(exc).__name__}: {str(exc)[:160]}")


## 9. Harmony Worker
Harmony-format worker construction and multi-turn tool execution.

In [ ]:

# Harmony solver with contract-first prompting and deterministic python certificates.

HARMONY_FINAL_JSON_SPEC = """Return ONLY one JSON object in the final channel with these keys:
{
  "answer": <integer 0..99999 or null>,
  "status": "solved" | "partial" | "stuck",
  "method": "proof_first" | "pot_python" | "hybrid",
  "proof_sketch": "<concise rigorous reasoning>",
  "certificate_type": "none" | "counterexample_check" | "optimization_witness" | "equivalence_check" | "exhaustive_check" | "formal_proof",
  "certificate_summary": "<what exact check was completed>",
  "assumptions": ["..."],
  "self_checks": ["..."],
  "final_statement": "<one sentence>"
}
No markdown. No boxed answer. No extra text.
"""

def make_system_message():
    content = SystemContent.new().with_reasoning_effort(ReasoningEffort.HIGH).with_conversation_start_date(CFG.current_date)
    try:
        content = content.with_python()
    except Exception:
        pass
    return Message.from_role_and_content(Role.SYSTEM, content)

def make_developer_message(persona, route, attempt_index):
    route_block = route_prompt(route)
    attempt_block = route_attempt_stance(route, attempt_index)
    inst = f"""# Instructions
Solve the olympiad problem exactly.
Use python only when it reduces risk.
When sending python:
- send executable code to the python recipient
- keep it deterministic, concise, and assert-heavy
- use exact arithmetic when possible
- the last printed non-empty line should be either a JSON object {{"final_answer": <int>}} or a single integer
- do not use randomness, simulation-only evidence, or floating optimization as final justification

Prefer:
- constructive witness + matching bound for optimization problems
- symbolic or exhaustive checks when possible
- explicit rejection of unsupported guesses

# Problem lens
{route_block}

# Attempt style
{attempt_block}

{HARMONY_FINAL_JSON_SPEC}
"""
    if persona:
        inst += "\n# Solver stance\n" + persona.strip() + "\n"
    return Message.from_role_and_content(Role.DEVELOPER, DeveloperContent.new().with_instructions(inst))

def make_user_message(problem_text, route):
    hint = domain_strategy_hint(problem_text)
    prompt = (
        problem_text.strip()
        + "\n\nDomain strategy hint (general, not an answer):\n"
        + hint
        + "\n\nRemember: if you use python, wait for the observation, then return final JSON in the final channel."
    )
    return Message.from_role_and_content(Role.USER, prompt)

def completion_tokens(prompt_ids, temperature, max_tokens, seed):
    ensure_inference_ready()
    response = CLIENT.completions.create(
        model=REQUEST_MODEL_NAME,
        prompt=prompt_ids,
        max_tokens=max_tokens,
        temperature=temperature,
        logprobs=CFG.top_logprobs,
        seed=seed,
        extra_body={"min_p": CFG.min_p, "stop_token_ids": STOP_TOKENS, "return_token_ids": True},
    )
    if not response.choices:
        return [], "", 1.0
    ch = response.choices[0]
    return get_choice_token_ids(ch), (getattr(ch, "text", "") or ""), mean_entropy_from_logprobs(getattr(ch, "logprobs", None))

def message_text(message):
    content = getattr(message, "content", None)
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    return "\n".join(str(getattr(x, "text", x)) for x in content)

def candidate_from_payload(payload, attempt_index, seed, py_calls, final_text, route, entropy, tool_artifacts, error=None):
    return make_attempt_result(
        payload.get("answer"),
        (1.0 / max(entropy, 0.08)) if payload.get("answer") is not None else 0.0,
        entropy,
        attempt_index,
        seed,
        py_calls,
        final_text,
        route,
        error=error,
        tool_artifacts=tool_artifacts,
    )

def run_harmony_attempt(problem_text, attempt_index, route):
    persona = CFG.personas[attempt_index % len(CFG.personas)] if CFG.use_personas else ""
    seed = int((CFG.startup_seed + attempt_index + 1) ** 2)
    temp = attempt_temperature(attempt_index)
    sandbox = SANDBOX_POOL.acquire()
    py_calls = 0
    final_texts = []
    entropies = []
    tool_artifacts = []
    messages = [make_system_message(), make_developer_message(persona, route, attempt_index), make_user_message(problem_text, route)]
    try:
        for turn in range(CFG.tool_rounds + 1):
            ids = ENCODING.render_conversation_for_completion(Conversation.from_messages(messages), Role.ASSISTANT)
            max_available = max(512, CFG.max_model_len - len(ids) - 128)
            turn_max = min(CFG.max_tokens, max_available)
            if turn_max < 512:
                break
            token_ids, raw_text, entropy = completion_tokens(ids, temp, turn_max, seed + turn)
            entropies.append(entropy)
            if not token_ids:
                ans = extract_answer(raw_text)
                return make_attempt_result(ans, (1.0 / max(entropy, 0.08)) if ans is not None else 0.0, entropy, attempt_index, seed, py_calls, raw_text, route, None if ans is not None else "no_tokens", tool_artifacts=tool_artifacts)
            parsed = ENCODING.parse_messages_from_completion_tokens(token_ids, Role.ASSISTANT)
            if not parsed:
                ans = extract_answer(raw_text)
                return make_attempt_result(ans, (1.0 / max(entropy, 0.08)) if ans is not None else 0.0, entropy, attempt_index, seed, py_calls, raw_text, route, None if ans is not None else "parse_empty", tool_artifacts=tool_artifacts)

            messages.extend(parsed)
            last = parsed[-1]
            text = message_text(last)
            final_texts.append(text)

            if getattr(last, "recipient", None) == "python":
                py_calls += 1
                output = sandbox.execute(text, timeout=CFG.python_timeout_seconds)
                tool_artifacts.append(build_tool_artifact(text, output))
                messages.append(Message.from_author_and_content(Author.new(Role.TOOL, "python"), output).with_channel("commentary"))
                continue

            channel = str(getattr(last, "channel", "") or "")
            combined_text = text if channel == "final" else ("\n".join(final_texts[-2:]) or raw_text)
            payload = extract_json_object(combined_text)
            if payload is not None:
                ok, err = validate_candidate_payload(payload)
                if ok:
                    mean_e = sum(entropies) / max(1, len(entropies))
                    weight = 1.0 / max(mean_e, 0.08)
                    if py_calls:
                        weight *= 1.12
                    return make_attempt_result(payload["answer"], weight if payload["answer"] is not None else 0.0, mean_e, attempt_index, seed, py_calls, combined_text, route, tool_artifacts=tool_artifacts)
                if channel == "final":
                    return AttemptResult(None, 0.0, entropy, attempt_index, seed, py_calls, combined_text, f"bad_final_json:{err}")

        blob = "\n".join(final_texts)
        payload = extract_json_object(blob)
        mean_e = sum(entropies) / max(1, len(entropies)) if entropies else 1.0
        if payload is not None:
            ok, err = validate_candidate_payload(payload)
            if ok:
                return make_attempt_result(payload["answer"], (0.8 / max(mean_e, 0.1)) if payload["answer"] is not None else 0.0, mean_e, attempt_index, seed, py_calls, blob, route, tool_artifacts=tool_artifacts)
            return AttemptResult(None, 0.0, mean_e, attempt_index, seed, py_calls, blob, f"final_json:{err}")
        ans = extract_answer(blob)
        return make_attempt_result(ans, (0.6 / max(mean_e, 0.1)) if ans is not None else 0.0, mean_e, attempt_index, seed, py_calls, blob, route, None if ans is not None else "no_answer", tool_artifacts=tool_artifacts)
    except Exception as e:
        return AttemptResult(None, 0.0, 1.0, attempt_index, seed, py_calls, "", f"{type(e).__name__}: {str(e)[:200]}")
    finally:
        SANDBOX_POOL.release(sandbox)


## 10. Solver Orchestration
Candidate aggregation, verification, final solving, and prediction entry points.

In [ ]:

def aggregate_candidate_stats(results):
    scores, counts = defaultdict(float), Counter()
    entropies, tools, confused = defaultdict(list), Counter(), Counter()
    approx_only, random_search = Counter(), Counter()
    weak_exactness, missing_extremal = Counter(), Counter()
    best_attempt_by_answer = {}
    errors = []
    for r in results:
        if r.error:
            errors.append(r.error)
        if r.answer is None:
            continue
        a = int(r.answer)
        counts[a] += 1
        scores[a] += float(r.weight)
        entropies[a].append(float(r.entropy))
        tools[a] += int(r.python_calls > 0)
        confused[a] += int(r.tool_confused)
        approx_only[a] += int(r.approx_only)
        random_search[a] += int(r.random_search)
        weak_exactness[a] += int(r.weak_exactness)
        missing_extremal[a] += int(r.missing_extremal_certificate)
        current = best_attempt_by_answer.get(a)
        if current is None or r.weight > current.weight:
            best_attempt_by_answer[a] = r
    if not scores:
        return [], {
            "scores": {}, "counts": {}, "tools": {}, "tool_confused": {},
            "approx_only": {}, "random_search": {}, "weak_exactness": {},
            "missing_extremal_certificate": {}, "entropy": {}, "errors": errors[:3],
        }, best_attempt_by_answer
    ranked = sorted(scores.items(), key=lambda x: (x[1], counts[x[0]], tools[x[0]]), reverse=True)
    details = {
        "scores": {int(k): round(v, 3) for k, v in ranked[:5]},
        "counts": {int(k): int(counts[k]) for k, _ in ranked[:5]},
        "tools": {int(k): int(tools[k]) for k, _ in ranked[:5]},
        "tool_confused": {int(k): int(confused[k]) for k, _ in ranked[:5]},
        "approx_only": {int(k): int(approx_only[k]) for k, _ in ranked[:5]},
        "random_search": {int(k): int(random_search[k]) for k, _ in ranked[:5]},
        "weak_exactness": {int(k): int(weak_exactness[k]) for k, _ in ranked[:5]},
        "missing_extremal_certificate": {int(k): int(missing_extremal[k]) for k, _ in ranked[:5]},
        "entropy": {int(k): round(sum(entropies[k]) / max(1, len(entropies[k])), 3) for k, _ in ranked[:5]},
        "certificate_type": {int(k): best_attempt_by_answer[k].certificate_type for k, _ in ranked[:5] if k in best_attempt_by_answer},
        "certificate_summary": {int(k): clip_text(best_attempt_by_answer[k].certificate_summary, 180) for k, _ in ranked[:5] if k in best_attempt_by_answer},
        "machine_checkable": {
            int(k): is_machine_checkable_attempt(best_attempt_by_answer[k], int(k), int(tools[k]))
            for k, _ in ranked[:5]
            if k in best_attempt_by_answer
        },
        "errors": errors[:3],
    }
    return ranked, details, best_attempt_by_answer

def deterministic_candidate_selection(ranked, details, best_attempt_by_answer):
    verdicts = []
    for answer, score in ranked[: max(CFG.verifier_max_candidates, 5)]:
        answer = int(answer)
        attempt = best_attempt_by_answer.get(answer)
        if attempt is None:
            continue
        verdicts.append({
            "answer": answer,
            "score": float(score),
            "votes": int(details["counts"].get(answer, 0)),
            "certificate_type": attempt.certificate_type,
            "certificate_rank": certificate_rank(attempt.certificate_type),
            "machine_checkable": is_machine_checkable_attempt(attempt, answer, int(details["tools"].get(answer, 0))),
            "clean_reasoning": not (attempt.tool_confused or attempt.approx_only or attempt.random_search),
            "summary": attempt.certificate_summary,
        })
    if not verdicts:
        return None, {"candidates": []}
    machine = [item for item in verdicts if item["machine_checkable"] and item["clean_reasoning"]]
    machine = sorted(machine, key=lambda item: (item["certificate_rank"], item["votes"], item["score"]), reverse=True)
    if machine:
        winner = int(machine[0]["answer"])
        return winner, {"candidates": verdicts, "mode": "machine_checkable", "winner": winner, "locked": False}
    clean = [item for item in verdicts if item["clean_reasoning"]]
    clean = sorted(clean, key=lambda item: (item["votes"], item["score"], item["certificate_rank"]), reverse=True)
    if clean and (len(clean) == 1 or clean[0]["votes"] >= max(2, clean[min(1, len(clean)-1)]["votes"] + 1)):
        winner = int(clean[0]["answer"])
        runner_up_votes = int(clean[1]["votes"]) if len(clean) > 1 else 0
        top_score = float(clean[0]["score"])
        runner_up_score = float(clean[1]["score"]) if len(clean) > 1 else 0.0
        strong_lock = (
            int(clean[0]["votes"]) >= CFG.strong_majority_min_votes
            and int(clean[0]["votes"]) >= runner_up_votes + CFG.strong_majority_vote_margin
            and (runner_up_score <= 0.0 or runner_up_score <= CFG.strong_majority_score_ratio * max(top_score, 1e-9))
        )
        return winner, {"candidates": verdicts, "mode": "clean_majority", "winner": winner, "locked": strong_lock}
    return None, {"candidates": verdicts, "mode": "needs_verifier", "winner": None, "locked": False}

def verifier_candidates_from_ranked(ranked):
    return [int(answer) for answer, _ in ranked[: CFG.verifier_max_candidates]]

def build_candidate_briefs(candidates, details, best_attempt_by_answer):
    briefs = []
    for answer in candidates:
        attempt = best_attempt_by_answer.get(int(answer))
        if attempt is None:
            continue
        briefs.append({
            "answer": int(answer),
            "score": float(details["scores"].get(int(answer), 0.0)),
            "votes": int(details["counts"].get(int(answer), 0)),
            "certificate_type": attempt.certificate_type,
            "certificate_summary": attempt.certificate_summary,
            "tool_confused": bool(attempt.tool_confused),
            "approx_only": bool(attempt.approx_only),
            "random_search": bool(attempt.random_search),
            "proof_sketch": clip_text(attempt.final_text, 1800),
        })
    return briefs

VERIFIER_SYSTEM_PROMPT = """Reasoning: high

You are an adversarial olympiad verifier.
You must choose ONLY from the listed candidate answers.
Return ONLY one JSON object:
{
  "selected_answer": <candidate integer or null>,
  "status": "verified" | "inconclusive" | "rejected_all",
  "reason": "<short explanation>",
  "reject_reasons": ["..."]
}

Rules:
- Prefer machine-checkable certificates over vote counts.
- Reject answers supported only by approximation, random search, or unsupported pattern matching.
- Never invent a new answer.
- No markdown.
"""

def run_candidate_verifier(problem_text, route, strategy_hint, ranked, details, results, best_attempt_by_answer):
    ensure_inference_ready()
    candidates = verifier_candidates_from_ranked(ranked)
    if len(candidates) < 2:
        return None
    briefs = build_candidate_briefs(candidates, details, best_attempt_by_answer)
    prompt = (
        problem_text.strip()
        + "\n\nProblem lens:\n"
        + route_prompt(route)
        + ("\n\nStrategy hint:\n" + strategy_hint if strategy_hint else "")
        + "\n\nCandidate artifacts to verify:\n"
        + json.dumps(briefs, ensure_ascii=False, indent=2)
        + "\n\nSolve from scratch and choose only from the listed candidates."
    )
    try:
        response = CLIENT.chat.completions.create(
            model=REQUEST_MODEL_NAME,
            messages=[{"role": "system", "content": VERIFIER_SYSTEM_PROMPT}, {"role": "user", "content": prompt}],
            temperature=CFG.verifier_temperature,
            top_p=0.95,
            max_tokens=CFG.verifier_max_tokens,
            seed=CFG.startup_seed + 7000,
        )
        if not response.choices:
            return {"candidates": candidates, "answer": None, "text": "", "error": "verifier_no_choices"}
        text = coerce_text(response.choices[0].message.content)
        obj = extract_json_object(text)
        if obj is None:
            return {"candidates": candidates, "answer": None, "text": text, "error": "verifier_bad_json"}
        ok, err = validate_verifier_payload(obj, candidates)
        if not ok:
            return {"candidates": candidates, "answer": None, "text": text, "error": err}
        return {"candidates": candidates, "answer": obj["selected_answer"], "text": text, "error": None}
    except Exception as exc:
        return {"candidates": candidates, "answer": None, "text": "", "error": f"verifier:{type(exc).__name__}: {str(exc)[:160]}"}

def choose_weighted_winner(ranked, details, best_attempt_by_answer):
    if not ranked:
        return None
    deterministic_answer, deterministic_summary = deterministic_candidate_selection(ranked, details, best_attempt_by_answer)
    details["deterministic"] = deterministic_summary
    if deterministic_answer is not None:
        return int(deterministic_answer)
    counts = details["counts"]
    tools = details["tools"]
    winner = int(ranked[0][0])
    if len(ranked) > 1:
        top_score = float(ranked[0][1])
        for cand, sc in ranked[1:5]:
            cand = int(cand)
            winner_attempt = best_attempt_by_answer.get(winner)
            cand_attempt = best_attempt_by_answer.get(cand)
            winner_cert_rank = certificate_rank(None if winner_attempt is None else winner_attempt.certificate_type)
            cand_cert_rank = certificate_rank(None if cand_attempt is None else cand_attempt.certificate_type)
            if cand_cert_rank > winner_cert_rank and sc >= 0.75 * top_score:
                winner = cand
                break
            if counts.get(cand, 0) > counts.get(winner, 0) and sc >= CFG.weak_leader_support_ratio * top_score:
                winner = cand
                break
            if tools.get(cand, 0) > tools.get(winner, 0) and counts.get(cand, 0) >= counts.get(winner, 0) and sc >= CFG.tool_supported_margin_ratio * top_score:
                winner = cand
                break
    return int(winner)

def should_run_verifier(route, ranked, details):
    if not CFG.verifier_enabled or len(ranked) < 2:
        return False
    det = details.get("deterministic") or {}
    top_score = float(ranked[0][1])
    second_score = float(ranked[1][1])
    winner = int(ranked[0][0])
    close_race = second_score >= CFG.verifier_close_score_ratio * top_score
    weak_support = details["counts"].get(winner, 0) <= 2
    no_machine_certificate = not details.get("machine_checkable", {}).get(winner, False)
    exactness_risk = bool(
        details["approx_only"].get(winner, 0)
        or details["random_search"].get(winner, 0)
        or details["weak_exactness"].get(winner, 0)
        or details["missing_extremal_certificate"].get(winner, 0)
    )
    locked_clean_majority = bool(det.get("locked")) and det.get("mode") == "clean_majority"
    if locked_clean_majority and not exactness_risk:
        return False
    broad_pool = len(ranked) >= 4 and float(ranked[min(3, len(ranked) - 1)][1]) >= 0.40 * top_score
    return close_race or (weak_support and broad_pool) or exactness_risk or (route_prefers_verifier(route) and no_machine_certificate)

def fallback_submission_answer(problem_text, problem_id=None, route=None, reason="unresolved"):
    fallback = None
    write_debug_event({
        "event": "submission_fallback",
        "problem_id": problem_id,
        "route": route,
        "reason": reason,
        "fallback_answer": fallback,
        "problem_excerpt": clip_text(problem_text, 240),
    })
    return fallback

def coerce_submission_int(answer):
    if answer is None:
        return 0
    return int(answer)

def summarize_attempt_result(attempt):
    text = getattr(attempt, "text", "") or ""
    error = getattr(attempt, "error", None)
    answer = getattr(attempt, "answer", None)
    python_calls = int(getattr(attempt, "python_calls", 0) or 0)

    if answer is not None:
        parse_status = "ok"
    elif error:
        err = str(error).lower()
        if "json" in err:
            parse_status = "bad_json"
        elif "parse" in err:
            parse_status = "parse_failure"
        elif "timeout" in err:
            parse_status = "timeout"
        elif "tool_confusion" in err:
            parse_status = "tool_confusion"
        elif "future:" in err or "chat_future:" in err:
            parse_status = "worker_exception"
        else:
            parse_status = "error"
    elif text.strip():
        parse_status = "no_answer_found"
    else:
        parse_status = "empty_output"

    lower = text.lower()
    if python_calls > 0:
        if "sympy" in lower or "symbolic" in lower:
            certificate_type = "symbolic"
        elif "counterexample" in lower:
            certificate_type = "counterexample_check"
        elif "enumerate" in lower or "search" in lower:
            certificate_type = "python_enum"
        else:
            certificate_type = "python_check"
    elif "construction" in lower or "attained" in lower or "equality case" in lower:
        certificate_type = "construction"
    elif "lower bound" in lower or "upper bound" in lower or "at least" in lower or "at most" in lower:
        certificate_type = "bound_argument"
    else:
        certificate_type = "none"

    worker_role = getattr(attempt, "worker_role", None) or "unknown"

    return {
        "attempt": getattr(attempt, "attempt_index", None),
        "seed": getattr(attempt, "seed", None),
        "answer": answer,
        "weight": getattr(attempt, "weight", None),
        "entropy": getattr(attempt, "entropy", None),
        "python_calls": python_calls,
        "error": error,
        "parse_status": parse_status,
        "certificate_type": certificate_type,
        "worker_role": worker_role,
        "model_role": getattr(attempt, "model_role", "primary"),
        "model_family": getattr(attempt, "model_family", "unknown"),
        "text_tail": clip_text(text, CFG.debug_text_chars),
    }

def chat_solve_problem(problem_text, problem_id=None, route=None):
    global PROBLEMS_REMAINING
    started = time.time()
    route = route or classify_problem_route(problem_text)
    results = []
    force_full = problem_requires_full_attempts(problem_text, route=route)
    strategy_hint = domain_strategy_hint(problem_text)
    with ThreadPoolExecutor(max_workers=CFG.workers) as ex:
        futures = {ex.submit(chat_attempt, problem_text, i, route): i for i in range(CFG.attempts)}
        for fut in as_completed(futures):
            try:
                r = fut.result(timeout=CFG.request_timeout_seconds)
            except Exception as exc:
                r = AttemptResult(None, 0.0, 1.0, futures[fut], 0, 0, "", f"chat_future:{type(exc).__name__}")
            results.append(r)
            counts = Counter(x.answer for x in results if x.answer is not None)
            if (not force_full) and counts and counts.most_common(1)[0][1] >= CFG.early_stop_votes:
                break
    PROBLEMS_REMAINING = max(0, PROBLEMS_REMAINING - 1)
    ranked, details, best_attempt_by_answer = aggregate_candidate_stats(results)
    answer = choose_weighted_winner(ranked, details, best_attempt_by_answer)
    verifier_result = None
    if answer is None and should_run_verifier(route, ranked, details):
        verifier_result = run_candidate_verifier(problem_text, route, strategy_hint, ranked, details, results, best_attempt_by_answer)
        if verifier_result and verifier_result.get("answer") is not None:
            answer = int(verifier_result["answer"])
    if verifier_result is not None:
        details["verifier"] = {
            "candidates": verifier_result.get("candidates"),
            "answer": verifier_result.get("answer"),
            "error": verifier_result.get("error"),
            "text": clip_text(verifier_result.get("text", ""), 1200),
        }
    if "rescue_round" in locals() and rescue_round is not None:
        details["rescue_round"] = rescue_round
    if answer is None:
        answer = fallback_submission_answer(problem_text, problem_id=problem_id, route=route, reason="chat_unresolved")
    elapsed = time.time() - started
    print(f"  -> answer={answer} | route=chat | problem_route={route} | scores={details.get('scores')} | counts={details.get('counts')} | tools={details.get('tools')} | attempts={sum(r.answer is not None for r in results)}/{CFG.attempts} | remaining={PROBLEMS_REMAINING} | elapsed={elapsed:.1f}s")
    attempt_summaries = [summarize_attempt_result(r) for r in results]
    valid_attempts = [r for r in results if r.answer is not None]
    invalid_attempts = len(results) - len(valid_attempts)

    failure_class = None
    if not valid_attempts:
        if any((r.error and "tool_confusion" in str(r.error).lower()) for r in results):
            failure_class = "tool_confusion"
        elif any((r.error and "timeout" in str(r.error).lower()) for r in results):
            failure_class = "timeout_budget"
        elif any((r.error and ("future:" in str(r.error).lower() or "chat_future:" in str(r.error).lower())) for r in results):
            failure_class = "worker_exception"
        elif any((r.text or "").strip() for r in results):
            failure_class = "no_answer_found"
        else:
            failure_class = "no_candidate"

    winner_support = {
        "answer": answer,
        "count": (details.get("counts") or {}).get(answer),
        "score": (details.get("scores") or {}).get(answer),
        "tool_count": (details.get("tools") or {}).get(answer),
        "verifier_used": bool(verifier_result),
    }
    write_debug_event({
        "event": "problem_solved",
        "problem_id": problem_id,
        "route": "chat",
        "final_source": "chat",
        "problem_route": route,
        "answer": answer,
        "details": details,
        "attempts": attempt_summaries,
        "valid_attempt_count": len(valid_attempts),
        "invalid_attempt_count": invalid_attempts,
        "failure_class": failure_class,
        "winner_support": winner_support,
        "elapsed_s": round(elapsed, 3),
        "model_path": str(MODEL_PATH),
        "force_full": force_full,
        "strategy_hint": strategy_hint,
        "problem_excerpt": clip_text(problem_text, 500),
    })
    return coerce_submission_int(answer)

def solve_problem(problem_text, problem_id=None):
    global PROBLEMS_REMAINING
    started = time.time()
    elapsed = time.time() - NOTEBOOK_START
    time_left = max(0.0, CFG.notebook_budget_seconds - elapsed)
    if time_left < CFG.hard_deadline_reserve_seconds:
        PROBLEMS_REMAINING = max(0, PROBLEMS_REMAINING - 1)
        return coerce_submission_int(fallback_submission_answer(problem_text, problem_id=problem_id, route="deadline", reason="hard_deadline_reserve"))
    problem_route = classify_problem_route(problem_text)
    problem_routes, route_scores = classify_problem_routes(problem_text)
    force_full = problem_requires_full_attempts(problem_text, route=problem_route)
    deadline = time.time() + max(60.0, time_left / max(1, PROBLEMS_REMAINING))
    strategy_hint = domain_strategy_hint(problem_text)
    write_debug_event({
        "event": "problem_start",
        "problem_id": problem_id,
        "route": "harmony",
        "problem_route": problem_route,
        "problem_routes": problem_routes,
        "route_scores": dict(route_scores),
        "model_path": str(MODEL_PATH),
        "adapter_path": None if ADAPTER_PATH is None else str(ADAPTER_PATH),
        "request_model": REQUEST_MODEL_NAME,
        "force_full": force_full,
        "time_left_s": round(time_left, 3),
        "deadline_budget_s": round(max(60.0, time_left / max(1, PROBLEMS_REMAINING)), 3),
        "strategy_hint": strategy_hint,
        "problem_excerpt": clip_text(problem_text, 500),
    })
    results, futures = [], {}
    with ThreadPoolExecutor(max_workers=CFG.workers) as ex:
        for i in range(CFG.attempts):
            futures[ex.submit(run_harmony_attempt, problem_text, i, problem_route)] = i
        for fut in as_completed(futures):
            if time.time() > deadline:
                break
            try:
                r = fut.result(timeout=max(5.0, deadline - time.time()))
            except Exception as e:
                r = AttemptResult(None, 0.0, 1.0, futures[fut], 0, 0, "", f"future:{type(e).__name__}")
            results.append(r)
            counts = Counter(x.answer for x in results if x.answer is not None)
            if (not force_full) and counts and counts.most_common(1)[0][1] >= CFG.early_stop_votes:
                break
    ranked, details, best_attempt_by_answer = aggregate_candidate_stats(results)
    answer = choose_weighted_winner(ranked, details, best_attempt_by_answer)
    valid = [r for r in results if r.answer is not None]
    if not valid:
        attempt_summaries = [summarize_attempt_result(r) for r in results]

        write_debug_event({
            "event": "fallback_to_chat",
            "problem_id": problem_id,
            "final_source": "chat_fallback",
            "reason": "no_valid_harmony_answers",
            "failure_class": "no_candidate",
            "valid_attempt_count": 0,
            "invalid_attempt_count": len(results),
            "attempts": attempt_summaries,
        })
        return chat_solve_problem(problem_text, problem_id=problem_id, route=problem_route)
    verifier_result = None
    if answer is None and should_run_verifier(problem_route, ranked, details):
        verifier_result = run_candidate_verifier(problem_text, problem_route, strategy_hint, ranked, details, results, best_attempt_by_answer)
        if verifier_result and verifier_result.get("answer") is not None:
            answer = int(verifier_result["answer"])
    PROBLEMS_REMAINING = max(0, PROBLEMS_REMAINING - 1)
    elapsed_problem = time.time() - started
    if verifier_result is not None:
        details["verifier"] = {
            "candidates": verifier_result.get("candidates"),
            "answer": verifier_result.get("answer"),
            "error": verifier_result.get("error"),
            "text": clip_text(verifier_result.get("text", ""), 1200),
        }
    if answer is None:
        answer = fallback_submission_answer(problem_text, problem_id=problem_id, route=problem_route, reason="harmony_unresolved")
    print(f"  -> answer={answer} | route=harmony | problem_route={problem_route} | scores={details.get('scores')} | counts={details.get('counts')} | tools={details.get('tools')} | attempts={len(valid)}/{CFG.attempts} | remaining={PROBLEMS_REMAINING} | elapsed={elapsed_problem:.1f}s")
    attempt_summaries = [summarize_attempt_result(r) for r in results]
    valid_attempts = [r for r in results if r.answer is not None]
    invalid_attempts = len(results) - len(valid_attempts)

    failure_class = None
    if not valid_attempts:
        failure_class = "no_candidate"
    elif not (details.get("scores") or {}):
        failure_class = "empty_worker_outputs"
    elif len((details.get("counts") or {})) > 1:
        failure_class = "conflicting_candidates"

    winner_support = {
        "answer": answer,
        "count": (details.get("counts") or {}).get(answer),
        "score": (details.get("scores") or {}).get(answer),
        "tool_count": (details.get("tools") or {}).get(answer),
        "verifier_used": bool(details.get("verifier")),
    }

    planner_workers = (
        details.get("workers")
        or details.get("planner_workers")
        or details.get("worker_schedule")
        or []
    )

    orch_route = details.get("route", "unknown")
    write_debug_event({
        "event": "problem_solved",
        "problem_id": problem_id,
        "route": "harmony",
        "final_source": f"harmony:{orch_route}",
        "problem_route": problem_route,
        "answer": answer,
        "details": details,
        "attempts": attempt_summaries,
        "valid_attempt_count": len(valid_attempts),
        "invalid_attempt_count": invalid_attempts,
        "failure_class": failure_class,
        "winner_support": winner_support,
        "planner_workers": planner_workers,
        "force_full": force_full,
        "elapsed_s": round(elapsed_problem, 3),
        "model_path": str(MODEL_PATH),
        "adapter_path": None if ADAPTER_PATH is None else str(ADAPTER_PATH),
        "request_model": REQUEST_MODEL_NAME,
        "strategy_hint": strategy_hint,
        "problem_excerpt": clip_text(problem_text, 500),
    })
    return coerce_submission_int(answer)


## 11. Evaluation Helpers
Reference, benchmark, and dataframe evaluation utilities.

In [ ]:
def predict(id_batch, problem_batch):
    problems = problem_batch.tolist() if hasattr(problem_batch, "tolist") else list(problem_batch)
    ids = id_batch.tolist() if hasattr(id_batch, "tolist") else list(id_batch)
    solver_fn = globals().get("solve_problem_v4_canonical") or solve_problem
    return pd.DataFrame({"answer": [coerce_submission_int(solver_fn(str(p), problem_id=str(i))) for i, p in zip(ids, problems)]})

def run_reference_eval():
    rp = COMP_PATH / "reference.csv"
    if not rp.exists(): print("No reference.csv present"); return
    df = pd.read_csv(rp); global PROBLEMS_REMAINING; PROBLEMS_REMAINING = len(df); correct = 0; rows = []
    for idx, row in enumerate(df.itertuples(index=False), start=1):
        expected = int(row.answer); print(f"\n--- Frontier TIR problem {idx}/{len(df)}: id={row.id} expected={expected} ---")
        solver_fn = globals().get("solve_problem_v4_canonical") or solve_problem
        predicted = solver_fn(str(row.problem), problem_id=str(row.id)); hit = predicted == expected; correct += int(hit); tag = "OK" if hit else "MISS"
        print(f"  -> {tag} id={row.id} | answer={predicted} | expected={expected} | running={correct}/{idx}")
        rows.append({"id": str(row.id), "expected": expected, "predicted": predicted, "correct": hit})
    rdf = pd.DataFrame(rows); print("\n" + "=" * 60); print(f"FRONTIER_TIR_REFERENCE_SCORE: {correct}/{len(df)}"); print("=" * 60)
    bad = rdf[~rdf["correct"]]
    if not bad.empty: print("Mistakes:"); print(bad.to_string(index=False))

def find_public_benchmark_path():
    if CFG.benchmark_path_env:
        explicit = Path(CFG.benchmark_path_env).expanduser()
        if explicit.is_file():
            return explicit
    preferred = ("AIMO3_IMO_Bench_Eval.csv", "aimo3_validation.csv", "validation.csv", "AIMO3_IMO_Bench_CoT.csv")
    for root in iter_input_roots():
        if not root.exists():
            continue
        for name in preferred:
            for p in root.rglob(name):
                if p.is_file(): return p
        for p in root.rglob("*.csv"):
            s = str(p).lower()
            if "aimo" in s and ("bench" in s or "validation" in s or "eval" in s):
                return p
    return None

def choose_eval_columns(df):
    lowered = {str(c).lower(): c for c in df.columns}
    problem_candidates = ("problem", "question", "prompt", "statement", "Problem", "Question")
    answer_candidates = ("answer", "final_answer", "target", "solution_answer", "correct_answer", "ground_truth", "expected", "label", "Answer")
    id_candidates = ("id", "problem_id", "name", "source_id", "ID")
    problem_col = next((lowered[c.lower()] for c in problem_candidates if c.lower() in lowered), None)
    answer_col = next((lowered[c.lower()] for c in answer_candidates if c.lower() in lowered), None)
    id_col = next((lowered[c.lower()] for c in id_candidates if c.lower() in lowered), None)
    if problem_col is None:
        problem_col = next((c for c in df.columns if "problem" in str(c).lower() or "question" in str(c).lower()), None)
    if answer_col is None:
        answer_col = next((c for c in df.columns if any(k in str(c).lower() for k in ("answer", "target", "truth", "expected", "label")) and c != problem_col), None)
    return id_col, problem_col, answer_col

def normalize_expected_answer(value):
    if isinstance(value, float) and math.isnan(value): return None
    text = str(value)
    try:
        return normalize_answer(int(float(text))) if re.fullmatch(r"[-+]?\d+(?:\.0+)?", text.strip()) else extract_answer(text)
    except Exception:
        return extract_answer(text)

def run_dataframe_eval(df, label, limit=None):
    id_col, problem_col, answer_col = choose_eval_columns(df)
    if problem_col is None or answer_col is None:
        print({"eval_skipped": label, "reason": "could_not_find_problem_or_answer_columns", "columns": list(df.columns)})
        return
    work = df[[c for c in (id_col, problem_col, answer_col) if c is not None]].copy()
    if limit and limit > 0:
        work = work.head(limit)
    global PROBLEMS_REMAINING
    PROBLEMS_REMAINING = len(work)
    correct = 0; rows = []
    print({"eval": label, "rows": len(work), "problem_col": str(problem_col), "answer_col": str(answer_col), "id_col": str(id_col)})
    for idx, (_, row) in enumerate(work.iterrows(), start=1):
        problem_text = str(row[problem_col])
        expected = normalize_expected_answer(row[answer_col])
        problem_id = str(row[id_col]) if id_col is not None else str(idx)
        print(f"\n--- {label} problem {idx}/{len(work)}: id={problem_id} expected={expected} ---")
        solver_fn = globals().get("solve_problem_v4_canonical") or solve_problem
        predicted = solver_fn(problem_text, problem_id=problem_id)
        hit = expected is not None and predicted == expected
        correct += int(hit)
        tag = "OK" if hit else "MISS"
        print(f"  -> {tag} id={problem_id} | answer={predicted} | expected={expected} | running={correct}/{idx}")
        rows.append({"id": problem_id, "expected": expected, "predicted": predicted, "correct": hit})
    rdf = pd.DataFrame(rows)
    print("\n" + "=" * 60); print(f"{label.upper()}_SCORE: {correct}/{len(work)}"); print("=" * 60)
    bad = rdf[~rdf["correct"]]
    if not bad.empty: print("Mistakes:"); print(bad.head(25).to_string(index=False))

def run_public_benchmark_eval():
    bp = find_public_benchmark_path()
    if bp is None: return False
    df = pd.read_csv(bp)
    run_dataframe_eval(df, f"benchmark:{bp.name}", CFG.benchmark_eval_limit)
    return True



## 12. Experiment Helpers
Convenience helpers for loading a target problem and running one-off experiments.

In [ ]:
def resolve_experiment_csv_path(csv_path=None):
    if csv_path is not None:
        path = Path(csv_path).expanduser()
        if not path.exists():
            raise FileNotFoundError(f"CSV not found: {path}")
        return path, "explicit"

    benchmark_path = find_public_benchmark_path()
    if benchmark_path is not None:
        benchmark_path = Path(benchmark_path)
        if benchmark_path.exists():
            return benchmark_path, "benchmark"

    if COMP_PATH is not None:
        fallbacks = [
            ("competition_test", COMP_PATH / "test.csv"),
            ("competition_reference", COMP_PATH / "reference.csv"),
        ]
        for source_kind, path in fallbacks:
            if path.exists():
                return path, source_kind

    raise FileNotFoundError(
        "No benchmark CSV found and no competition fallback CSV is available. "
        "Set AIMO3_BENCHMARK_PATH, pass csv_path explicitly, or mount the competition dataset."
    )

def get_problem_catalog(csv_path=None):
    path, source_kind = resolve_experiment_csv_path(csv_path=csv_path)
    df = pd.read_csv(path)
    return path, df

def _norm_col(name):
    return (
        str(name)
        .strip()
        .lower()
        .replace("_", " ")
        .replace("-", " ")
    )

def _find_column(df, candidates):
    norm_map = {_norm_col(c): c for c in df.columns}
    for cand in candidates:
        hit = norm_map.get(_norm_col(cand))
        if hit is not None:
            return hit
    return None

def load_problem_for_experiment(problem_id=None, csv_path=None, row_index=None):
    path, source_kind = resolve_experiment_csv_path(csv_path=csv_path)
    df = pd.read_csv(path)

    id_col = _find_column(df, [
        "id",
        "problem_id",
        "problem id",
        "Problem ID",
    ])
    problem_col = _find_column(df, [
        "problem",
        "question",
        "prompt",
        "Problem",
    ])
    answer_col = _find_column(df, [
        "answer",
        "short answer",
        "short_answer",
        "final answer",
        "Short Answer",
    ])

    if problem_col is None:
        raise RuntimeError(
            f"Could not find problem text column in {path}. "
            f"Available columns: {list(df.columns)}"
        )

    row = None

    if problem_id is not None:
        if id_col is None:
            raise RuntimeError(
                f"Requested problem_id={problem_id}, but {path} has no recognizable id column. "
                f"Available columns: {list(df.columns)}"
            )
        matches = df[df[id_col].astype(str).str.strip() == str(problem_id).strip()]
        if matches.empty:
            raise RuntimeError(
                f"problem_id={problem_id} not found in column {id_col}. "
                f"Available sample ids: {df[id_col].astype(str).head(10).tolist()}"
            )
        row = matches.iloc[0]

    elif row_index is not None:
        if row_index < 0 or row_index >= len(df):
            raise RuntimeError(f"row_index {row_index} out of range for {path} with {len(df)} rows")
        row = df.iloc[row_index]

    else:
        row = df.iloc[0]

    resolved_problem_id = str(row[id_col]) if id_col is not None else f"row-{row.name}"
    problem_text = str(row[problem_col])
    expected_answer = None if answer_col is None else row[answer_col]

    return {
        "problem_id": resolved_problem_id,
        "problem_text": problem_text,
        "expected_answer": expected_answer,
        "id_col": id_col,
        "problem_col": problem_col,
        "answer_col": answer_col,
        "row_index": int(row.name) if hasattr(row, "name") else None,
        "csv_path": str(path),
        "source_kind": source_kind,
    }

def run_single_experiment(problem_text=None, problem_id=None, csv_path=None, row_index=0, reset_budget=True):
    ensure_inference_ready()
    payload = None
    if problem_text is None:
        payload = load_problem_for_experiment(problem_id=problem_id, csv_path=csv_path, row_index=row_index)
        problem_text = payload["problem_text"]
        problem_id = payload["problem_id"]
    if reset_budget:
        reset_notebook_budget(1)
    solver_fn = globals().get("solve_problem_v4_canonical") or solve_problem
    answer = solver_fn(str(problem_text), problem_id=str(problem_id) if problem_id is not None else "adhoc")
    result = {
        "problem_id": None if problem_id is None else str(problem_id),
        "answer": int(answer),
        "expected_answer": None if payload is None else payload.get("expected_answer"),
        "csv_path": None if payload is None else payload.get("csv_path"),
        "source_kind": None if payload is None else payload.get("source_kind"),
    }
    print(result)
    return result

def build_inference_server():
    if aimo3_inference_server is None:
        raise RuntimeError("kaggle_evaluation.aimo_3_inference_server is unavailable in this environment.")
    if COMP_PATH is None:
        raise RuntimeError("Competition path is unavailable. Set AIMO3_COMPETITION_PATH.")
    start_frontier_server(force_restart=False)
    ensure_inference_ready()
    return aimo3_inference_server.AIMO3InferenceServer(predict)

def serve_kaggle_inference():
    server = build_inference_server()
    server.serve()

def write_competition_submission(output_path="/kaggle/working/submission.parquet"):
    if COMP_PATH is None or not (COMP_PATH / "test.csv").exists():
        raise RuntimeError("Competition test.csv is unavailable. Set AIMO3_COMPETITION_PATH.")
    test_df = pd.read_csv(COMP_PATH / "test.csv")
    pred_df = predict(test_df["id"], test_df["problem"]).copy()
    pred_df.insert(0, "id", test_df["id"].astype(str).tolist())
    pred_df["answer"] = pred_df["answer"].astype(int)

    sample_csv = COMP_PATH / "sample_submission.csv"
    sample_parquet = COMP_PATH / "sample_submission.parquet"
    if sample_csv.exists():
        submission = pd.read_csv(sample_csv)
        if "id" in submission.columns:
            submission["id"] = test_df["id"].astype(str).tolist()
        submission["answer"] = pred_df["answer"].tolist()
    elif sample_parquet.exists():
        submission = pd.read_parquet(sample_parquet)
        if "id" in submission.columns:
            submission["id"] = test_df["id"].astype(str).tolist()
        submission["answer"] = pred_df["answer"].tolist()
    else:
        submission = pred_df[["id", "answer"]].copy()

    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    submission.to_parquet(output, index=False)
    print({"submission_path": str(output), "rows": len(submission)})
    return submission

def run_default_local_mode():
    if run_public_benchmark_eval():
        return "benchmark"
    if COMP_PATH is not None and (COMP_PATH / "test.csv").exists():
        submission = write_competition_submission()
        print(submission)
        return "test"
    if COMP_PATH is not None and (COMP_PATH / "reference.csv").exists():
        run_reference_eval()
        return "reference"
    warning = {
        "run_mode": "standalone",
        "warning": "No benchmark, reference, or test dataset found. Set AIMO3_BENCHMARK_PATH or AIMO3_COMPETITION_PATH.",
    }
    print(warning)
    return warning


## 10B. Planner + Specialized Workers + Adversarial Verifier

This upgrade replaces the flat N-attempt search with a staged harness:

1. Planner: route-aware decomposition and worker allocation  
2. Specialized Harmony workers: proof, counterexample, construction/bounds, algebra/CAS, recurrence/casework  
3. Candidate aggregation with certificate-first ranking  
4. Adversarial verifier that selects only from produced candidates  
5. Fallback to simpler chat path only when the modular path fails


In [ ]:

# Plan-driven modular harness override.

PLANNER_SYSTEM_PROMPT = """Reasoning: high

You are a planner for olympiad-math solving.
Return ONLY one JSON object:
{
  "problem_type": "<short label>",
  "subgoals": ["<goal 1>", "<goal 2>", "<goal 3>"],
  "worker_roles": ["proof", "counterexample", "construction", "algebra", "casework"],
  "verification_focus": ["<what must be disproved or checked>"],
  "tool_guidance": ["<when python/sympy is appropriate>"]
}

Rules:
- Do not solve the full problem.
- Subgoals must be concrete and falsifiable.
- Prefer at most 5 worker roles.
- No markdown and no extra prose.
"""

WORKER_ROLE_LIBRARY = {
    "proof": {
        "method": "proof_first",
        "instruction": (
            "Primary goal: derive a rigorous structural solution. "
            "Prefer invariants, factorization, descent, extremal arguments, bijections, or exact reductions. "
            "Do not rely on numerical pattern matching."
        ),
    },
    "counterexample": {
        "method": "hybrid",
        "instruction": (
            "Primary goal: attack likely wrong assumptions. "
            "Use small exact cases, boundary cases, parity/modular obstructions, or exact python checks to refute tempting but false patterns. "
            "If you find a contradiction that isolates the correct answer, state it cleanly."
        ),
    },
    "construction": {
        "method": "hybrid",
        "instruction": (
            "Primary goal: for extremal/optimization/counting problems, produce a constructive witness or equality case and pair it with a matching upper/lower bound. "
            "Do not accept a candidate without both feasibility and optimality support."
        ),
    },
    "algebra": {
        "method": "pot_python",
        "instruction": (
            "Primary goal: use symbolic algebra or exact computation when it materially reduces risk. "
            "Use python only for exact enumeration, sympy simplification, recurrences, or algebraic verification. "
            "The final answer must still be justified in the returned JSON."
        ),
    },
    "casework": {
        "method": "hybrid",
        "instruction": (
            "Primary goal: partition the problem into a small number of exact cases, prove each case, and eliminate impossible branches. "
            "Do not leave unclosed cases."
        ),
    },
    "recurrence": {
        "method": "hybrid",
        "instruction": (
            "Primary goal: isolate the recurrence transition, derive an invariant, monotonicity argument, or closed form, and verify early terms exactly."
        ),
    },
}

VERIFIER_V2_SYSTEM_PROMPT = """Reasoning: high

You are an adversarial olympiad verifier.
You must choose ONLY from the listed candidate answers.
Return ONLY one JSON object:
{
  "selected_answer": <candidate integer or null>,
  "status": "verified" | "inconclusive" | "rejected_all",
  "reason": "<short explanation>",
  "reject_reasons": ["..."],
  "preferred_certificate": "<certificate type or none>"
}

Rules:
- Re-solve from the original problem, not from majority vote.
- Prefer machine-checkable certificates and structurally complete proofs over popularity.
- Reject candidates supported only by approximation, floating search, random search, or unproved pattern matching.
- Never invent a new answer.
- No markdown.
"""

def validate_plan_payload(obj):
    if not isinstance(obj, dict):
        return False, "plan_not_dict"
    required = {"problem_type", "subgoals", "worker_roles", "verification_focus", "tool_guidance"}
    missing = [k for k in required if k not in obj]
    if missing:
        return False, "plan_missing:" + ",".join(missing)
    for key in ("subgoals", "worker_roles", "verification_focus", "tool_guidance"):
        if not isinstance(obj.get(key), list):
            return False, f"plan_bad_{key}"
    if not isinstance(obj.get("problem_type"), str):
        return False, "plan_bad_problem_type"
    obj["worker_roles"] = [str(x).strip().lower() for x in obj["worker_roles"] if str(x).strip()]
    obj["worker_roles"] = [x for x in obj["worker_roles"] if x in WORKER_ROLE_LIBRARY]
    if not obj["worker_roles"]:
        obj["worker_roles"] = ["proof", "algebra", "counterexample"]
    obj["worker_roles"] = obj["worker_roles"][:5]
    obj["subgoals"] = [clip_text(str(x), 200) for x in obj["subgoals"][:6]]
    obj["verification_focus"] = [clip_text(str(x), 200) for x in obj["verification_focus"][:6]]
    obj["tool_guidance"] = [clip_text(str(x), 200) for x in obj["tool_guidance"][:6]]
    return True, None

def plan_problem(problem_text, route):
    ensure_inference_ready()
    prompt = (
        problem_text.strip()
        + "\n\nPrimary route:\n" + str(route)
        + "\n\nRoute hint:\n" + route_prompt(route)
        + "\n\nReturn JSON only."
    )
    try:
        response = CLIENT.chat.completions.create(
            model=REQUEST_MODEL_NAME,
            messages=[
                {"role": "system", "content": PLANNER_SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ],
            temperature=0.0,
            top_p=0.95,
            max_tokens=1200,
            seed=CFG.startup_seed + 6100,
        )
        if not response.choices:
            raise RuntimeError("planner_no_choices")
        text = sanitize_model_response_text(response.choices[0].message.content, model_spec["family"])
        obj = extract_json_object(text)
        if obj is None:
            raise RuntimeError("planner_bad_json")
        ok, err = validate_plan_payload(obj)
        if not ok:
            raise RuntimeError(err)
        obj["_raw"] = clip_text(text, 1200)
        return obj
    except Exception as exc:
        fallback_roles = ["proof", "algebra", "counterexample"]
        if route in {"optimization_extremal", "geometry_inequality"}:
            fallback_roles = ["construction", "proof", "counterexample"]
        elif route in {"recurrence_sequence_structure"}:
            fallback_roles = ["recurrence", "proof", "algebra"]
        elif route in {"subset_sum_threshold", "combinatorics_counting"}:
            fallback_roles = ["casework", "construction", "counterexample"]
        return {
            "problem_type": route,
            "subgoals": [
                "Isolate the structural core of the problem.",
                "Derive or refute candidate patterns using exact reasoning.",
                "Verify the final candidate with an independent check.",
            ],
            "worker_roles": fallback_roles,
            "verification_focus": ["Check hidden edge cases and reject unsupported patterns."],
            "tool_guidance": ["Use python only for exact enumeration, symbolic algebra, or deterministic verification."],
            "_raw": f"planner_fallback:{type(exc).__name__}:{str(exc)[:160]}",
        }

def make_role_developer_message(persona, route, attempt_index, role_name, plan):
    role_cfg = WORKER_ROLE_LIBRARY.get(role_name, WORKER_ROLE_LIBRARY["proof"])
    route_block = route_prompt(route)
    attempt_block = route_attempt_stance(route, attempt_index)
    subgoals = "\n".join(f"- {x}" for x in plan.get("subgoals", [])[:6])
    verify_focus = "\n".join(f"- {x}" for x in plan.get("verification_focus", [])[:6])
    tool_guidance = "\n".join(f"- {x}" for x in plan.get("tool_guidance", [])[:6])
    inst = f"""# Instructions
You are a specialized olympiad worker.

# Primary role
{role_name}

# Role objective
{role_cfg['instruction']}

# Global problem lens
{route_block}

# Planner subgoals
{subgoals}

# Verification focus
{verify_focus}

# Tool guidance
{tool_guidance}

# Attempt style
{attempt_block}

Solve the olympiad problem exactly.
Use python only when it reduces risk.
When sending python:
- send executable code to the python recipient
- keep it deterministic, concise, and assert-heavy
- use exact arithmetic when possible
- the last printed non-empty line should be either a JSON object {{"final_answer": <int>}} or a single integer
- do not use randomness, simulation-only evidence, or floating optimization as final justification

Prefer:
- explicit contradictions for false branches
- exact certificates when possible
- no guessing

{HARMONY_FINAL_JSON_SPEC}
"""
    if persona:
        inst += "\n# Solver stance\n" + persona.strip() + "\n"
    return Message.from_role_and_content(Role.DEVELOPER, DeveloperContent.new().with_instructions(inst))

def run_harmony_role_attempt(problem_text, attempt_index, route, role_name, plan):
    persona = CFG.personas[attempt_index % len(CFG.personas)] if CFG.use_personas else ""
    seed = int((CFG.startup_seed + attempt_index + 1) ** 2 + (abs(hash(role_name)) % 100000))
    temp = max(0.15, attempt_temperature(attempt_index) - (0.10 if role_name in {"proof", "construction"} else 0.0))
    sandbox = SANDBOX_POOL.acquire()
    py_calls = 0
    final_texts = []
    entropies = []
    tool_artifacts = []
    messages = [
        make_system_message(),
        make_role_developer_message(persona, route, attempt_index, role_name, plan),
        make_user_message(problem_text, route),
    ]
    try:
        for turn in range(CFG.tool_rounds + 1):
            ids = ENCODING.render_conversation_for_completion(Conversation.from_messages(messages), Role.ASSISTANT)
            max_available = max(512, CFG.max_model_len - len(ids) - 128)
            turn_max = min(CFG.max_tokens, max_available)
            if turn_max < 512:
                break
            token_ids, raw_text, entropy = completion_tokens(ids, temp, turn_max, seed + turn)
            entropies.append(entropy)
            if not token_ids:
                ans = extract_answer(raw_text)
                result = make_attempt_result(ans, (1.0 / max(entropy, 0.08)) if ans is not None else 0.0, entropy, attempt_index, seed, py_calls, raw_text, route, None if ans is not None else "no_tokens", tool_artifacts=tool_artifacts)
                result.method = role_name
                return result
            parsed = ENCODING.parse_messages_from_completion_tokens(token_ids, Role.ASSISTANT)
            if not parsed:
                ans = extract_answer(raw_text)
                result = make_attempt_result(ans, (1.0 / max(entropy, 0.08)) if ans is not None else 0.0, entropy, attempt_index, seed, py_calls, raw_text, route, None if ans is not None else "parse_empty", tool_artifacts=tool_artifacts)
                result.method = role_name
                return result

            messages.extend(parsed)
            last = parsed[-1]
            text = message_text(last)
            final_texts.append(text)

            if getattr(last, "recipient", None) == "python":
                py_calls += 1
                timeout = CFG.python_timeout_seconds + (4 if role_name in {"algebra", "counterexample", "casework"} else 0)
                output = sandbox.execute(text, timeout=timeout)
                tool_artifacts.append(build_tool_artifact(text, output))
                messages.append(Message.from_author_and_content(Author.new(Role.TOOL, "python"), output).with_channel("commentary"))
                continue

            channel = str(getattr(last, "channel", "") or "")
            combined_text = text if channel == "final" else ("\n".join(final_texts[-2:]) or raw_text)
            payload = extract_json_object(combined_text)
            if payload is not None:
                ok, err = validate_candidate_payload(payload)
                if ok:
                    mean_e = sum(entropies) / max(1, len(entropies))
                    weight = 1.0 / max(mean_e, 0.08)
                    if py_calls:
                        weight *= 1.12
                    if role_name in {"proof", "construction"} and payload.get("certificate_type") != "none":
                        weight *= 1.08
                    result = make_attempt_result(payload["answer"], weight if payload["answer"] is not None else 0.0, mean_e, attempt_index, seed, py_calls, combined_text, route, tool_artifacts=tool_artifacts)
                    result.method = role_name
                    return result
                if channel == "final":
                    return AttemptResult(None, 0.0, entropy, attempt_index, seed, py_calls, combined_text, f"bad_final_json:{err}", route=route, method=role_name)

        blob = "\n".join(final_texts)
        payload = extract_json_object(blob)
        mean_e = sum(entropies) / max(1, len(entropies)) if entropies else 1.0
        if payload is not None:
            ok, err = validate_candidate_payload(payload)
            if ok:
                result = make_attempt_result(payload["answer"], (0.8 / max(mean_e, 0.1)) if payload["answer"] is not None else 0.0, mean_e, attempt_index, seed, py_calls, blob, route, tool_artifacts=tool_artifacts)
                result.method = role_name
                return result
            return AttemptResult(None, 0.0, mean_e, attempt_index, seed, py_calls, blob, f"final_json:{err}", route=route, method=role_name)
        ans = extract_answer(blob)
        result = make_attempt_result(ans, (0.6 / max(mean_e, 0.1)) if ans is not None else 0.0, mean_e, attempt_index, seed, py_calls, blob, route, None if ans is not None else "no_answer", tool_artifacts=tool_artifacts)
        result.method = role_name
        return result
    except Exception as e:
        return AttemptResult(None, 0.0, 1.0, attempt_index, seed, py_calls, "", f"{type(e).__name__}: {str(e)[:200]}", route=route, method=role_name)
    finally:
        SANDBOX_POOL.release(sandbox)

def build_worker_schedule(plan, route):
    roles = [r for r in plan.get("worker_roles", []) if r in WORKER_ROLE_LIBRARY]
    if not roles:
        roles = ["proof", "algebra", "counterexample"]
    roles = roles[: max(1, min(5, len(roles)))]
    total = max(roles.__len__(), CFG.attempts)
    schedule = []
    for i in range(CFG.attempts):
        schedule.append(roles[i % len(roles)])
    if route in {"optimization_extremal", "geometry_inequality"} and "construction" not in schedule:
        schedule[0] = "construction"
    if route in {"functional_equation_structure", "polynomial_root_structure"} and "algebra" not in schedule and len(schedule) > 1:
        schedule[1] = "algebra"
    if "proof" not in schedule:
        schedule[-1] = "proof"
    return schedule

def build_candidate_briefs_v2(candidates, details, best_attempt_by_answer):
    briefs = []
    for answer in candidates:
        attempt = best_attempt_by_answer.get(int(answer))
        if attempt is None:
            continue
        artifacts = []
        for artifact in list(attempt.certificate_payload or ())[:2]:
            artifacts.append({
                "status": artifact.get("status"),
                "final_answer": artifact.get("final_answer"),
                "code": clip_text(artifact.get("code", ""), 360),
                "output": clip_text(artifact.get("output", ""), 420),
            })
        briefs.append({
            "answer": int(answer),
            "score": float(details["scores"].get(int(answer), 0.0)),
            "votes": int(details["counts"].get(int(answer), 0)),
            "worker_method": attempt.method,
            "certificate_type": attempt.certificate_type,
            "certificate_summary": attempt.certificate_summary,
            "tool_confused": bool(attempt.tool_confused),
            "approx_only": bool(attempt.approx_only),
            "random_search": bool(attempt.random_search),
            "weak_exactness": bool(attempt.weak_exactness),
            "proof_sketch": clip_text(attempt.final_text, 1400),
            "tool_artifacts": artifacts,
        })
    return briefs

def run_candidate_verifier_v2(problem_text, route, strategy_hint, plan, ranked, details, best_attempt_by_answer):
    client, model_spec = ensure_inference_ready(role="verifier")
    candidates = verifier_candidates_from_ranked(ranked)
    if len(candidates) < 2:
        return None
    briefs = build_candidate_briefs_v2(candidates, details, best_attempt_by_answer)
    prompt = (
        problem_text.strip()
        + "\n\nPrimary route:\n" + route
        + "\n\nRoute hint:\n" + route_prompt(route)
        + ("\n\nStrategy hint:\n" + strategy_hint if strategy_hint else "")
        + "\n\nPlanner output:\n" + json.dumps({k: v for k, v in plan.items() if not str(k).startswith("_")}, ensure_ascii=False, indent=2)
        + "\n\nCandidate artifacts to verify:\n" + json.dumps(briefs, ensure_ascii=False, indent=2)
        + "\n\nSelect only from the listed candidates."
    )
    try:
        response = client.chat.completions.create(
            model=model_spec["request_model_name"],
            messages=build_model_chat_messages(model_spec, "verifier", VERIFIER_V2_SYSTEM_PROMPT, prompt),
            temperature=0.0,
            top_p=0.95,
            max_tokens=CFG.verifier_max_tokens,
            seed=CFG.startup_seed + 7100,
        )
        if not response.choices:
            return {"candidates": candidates, "answer": None, "text": "", "error": "verifier_no_choices"}
        text = coerce_text(response.choices[0].message.content)
        obj = extract_json_object(text)
        if obj is None:
            return {"candidates": candidates, "answer": None, "text": text, "error": "verifier_bad_json"}
        ok, err = validate_verifier_payload(obj, candidates)
        if not ok:
            return {"candidates": candidates, "answer": None, "text": text, "error": err}
        return {"candidates": candidates, "answer": obj["selected_answer"], "text": text, "error": None}
    except Exception as exc:
        return {"candidates": candidates, "answer": None, "text": "", "error": f"verifier:{type(exc).__name__}: {str(exc)[:160]}"}

def solve_problem(problem_text, problem_id=None):
    global PROBLEMS_REMAINING
    started = time.time()
    elapsed = time.time() - NOTEBOOK_START
    time_left = max(0.0, CFG.notebook_budget_seconds - elapsed)
    if time_left < CFG.hard_deadline_reserve_seconds:
        PROBLEMS_REMAINING = max(0, PROBLEMS_REMAINING - 1)
        return coerce_submission_int(fallback_submission_answer(problem_text, problem_id=problem_id, route="deadline", reason="hard_deadline_reserve"))

    problem_route = classify_problem_route(problem_text)
    problem_routes, route_scores = classify_problem_routes(problem_text)
    force_full = problem_requires_full_attempts(problem_text, route=problem_route)
    deadline = time.time() + max(60.0, time_left / max(1, PROBLEMS_REMAINING))
    strategy_hint = domain_strategy_hint(problem_text)
    plan = plan_problem(problem_text, problem_route)
    worker_schedule = build_worker_schedule(plan, problem_route)

    write_debug_event({
        "event": "problem_start_v2",
        "problem_id": problem_id,
        "route": "harmony_plan_workers",
        "problem_route": problem_route,
        "problem_routes": problem_routes,
        "route_scores": dict(route_scores),
        "model_path": str(MODEL_PATH),
        "adapter_path": None if ADAPTER_PATH is None else str(ADAPTER_PATH),
        "request_model": REQUEST_MODEL_NAME,
        "force_full": force_full,
        "time_left_s": round(time_left, 3),
        "deadline_budget_s": round(max(60.0, time_left / max(1, PROBLEMS_REMAINING)), 3),
        "strategy_hint": strategy_hint,
        "plan": {k: v for k, v in plan.items() if not str(k).startswith("_")},
        "worker_schedule": worker_schedule,
        "problem_excerpt": clip_text(problem_text, 500),
    })

    results, futures = [], {}
    with ThreadPoolExecutor(max_workers=CFG.workers) as ex:
        for i, role_name in enumerate(worker_schedule):
            futures[ex.submit(run_harmony_role_attempt, problem_text, i, problem_route, role_name, plan)] = (i, role_name)
        for fut in as_completed(futures):
            if time.time() > deadline:
                break
            idx, role_name = futures[fut]
            try:
                r = fut.result(timeout=max(5.0, deadline - time.time()))
            except Exception as e:
                r = AttemptResult(None, 0.0, 1.0, idx, 0, 0, "", f"future:{type(e).__name__}", route=problem_route, method=role_name)
            results.append(r)
            counts = Counter(x.answer for x in results if x.answer is not None)
            if (not force_full) and counts and counts.most_common(1)[0][1] >= CFG.early_stop_votes:
                break

    PROBLEMS_REMAINING = max(0, PROBLEMS_REMAINING - 1)
    ranked, details, best_attempt_by_answer = aggregate_candidate_stats(results)
    rescue_round = None
    rescue_needed, rescue_reasons = should_run_rescue_round(route_context, ranked, details, len(valid))
    if rescue_needed:
        rescue_round = {
            "scores_before": dict(details.get("scores") or {}),
            "counts_before": dict(details.get("counts") or {}),
        }
        rescue_results, rescue_meta = run_rescue_round(
            problem_text, route_context, plan, ranked, details, best_attempt_by_answer, deadline,
        )
        rescue_round.update(rescue_meta or {})
        if rescue_round is None:
            rescue_round = {}
        rescue_round["reasons"] = rescue_reasons
        if rescue_results:
            results.extend(rescue_results)
            ranked, details, best_attempt_by_answer = aggregate_candidate_stats(results)
            valid = [r for r in results if r.answer is not None]
            write_debug_event({
                "event": "rescue_round_v4",
                "problem_id": problem_id,
                "problem_route": problem_route,
                "reasons": rescue_reasons,
                "details_before": {
                    "scores": rescue_round.get("scores_before") or {},
                    "counts": rescue_round.get("counts_before") or {},
                },
                "rescue_meta": rescue_round,
            })

    answer = choose_weighted_winner(ranked, details, best_attempt_by_answer)

    verifier_result = None
    need_verifier = should_run_verifier(problem_route, ranked, details) or (answer is None and len(ranked) >= 2)
    if need_verifier:
        verifier_result = run_candidate_verifier_v2(problem_text, problem_route, strategy_hint, plan, ranked, details, best_attempt_by_answer)
        if verifier_result and verifier_result.get("answer") is not None:
            answer = int(verifier_result["answer"])

    if answer is None:
        answer = chat_solve_problem(problem_text, problem_id=problem_id, route=problem_route)

    if verifier_result is not None:
        details["verifier"] = {
            "candidates": verifier_result.get("candidates"),
            "answer": verifier_result.get("answer"),
            "error": verifier_result.get("error"),
            "text": clip_text(verifier_result.get("text", ""), 1200),
        }
    if "rescue_round" in locals() and rescue_round is not None:
        details["rescue_round"] = rescue_round

    elapsed = time.time() - started
    print(
        f"  -> answer={answer} | route=harmony_plan_workers | problem_route={problem_route} "
        f"| workers={worker_schedule} | scores={details.get('scores')} | counts={details.get('counts')} "
        f"| tools={details.get('tools')} | attempts={sum(r.answer is not None for r in results)}/{len(worker_schedule)} "
        f"| remaining={PROBLEMS_REMAINING} | elapsed={elapsed:.1f}s"
    )
    write_debug_event({
        "event": "problem_solved_v2",
        "problem_id": problem_id,
        "route": "harmony_plan_workers",
        "problem_route": problem_route,
        "answer": answer,
        "details": details,
        "plan": {k: v for k, v in plan.items() if not str(k).startswith("_")},
        "worker_schedule": worker_schedule,
        "attempts": [summarize_attempt_result(r) for r in results],
        "elapsed_s": round(elapsed, 3),
        "model_path": str(MODEL_PATH),
        "force_full": force_full,
        "strategy_hint": strategy_hint,
        "problem_excerpt": clip_text(problem_text, 500),
    })
    return int(answer)


In [ ]:
## 10C. Chat-Based Role Workers with Forced Tool Use (v2)
#
# Key fixes over v1:
# 1. Two-phase code workers: FORCE model to write Python first, execute, then derive answer
# 2. Proof workers: single-shot JSON (no confusing optional code instructions)
# 3. Minimum 8 workers regardless of CFG.attempts
# 4. Penalize answer=0 for non-trivial combinatorics
# 5. Broader code extraction regex
# 6. Higher weight bonus for tool-supported answers

MIN_TOTAL_WORKERS = 8

CODE_BLOCK_RE = re.compile(r'```(?:[Pp]ython3?|[Pp]y)?\s*\n(.*?)```', re.DOTALL)
BARE_CODE_RE = re.compile(
    r'((?:import |from )\S.*(?:\n(?:import |from |[ \t]|[a-zA-Z_]\w*\s*=|for |while |if |print|def |assert |#|\s*$).*)*)',
    re.MULTILINE,
)

def extract_code_from_text(text):
    """Extract Python code: try fenced blocks first, then bare code heuristic."""
    blocks = CODE_BLOCK_RE.findall(text)
    if blocks:
        return max(blocks, key=len).strip()
    candidates = BARE_CODE_RE.findall(text)
    if candidates:
        best = max(candidates, key=len).strip()
        if best.count('\n') >= 2 and ('import' in best or 'print' in best or 'for ' in best):
            return best
    return None

# ---- SYSTEM PROMPTS ----

CODE_WORKER_SYSTEM = """Reasoning: high

You are solving an olympiad math problem using Python computation.

Write a COMPLETE, EXECUTABLE Python script that computes the answer.

Rules:
- You MUST write Python code inside a ```python block
- Use exact arithmetic: fractions.Fraction, sympy, integers only - NO floats for final answers
- Available packages: math, itertools, functools, fractions, collections, decimal, sympy, numpy
- The script MUST print exactly one integer on its last output line: the final answer
- Include assertions or checks to verify correctness
- NO randomness, NO approximation, NO simulation-only evidence
- Code must be deterministic and terminate within 12 seconds
- For counting problems: enumerate exactly or use inclusion-exclusion
- For optimization: prove both bound and construction
"""

ANSWER_FROM_CODE_SYSTEM = """You executed Python code to help solve a math problem.
Given the original problem, the code, and its output, determine the final answer.

Return ONLY one JSON object:
{
  "answer": <integer 0..99999 or null>,
  "status": "solved" | "partial" | "stuck",
  "method": "pot_python",
  "proof_sketch": "<what the code computed and why it is correct>",
  "certificate_type": "exhaustive_check" | "python_check" | "none",
  "certificate_summary": "<what was verified by the code>",
  "assumptions": [],
  "self_checks": [],
  "final_statement": "<one sentence>"
}
No extra text before or after the JSON.
"""

PROOF_WORKER_SYSTEM = """Reasoning: high

You are a specialized olympiad-math solver.
Solve the problem using rigorous mathematical reasoning.
The final answer is a non-negative integer in [0, 99999].

Return ONLY one JSON object:
{
  "answer": <integer 0..99999 or null>,
  "status": "solved" | "partial" | "stuck",
  "method": "proof_first",
  "proof_sketch": "<concise but rigorous reasoning>",
  "certificate_type": "none" | "counterexample_check" | "optimization_witness" | "equivalence_check" | "exhaustive_check",
  "certificate_summary": "<what was verified>",
  "assumptions": [],
  "self_checks": [],
  "final_statement": "<one sentence>"
}
No markdown. No extra text before or after the JSON.
"""

# ---- WORKER FUNCTIONS ----

def code_first_worker(problem_text, attempt_index, route, role_name, plan):
    """Force the model to write Python code, execute it, then derive the answer."""
    ensure_inference_ready()
    sandbox = SANDBOX_POOL.acquire()
    seed = int((CFG.startup_seed + attempt_index + 1) ** 2)
    # Wider temperature spread: low for first attempts, high for exploration
    base_temp = attempt_temperature(attempt_index)
    if attempt_index < 2:
        temp = max(0.15, base_temp * 0.7)  # Conservative first attempts
    elif attempt_index >= 6:
        temp = min(1.2, base_temp * 1.3)  # Exploratory late attempts
    else:
        temp = max(0.2, base_temp)
    py_calls = 0
    tool_artifacts = []

    try:
        role_cfg = WORKER_ROLE_LIBRARY.get(role_name, WORKER_ROLE_LIBRARY["proof"])
        route_block = route_prompt(route)
        strategy_hint = domain_strategy_hint(problem_text)
        subgoals = "\n".join(f"- {x}" for x in plan.get("subgoals", [])[:4])

        system = (
            CODE_WORKER_SYSTEM
            + f"\n# Role: {role_name}\n{role_cfg['instruction']}"
            + f"\n# Problem lens\n{route_block}"
        )
        # Diversify: each attempt gets a different algorithmic strategy hint
        strategy_idx = attempt_index % len(CODE_STRATEGIES)
        algo_hint = CODE_STRATEGIES[strategy_idx]

        user_msg = (
            problem_text.strip()
            + "\n\nDomain hint: " + strategy_hint
            + "\n\nSubgoals:\n" + subgoals
            + "\n\nAlgorithmic approach to try: " + algo_hint
            + "\n\nWrite Python code in a ```python block to compute the exact answer."
        )

        # PHASE 1: Get Python code from the model
        resp1 = CLIENT.chat.completions.create(
            model=REQUEST_MODEL_NAME,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user_msg},
            ],
            temperature=temp,
            top_p=0.95,
            max_tokens=min(CFG.max_tokens, 12000),
            seed=seed,
        )
        if not resp1.choices:
            return AttemptResult(
                None, 0.0, 1.0, attempt_index, seed, 0, "",
                "code_worker_no_choices", route=route, method=role_name,
            )

        text1 = coerce_text(resp1.choices[0].message.content)
        code = extract_code_from_text(text1)

        if not code:
            # Model didn't produce code; try to parse as direct answer
            payload = extract_json_object(text1)
            if payload:
                ok, _ = validate_candidate_payload(payload)
                if ok:
                    result = make_attempt_result(
                        payload["answer"], 0.5 if payload["answer"] is not None else 0.0,
                        0.7, attempt_index, seed, 0, text1, route,
                        tool_artifacts=tool_artifacts,
                    )
                    result.method = role_name
                    return result
            ans = extract_answer(text1)
            result = make_attempt_result(
                ans, 0.4 if ans is not None else 0.0,
                0.8, attempt_index, seed, 0, text1, route,
                None if ans is not None else "no_code_no_answer",
                tool_artifacts=tool_artifacts,
            )
            result.method = role_name
            return result

        # PHASE 2: Execute the code
        py_calls = 1
        timeout = CFG.python_timeout_seconds + 4
        output = sandbox.execute(code, timeout=timeout)
        tool_artifacts.append(build_tool_artifact(code, output))

        # PHASE 3: Ask model to interpret the output and give final JSON answer
        answer_msg = (
            f"Original problem:\n{problem_text.strip()}\n\n"
            f"Python code:\n```python\n{code}\n```\n\n"
            f"Execution result:\n{output}\n\n"
            "Based on the code and its output, provide the final answer as JSON."
        )
        resp2 = CLIENT.chat.completions.create(
            model=REQUEST_MODEL_NAME,
            messages=[
                {"role": "system", "content": ANSWER_FROM_CODE_SYSTEM},
                {"role": "user", "content": answer_msg},
            ],
            temperature=0.0,
            top_p=0.95,
            max_tokens=2000,
            seed=seed + 1000,
        )

        combined = text1
        if resp2.choices:
            text2 = coerce_text(resp2.choices[0].message.content)
            combined = text1 + "\n---\n" + text2

        # Parse answer
        payload = extract_json_object(combined)
        if payload is not None:
            ok, err = validate_candidate_payload(payload)
            if ok:
                weight = 1.30  # Strong bonus for tool-supported answers
                result = make_attempt_result(
                    payload["answer"],
                    weight if payload["answer"] is not None else 0.0,
                    0.4, attempt_index, seed, py_calls, combined, route,
                    tool_artifacts=tool_artifacts,
                )
                result.method = role_name
                return result

        # Fallback: try to parse integer from code output directly
        ans = extract_answer(combined)
        if ans is None and output:
            ans = extract_answer(output)
        result = make_attempt_result(
            ans, 0.8 if ans is not None else 0.0,
            0.5, attempt_index, seed, py_calls, combined, route,
            None if ans is not None else "code_no_answer",
            tool_artifacts=tool_artifacts,
        )
        result.method = role_name
        return result

    except Exception as e:
        return AttemptResult(
            None, 0.0, 1.0, attempt_index, seed, py_calls, "",
            f"{type(e).__name__}: {str(e)[:200]}",
            route=route, method=role_name,
        )
    finally:
        SANDBOX_POOL.release(sandbox)


def proof_worker(problem_text, attempt_index, route, role_name, plan):
    """Single-shot proof/reasoning worker that returns JSON directly."""
    ensure_inference_ready()
    seed = int((CFG.startup_seed + attempt_index + 1) ** 2)
    base_temp = attempt_temperature(attempt_index)
    if attempt_index < 2:
        temp = max(0.1, base_temp * 0.6)
    elif attempt_index >= 6:
        temp = min(1.15, base_temp * 1.3)
    else:
        temp = max(0.15, base_temp)

    try:
        role_cfg = WORKER_ROLE_LIBRARY.get(role_name, WORKER_ROLE_LIBRARY["proof"])
        route_block = route_prompt(route)
        attempt_block = route_attempt_stance(route, attempt_index)
        strategy_hint = domain_strategy_hint(problem_text)
        subgoals = "\n".join(f"- {x}" for x in plan.get("subgoals", [])[:4])

        system = (
            PROOF_WORKER_SYSTEM
            + f"\n# Role: {role_name}\n{role_cfg['instruction']}"
            + f"\n# Problem lens\n{route_block}"
            + f"\n# Attempt style\n{attempt_block}"
        )
        user_msg = (
            problem_text.strip()
            + "\n\nDomain hint: " + strategy_hint
            + "\n\nSubgoals:\n" + subgoals
            + "\n\nReturn JSON only."
        )

        resp = CLIENT.chat.completions.create(
            model=REQUEST_MODEL_NAME,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user_msg},
            ],
            temperature=temp,
            top_p=0.95,
            max_tokens=min(CFG.max_tokens, 16384),
            seed=seed,
        )

        if not resp.choices:
            return AttemptResult(
                None, 0.0, 1.0, attempt_index, seed, 0, "",
                "proof_no_choices", route=route, method=role_name,
            )

        text = coerce_text(resp.choices[0].message.content)
        payload = extract_json_object(text)
        if payload is not None:
            ok, err = validate_candidate_payload(payload)
            if ok:
                weight = 1.0
                result = make_attempt_result(
                    payload["answer"],
                    weight if payload["answer"] is not None else 0.0,
                    0.5, attempt_index, seed, 0, text, route,
                )
                result.method = role_name
                return result

        ans = extract_answer(text)
        result = make_attempt_result(
            ans, 0.5 if ans is not None else 0.0,
            0.6, attempt_index, seed, 0, text, route,
            None if ans is not None else "no_answer",
        )
        result.method = role_name
        return result

    except Exception as e:
        return AttemptResult(
            None, 0.0, 1.0, attempt_index, seed, 0, "",
            f"{type(e).__name__}: {str(e)[:200]}",
            route=route, method=role_name,
        )


# ---- IMPROVED WORKER SCHEDULE ----

CODE_FIRST_ROLES = {"algebra", "counterexample", "casework", "recurrence"}
PROOF_FIRST_ROLES = {"proof", "construction"}

def build_mixed_worker_schedule(plan, route):
    """Build a schedule that mixes code-first and proof-first workers.
    Always produces at least MIN_TOTAL_WORKERS entries."""
    roles = [r for r in plan.get("worker_roles", []) if r in WORKER_ROLE_LIBRARY]
    if not roles:
        roles = ["proof", "algebra", "counterexample"]
    roles = roles[:5]

    n_workers = max(MIN_TOTAL_WORKERS, CFG.attempts)
    schedule = []
    for i in range(n_workers):
        schedule.append(roles[i % len(roles)])

    # Ensure mix: at least 2 code-first and 2 proof-first
    code_count = sum(1 for r in schedule if r in CODE_FIRST_ROLES)
    proof_count = sum(1 for r in schedule if r in PROOF_FIRST_ROLES)

    if code_count < 2:
        for i in range(len(schedule)):
            if schedule[i] not in CODE_FIRST_ROLES:
                schedule[i] = "algebra"
                code_count += 1
                if code_count >= 2:
                    break

    if proof_count < 2:
        for i in range(len(schedule) - 1, -1, -1):
            if schedule[i] not in PROOF_FIRST_ROLES:
                schedule[i] = "proof"
                proof_count += 1
                if proof_count >= 2:
                    break

    # Route-specific overrides
    if route in {"optimization_extremal", "geometry_inequality"} and "construction" not in schedule:
        schedule[0] = "construction"
    if route in {"functional_equation_structure", "polynomial_root_structure"} and "algebra" not in schedule:
        schedule[1] = "algebra"

    # Flag last 2 code-first workers as rephrased for diversity
    rephrase_mask = [False] * len(schedule)
    code_indices = [i for i, r in enumerate(schedule) if r in CODE_FIRST_ROLES]
    # Rephrase the last 2 code workers
    for ci in code_indices[-2:]:
        rephrase_mask[ci] = True

    return schedule, rephrase_mask


def dispatch_worker(problem_text, attempt_index, route, role_name, plan, rephrase=False):
    """Route to the appropriate worker type based on role and rephrase flag."""
    if rephrase:
        return rephrase_code_worker(problem_text, attempt_index, route, role_name, plan)
    elif role_name in CODE_FIRST_ROLES:
        return code_first_worker(problem_text, attempt_index, route, role_name, plan)
    else:
        return proof_worker(problem_text, attempt_index, route, role_name, plan)


# ---- IMPROVED SOLVE_PROBLEM ----



# ---- REPHRASED CODE WORKER ----
# Feeds the model a simplified/rephrased version of the problem
# to break it out of wrong reasoning patterns

REPHRASE_SYSTEM = """Reasoning: high

You are solving an olympiad math problem. The problem has been REPHRASED below to highlight key constraints.

Write a COMPLETE, EXECUTABLE Python script that computes the answer.

Rules:
- You MUST write Python code inside a ```python block
- Use exact arithmetic: fractions.Fraction, sympy, integers only - NO floats for final answers
- Available packages: math, itertools, functools, fractions, collections, decimal, sympy, numpy
- The script MUST print exactly one integer on its last output line: the final answer
- Include assertions or checks to verify correctness
- NO randomness, NO approximation, NO simulation-only evidence
- If the search space is small enough, ENUMERATE ALL CASES
- Code must be deterministic and terminate within 12 seconds
"""

REPHRASE_PROMPT = """Take the following math competition problem and:
1. Restate it in simpler, more explicit language
2. List ALL constraints clearly
3. State what type of answer is expected (count, minimum, maximum, value, etc.)
4. Suggest what small cases to check first

Problem:
{problem_text}

Restate the problem clearly and simply, then write Python code in a ```python block to solve it.
"""


def rephrase_code_worker(problem_text, attempt_index, route, role_name, plan):
    """Code worker that first rephrases the problem, then solves the rephrased version."""
    ensure_inference_ready()
    sandbox = SANDBOX_POOL.acquire()
    seed = CFG.startup_seed + 8000 + attempt_index * 100
    # Higher temperature for exploration
    temp = min(1.1, max(0.5, attempt_temperature(attempt_index) * 1.2))
    py_calls = 0
    tool_artifacts = []

    try:
        role_cfg = WORKER_ROLE_LIBRARY.get(role_name, WORKER_ROLE_LIBRARY["proof"])
        route_block = route_prompt(route)
        strategy_hint = domain_strategy_hint(problem_text)
        subgoals = "\n".join(f"- {x}" for x in plan.get("subgoals", [])[:4])

        # Pick a diverse strategy
        strategy_idx = attempt_index % len(CODE_STRATEGIES)
        algo_hint = CODE_STRATEGIES[strategy_idx]

        system = (
            REPHRASE_SYSTEM
            + f"\n# Role: {role_name}\n{role_cfg['instruction']}"
            + f"\n# Problem lens\n{route_block}"
            + f"\n# Algorithmic approach\n{algo_hint}"
        )

        user_msg = REPHRASE_PROMPT.format(problem_text=problem_text.strip())
        user_msg += f"\n\nDomain hint: {strategy_hint}"
        user_msg += f"\n\nSubgoals:\n{subgoals}"

        # PHASE 1: Get rephrased problem + code
        resp1 = CLIENT.chat.completions.create(
            model=REQUEST_MODEL_NAME,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user_msg},
            ],
            temperature=temp,
            top_p=0.95,
            max_tokens=min(CFG.max_tokens, 14000),
            seed=seed,
        )

        if not resp1.choices:
            return AttemptResult(
                None, 0.0, 1.0, attempt_index, seed, 0, "",
                "rephrase_no_choices", route=route, method="rephrase",
            )

        text1 = coerce_text(resp1.choices[0].message.content)
        code = extract_code_from_text(text1)

        if not code:
            # Try to parse as direct answer
            payload = extract_json_object(text1)
            if payload:
                ok, _ = validate_candidate_payload(payload)
                if ok:
                    result = make_attempt_result(
                        payload["answer"], 0.5 if payload["answer"] is not None else 0.0,
                        0.7, attempt_index, seed, 0, text1, route,
                        tool_artifacts=tool_artifacts,
                    )
                    result.method = "rephrase"
                    return result
            ans = extract_answer(text1)
            result = make_attempt_result(
                ans, 0.4 if ans is not None else 0.0,
                0.8, attempt_index, seed, 0, text1, route,
                None if ans is not None else "rephrase_no_code",
                tool_artifacts=tool_artifacts,
            )
            result.method = "rephrase"
            return result

        # PHASE 2: Execute the code
        py_calls = 1
        timeout = CFG.python_timeout_seconds + 4
        output = sandbox.execute(code, timeout=timeout)
        tool_artifacts.append(build_tool_artifact(code, output))

        # PHASE 3: Interpret the result
        answer_msg = (
            f"Original problem:\n{problem_text.strip()}\n\n"
            f"Python code:\n```python\n{code}\n```\n\n"
            f"Execution result:\n{output}\n\n"
            "Based on the code and its output, provide the final answer as JSON."
        )
        resp2 = CLIENT.chat.completions.create(
            model=REQUEST_MODEL_NAME,
            messages=[
                {"role": "system", "content": ANSWER_FROM_CODE_SYSTEM},
                {"role": "user", "content": answer_msg},
            ],
            temperature=0.0,
            max_tokens=2000,
            seed=seed + 1000,
        )

        combined = text1
        if resp2.choices:
            text2 = coerce_text(resp2.choices[0].message.content)
            combined = text1 + "\n---\n" + text2

        payload = extract_json_object(combined)
        if payload is not None:
            ok, err = validate_candidate_payload(payload)
            if ok:
                weight = 1.30  # Tool-supported bonus
                result = make_attempt_result(
                    payload["answer"],
                    weight if payload["answer"] is not None else 0.0,
                    0.4, attempt_index, seed, py_calls, combined, route,
                    tool_artifacts=tool_artifacts,
                )
                result.method = "rephrase"
                return result

        ans = extract_answer(combined)
        if ans is None and output:
            ans = extract_answer(output)
        result = make_attempt_result(
            ans, 0.8 if ans is not None else 0.0,
            0.5, attempt_index, seed, py_calls, combined, route,
            None if ans is not None else "rephrase_code_no_answer",
            tool_artifacts=tool_artifacts,
        )
        result.method = "rephrase"
        return result

    except Exception as e:
        return AttemptResult(
            None, 0.0, 1.0, attempt_index, seed, py_calls, "",
            f"rephrase:{type(e).__name__}: {str(e)[:200]}",
            route=route, method="rephrase",
        )
    finally:
        SANDBOX_POOL.release(sandbox)


# ---- CHALLENGE / SELF-CRITIQUE WORKER ----

CHALLENGE_SYSTEM = """Reasoning: high

You are a critical math verifier. Another solver claims the answer to the problem below is {majority_answer}.

Your job:
1. Write Python code to CHECK if {majority_answer} is correct
2. If {majority_answer} is WRONG, compute the correct answer
3. Be skeptical - test edge cases, boundary cases, and small exact instances before trusting the claim

Write your verification code in a ```python block.
The code MUST:
- Explicitly try to falsify {majority_answer} on exact small/reduced instances before confirming it
- Explicitly test whether {majority_answer} satisfies the problem constraints
- For extremal, threshold, or counting claims, compare nearby candidates such as n-1, n, and n+1 when meaningful
- Use exhaustive enumeration on reduced instances whenever feasible
- Print "VERIFIED: {majority_answer}" if correct, or "WRONG: <correct_answer>" if incorrect
- Use exact arithmetic (no floats for answers)
- Available packages: math, itertools, functools, fractions, collections, decimal, sympy, numpy
"""

CHALLENGE_ANSWER_SYSTEM = """You verified a claimed answer to a math problem.
Given the verification code and its output, determine the final answer.

Return ONLY one JSON object:
{
  "answer": <integer 0..99999 or null>,
  "status": "verified_correct" | "found_error" | "inconclusive",
  "method": "pot_python",
  "proof_sketch": "<what the verification found>",
  "certificate_type": "counterexample_check" | "exhaustive_check" | "python_check" | "none",
  "certificate_summary": "<what was checked and what the result was>",
  "assumptions": [],
  "self_checks": [],
  "final_statement": "<one sentence>"
}
No extra text.
"""

CODE_STRATEGIES = [
    "Use brute-force enumeration to check ALL cases. Even if the search space seems large, try to enumerate exactly.",
    "Use a mathematical formula or closed-form expression. Derive it symbolically with sympy if needed.",
    "Use dynamic programming or recursion with memoization. Build up from small cases.",
    "Use inclusion-exclusion or generating functions. Count by complementary counting if helpful.",
]

CHALLENGE_STRATEGIES = [
    "Start by brute-forcing the smallest nontrivial instances and try to falsify the claimed answer.",
    "Search for a construction or obstruction that proves the claimed answer is too small or too large.",
    "If the problem looks extremal or threshold-based, compare nearby candidates such as n-1, n, and n+1.",
    "Derive an invariant or recurrence and verify the first exact cases with python before trusting the claim.",
]


def challenge_worker(problem_text, attempt_index, route, majority_answer, plan):
    """Self-critique worker that tries to DISPROVE the majority answer."""
    ensure_inference_ready()
    sandbox = SANDBOX_POOL.acquire()
    seed = CFG.startup_seed + 9000 + attempt_index * 100
    py_calls = 0
    tool_artifacts = []

    try:
        route_block = route_prompt(route)
        strategy_hint = domain_strategy_hint(problem_text)
        challenge_strategy = CHALLENGE_STRATEGIES[attempt_index % len(CHALLENGE_STRATEGIES)]

        system = CHALLENGE_SYSTEM.replace("{majority_answer}", str(majority_answer))
        system += f"\n# Problem lens\n{route_block}"

        user_msg = (
            problem_text.strip()
            + "\n\nThe claimed answer is: " + str(majority_answer)
            + "\n\nDomain hint: " + strategy_hint
            + "\n\nAdversarial strategy: " + challenge_strategy
            + "\n\nWrite Python code to VERIFY or DISPROVE this answer. Be thorough."
        )

        resp1 = CLIENT.chat.completions.create(
            model=REQUEST_MODEL_NAME,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user_msg},
            ],
            temperature=CFG.challenge_temperature,
            top_p=0.95,
            max_tokens=min(CFG.max_tokens, 12000),
            seed=seed,
        )

        if not resp1.choices:
            return AttemptResult(
                None, 0.0, 1.0, attempt_index, seed, 0, "",
                "challenge_no_choices", route=route, method="challenge",
            )

        text1 = coerce_text(resp1.choices[0].message.content)
        code = extract_code_from_text(text1)

        if not code:
            return AttemptResult(
                None, 0.0, 1.0, attempt_index, seed, 0, text1,
                "challenge_no_code", route=route, method="challenge",
            )

        py_calls = 1
        output = sandbox.execute(code, timeout=CFG.python_timeout_seconds + 8)
        tool_artifacts.append(build_tool_artifact(code, output))

        # Interpret the result
        answer_msg = (
            f"Original problem:\n{problem_text.strip()}\n\n"
            f"Claimed answer: {majority_answer}\n\n"
            f"Verification code:\n```python\n{code}\n```\n\n"
            f"Execution result:\n{output}\n\n"
            "Based on the verification, provide the final answer as JSON."
        )

        resp2 = CLIENT.chat.completions.create(
            model=REQUEST_MODEL_NAME,
            messages=[
                {"role": "system", "content": CHALLENGE_ANSWER_SYSTEM},
                {"role": "user", "content": answer_msg},
            ],
            temperature=0.0,
            max_tokens=2000,
            seed=seed + 1000,
        )

        combined = text1
        if resp2.choices:
            text2 = coerce_text(resp2.choices[0].message.content)
            combined = text1 + "\n---\n" + text2

        payload = extract_json_object(combined)
        if payload is not None:
            ok, err = validate_candidate_payload(payload)
            if ok:
                ans = payload["answer"]
                # If challenge found a DIFFERENT answer, give it a massive boost
                if ans is not None and int(ans) != int(majority_answer):
                    weight = CFG.challenge_discovery_weight
                elif ans is not None:
                    weight = CFG.challenge_confirm_weight
                else:
                    weight = 0.0
                status = payload.get("status", "")
                result = make_attempt_result(
                    ans, weight,
                    0.3, attempt_index, seed, py_calls, combined, route,
                    tool_artifacts=tool_artifacts,
                )
                result.method = "challenge"
                return result

        ans = extract_answer(combined)
        if ans is None and output:
            ans = extract_answer(output)
        result = make_attempt_result(
            ans, 1.0 if ans is not None else 0.0,
            0.5, attempt_index, seed, py_calls, combined, route,
            None if ans is not None else "challenge_no_answer",
            tool_artifacts=tool_artifacts,
        )
        result.method = "challenge"
        return result

    except Exception as e:
        return AttemptResult(
            None, 0.0, 1.0, attempt_index, seed, py_calls, "",
            f"challenge:{type(e).__name__}: {str(e)[:200]}",
            route=route, method="challenge",
        )
    finally:
        SANDBOX_POOL.release(sandbox)


def solve_problem_legacy_v2(problem_text, problem_id=None):
    """Plan-driven solver with forced code workers + proof workers."""
    global PROBLEMS_REMAINING
    started = time.time()
    elapsed = time.time() - NOTEBOOK_START
    time_left = max(0.0, CFG.notebook_budget_seconds - elapsed)

    if time_left < CFG.hard_deadline_reserve_seconds:
        PROBLEMS_REMAINING = max(0, PROBLEMS_REMAINING - 1)
        return coerce_submission_int(
            fallback_submission_answer(
                problem_text, problem_id=problem_id,
                route="deadline", reason="hard_deadline_reserve",
            )
        )

    problem_route = classify_problem_route(problem_text)
    problem_routes, route_scores = classify_problem_routes(problem_text)
    force_full = problem_requires_full_attempts(problem_text, route=problem_route)
    deadline = time.time() + max(60.0, time_left / max(1, PROBLEMS_REMAINING))
    strategy_hint = domain_strategy_hint(problem_text)

    plan = plan_problem(problem_text, problem_route)
    worker_schedule, rephrase_mask = build_mixed_worker_schedule(plan, problem_route)

    write_debug_event({
        "event": "problem_start_v3",
        "problem_id": problem_id,
        "route": "chat_plan_workers_v2",
        "problem_route": problem_route,
        "worker_schedule": worker_schedule,
        "plan": {k: v for k, v in plan.items() if not str(k).startswith("_")},
        "problem_excerpt": clip_text(problem_text, 500),
    })

    # ---- Phase 1: Run mixed code + proof workers ----
    results = []
    futures = {}
    with ThreadPoolExecutor(max_workers=max(CFG.workers, MIN_TOTAL_WORKERS)) as ex:
        for i, role_name in enumerate(worker_schedule):
            is_rephrased = rephrase_mask[i] if i < len(rephrase_mask) else False
            futures[ex.submit(
                dispatch_worker,
                problem_text, i, problem_route, role_name, plan, rephrase=is_rephrased,
            )] = (i, role_name)

        for fut in as_completed(futures):
            if time.time() > deadline:
                break
            idx, role_name = futures[fut]
            try:
                r = fut.result(timeout=max(5.0, deadline - time.time()))
            except Exception as e:
                r = AttemptResult(
                    None, 0.0, 1.0, idx, 0, 0, "",
                    f"future:{type(e).__name__}",
                    route=problem_route, method=role_name,
                )
            results.append(r)

            counts = Counter(x.answer for x in results if x.answer is not None)
            if (not force_full) and counts and counts.most_common(1)[0][1] >= CFG.early_stop_votes:
                break

    ranked, details, best_attempt_by_answer = aggregate_candidate_stats(results)
    valid = [r for r in results if r.answer is not None]

    # ---- Phase 2: If too few valid results, add chat fallback ----
    if len(valid) < 3 and time.time() < deadline - 60:
        write_debug_event({
            "event": "fallback_to_chat_v3",
            "problem_id": problem_id,
            "valid_count": len(valid),
        })
        with ThreadPoolExecutor(max_workers=CFG.workers) as ex:
            chat_futures = {
                ex.submit(chat_attempt, problem_text, i, problem_route): i
                for i in range(CFG.attempts)
            }
            for fut in as_completed(chat_futures):
                if time.time() > deadline:
                    break
                try:
                    r = fut.result(timeout=max(5.0, deadline - time.time()))
                except Exception as exc:
                    r = AttemptResult(
                        None, 0.0, 1.0, chat_futures[fut], 0, 0, "",
                        f"chat_future:{type(exc).__name__}",
                    )
                results.append(r)

        ranked, details, best_attempt_by_answer = aggregate_candidate_stats(results)
        valid = [r for r in results if r.answer is not None]

    # ---- Phase 2.5: Self-critique / challenge round ----
    # If strong consensus exists, challenge it with verification workers
    ranked_for_critique, _, _ = aggregate_candidate_stats(results)
    if ranked_for_critique and time.time() < deadline - 90:
        top_answer = int(ranked_for_critique[0][0])
        top_count = sum(1 for r in results if r.answer is not None and int(r.answer) == top_answer)
        total_valid = sum(1 for r in results if r.answer is not None)

        # Trigger challenge only on close races; do not spend challenger budget on already-settled majorities.
        close_race = len(ranked) > 1 and float(ranked[1][1]) >= CFG.challenge_close_ratio * max(float(ranked[0][1]), 1e-9)
        if close_race:
            write_debug_event({
                "event": "challenge_round",
                "problem_id": problem_id,
                "majority_answer": top_answer,
                "majority_votes": top_count,
                "total_valid": total_valid,
            })

            challenge_results = []
            n_challengers = 3
            with ThreadPoolExecutor(max_workers=n_challengers) as ex:
                ch_futures = {
                    ex.submit(
                        challenge_worker, problem_text, 100 + ci,
                        problem_route, top_answer, plan,
                    ): ci
                    for ci in range(n_challengers)
                }
                for fut in as_completed(ch_futures):
                    if time.time() > deadline - 30:
                        break
                    try:
                        r = fut.result(timeout=max(5.0, deadline - time.time() - 30))
                    except Exception as exc:
                        r = AttemptResult(
                            None, 0.0, 1.0, ch_futures[fut], 0, 0, "",
                            f"challenge_future:{type(exc).__name__}",
                            route=problem_route, method="challenge",
                        )
                    challenge_results.append(r)

            # Merge challenge results into the pool
            results.extend(challenge_results)
            ranked, details, best_attempt_by_answer = aggregate_candidate_stats(results)
            valid = [r for r in results if r.answer is not None]
            details["challenge_round"] = {
                "majority_challenged": top_answer,
                "challengers": len(challenge_results),
                "challenge_answers": [r.answer for r in challenge_results if r.answer is not None],
            }

    # ---- Phase 3: Candidate quality adjustments ----
    # Penalize answer=0 for non-trivial counting/optimization routes
    if 0 in best_attempt_by_answer and problem_route in {
        "combinatorics_counting", "optimization_extremal", "subset_sum_threshold",
        "number_theory_exact", "recurrence_sequence_structure",
    }:
        attempt_0 = best_attempt_by_answer[0]
        if attempt_0.python_calls == 0:
            # answer=0 without tool support in a counting/optimization problem is suspicious
            for i, (ans, sc) in enumerate(ranked):
                if int(ans) == 0:
                    ranked[i] = (ans, sc * 0.15)
            ranked.sort(key=lambda x: x[1], reverse=True)
            details["zero_penalty"] = True

    answer = choose_weighted_winner(ranked, details, best_attempt_by_answer)

    # ---- Phase 4: Adversarial verifier ----
    verifier_result = None
    need_verifier = (
        should_run_verifier(problem_route, ranked, details)
        or (answer is None and len(ranked) >= 2)
        or (len(ranked) >= 2 and details.get("zero_penalty"))
    )
    if need_verifier and time.time() < deadline - 30:
        verifier_result = run_candidate_verifier_v2(
            problem_text, problem_route, strategy_hint, plan,
            ranked, details, best_attempt_by_answer,
        )
        if verifier_result and verifier_result.get("answer") is not None:
            proposed_answer = int(verifier_result["answer"])
            nonzero_machine_answers = sorted(
                int(ans)
                for ans, attempt in best_attempt_by_answer.items()
                if int(ans) != 0
                and is_machine_checkable_attempt(
                    attempt,
                    int(ans),
                    int((details.get("tools") or {}).get(int(ans), 0)),
                )
            )
            reject_zero_override = bool(
                proposed_answer == 0
                and details.get("zero_penalty")
                and nonzero_machine_answers
            )
            if reject_zero_override:
                details["verifier_override_rejected"] = {
                    "proposed_answer": proposed_answer,
                    "reason": "zero_penalty_nonzero_machine_candidate_present",
                    "nonzero_machine_answers": nonzero_machine_answers[:5],
                }
            else:
                answer = proposed_answer

    # ---- Finalize ----
    PROBLEMS_REMAINING = max(0, PROBLEMS_REMAINING - 1)

    if verifier_result is not None:
        details["verifier"] = {
            "candidates": verifier_result.get("candidates"),
            "answer": verifier_result.get("answer"),
            "error": verifier_result.get("error"),
            "text": clip_text(verifier_result.get("text", ""), 1200),
        }
    if "rescue_round" in locals() and rescue_round is not None:
        details["rescue_round"] = rescue_round

    if answer is None:
        answer = fallback_submission_answer(
            problem_text, problem_id=problem_id,
            route=problem_route, reason="unresolved_v3",
        )

    elapsed_s = time.time() - started
    valid_count = sum(1 for r in results if r.answer is not None)
    tool_total = sum(1 for r in results if r.answer is not None and r.python_calls > 0)

    print(
        f"  -> answer={answer} | route=chat_plan_workers_v2"
        f" | problem_route={problem_route}"
        f" | workers={worker_schedule}"
        f" | scores={details.get('scores')}"
        f" | counts={details.get('counts')}"
        f" | tools={details.get('tools')}"
        f" | attempts={valid_count}/{len(results)}"
        f" | tool_workers={tool_total}"
        f" | remaining={PROBLEMS_REMAINING}"
        f" | elapsed={elapsed_s:.1f}s"
    )

    write_debug_event({
        "event": "problem_solved_v3",
        "problem_id": problem_id,
        "route": "chat_plan_workers_v2",
        "problem_route": problem_route,
        "answer": answer,
        "details": details,
        "worker_schedule": worker_schedule,
        "attempts": [summarize_attempt_result(r) for r in results],
        "valid_count": valid_count,
        "tool_workers": tool_total,
        "elapsed_s": round(elapsed_s, 3),
        "problem_excerpt": clip_text(problem_text, 500),
    })

    return coerce_submission_int(answer)

print(f"v3.4 loaded: code_first + proof + rephrase + stronger challenge, MIN_TOTAL_WORKERS={MIN_TOTAL_WORKERS}")


## 13. Start or Restart vLLM
Run this cell only when you are ready to query the model.

In [ ]:
FORCE_RESTART_SERVER = True
start_frontier_server(force_restart=FORCE_RESTART_SERVER)


## 14. Experiment Configuration
Either set a `problem_id` from the benchmark CSV or provide raw `EXPERIMENT_PROBLEM_TEXT`.

In [ ]:
EXPERIMENT_CSV_PATH = "/kaggle/input/datasets/ritwikakancharla/aimo-3-benchmark/AIMO3_IMO_Bench_Eval.csv"
EXPERIMENT_PROBLEM_ID = "imo-bench-algebra-005"
EXPERIMENT_ROW_INDEX = 0
EXPERIMENT_PROBLEM_TEXT = None
EXPERIMENT_RESET_BUDGET = True


## 15. Run One Experiment
Executes a single solve using the current experiment configuration.

In [ ]:
experiment_result = run_single_experiment(
    problem_text=EXPERIMENT_PROBLEM_TEXT,
    problem_id=EXPERIMENT_PROBLEM_ID,
    csv_path=EXPERIMENT_CSV_PATH,
    row_index=EXPERIMENT_ROW_INDEX,
    reset_budget=EXPERIMENT_RESET_BUDGET,
)
experiment_result


## 16. Optional Batch or Kaggle Serve
Manual entry points for batch evaluation, local runs, or the Kaggle gateway.

In [ ]:
import pandas as pd
import time
from pathlib import Path

def run_batch_experiment(
    csv_path,
    start_idx=0,
    end_idx=None,
    output_path="/kaggle/working/aimo3_batch_results.csv",
):
    df = pd.read_csv(csv_path)
    if end_idx is None:
        end_idx = len(df)

    id_col = "Problem ID"
    problem_col = "Problem"
    answer_col = "Short Answer"

    total = min(end_idx, len(df)) - start_idx
    results = []

    ok_count = 0
    miss_count = 0
    error_count = 0

    batch_start = time.time()

    for offset, idx in enumerate(range(start_idx, min(end_idx, len(df))), start=1):
        row = df.iloc[idx]
        problem_id = str(row[id_col])
        problem_text = str(row[problem_col])
        expected_answer = row[answer_col] if answer_col in df.columns else None

        print("\n" + "=" * 100)
        print(f"RUN ({offset}/{total}) | row={idx} | problem_id={problem_id}")
        print("=" * 100)

        started = time.time()

        try:
            result = run_single_experiment(
                problem_text=problem_text,
                problem_id=problem_id,
                csv_path=csv_path,
                row_index=idx,
                reset_budget=True,
            )

            predicted = None
            route = None
            problem_route = None
            failure_class = None
            elapsed_s = round(time.time() - started, 2)

            if isinstance(result, dict):
                predicted = result.get("predicted_answer", result.get("answer"))
                route = result.get("route")
                problem_route = result.get("problem_route")
                failure_class = result.get("failure_class")

            is_correct = None
            if expected_answer is not None and predicted is not None:
                try:
                    is_correct = int(predicted) == int(expected_answer)
                except Exception:
                    is_correct = False

            if is_correct is True:
                ok_count += 1
                label = "OK"
            else:
                miss_count += 1
                label = "MISS"

            print(
                f"{label} ({offset}/{total}) | "
                f"problem_id={problem_id} | "
                f"pred={predicted} | exp={expected_answer} | "
                f"ok={ok_count} miss={miss_count} err={error_count} | "
                f"elapsed={elapsed_s}s"
            )

            attempted = ok_count + miss_count
            acc = (ok_count / attempted) if attempted > 0 else 0.0
            print(
                f"TRACKER | completed={attempted}/{total} | "
                f"accuracy={acc:.4f} | remaining={total - offset}"
            )

            results.append({
                "row_index": idx,
                "problem_id": problem_id,
                "expected_answer": expected_answer,
                "predicted_answer": predicted,
                "is_correct": is_correct,
                "status": "ok",
                "label": label,
                "route": route,
                "problem_route": problem_route,
                "failure_class": failure_class,
                "elapsed_s": elapsed_s,
            })

        except Exception as e:
            error_count += 1
            elapsed_s = round(time.time() - started, 2)

            print(
                f"ERROR ({offset}/{total}) | "
                f"problem_id={problem_id} | "
                f"ok={ok_count} miss={miss_count} err={error_count} | "
                f"elapsed={elapsed_s}s | error={e}"
            )

            attempted = ok_count + miss_count
            acc = (ok_count / attempted) if attempted > 0 else 0.0
            print(
                f"TRACKER | completed={attempted + error_count}/{total} | "
                f"accuracy(on non-error)={acc:.4f} | remaining={total - offset}"
            )

            results.append({
                "row_index": idx,
                "problem_id": problem_id,
                "expected_answer": expected_answer,
                "predicted_answer": None,
                "is_correct": False,
                "status": "error",
                "label": "ERROR",
                "route": None,
                "problem_route": None,
                "failure_class": "exception",
                "elapsed_s": elapsed_s,
                "error": str(e),
            })

        pd.DataFrame(results).to_csv(output_path, index=False)

    total_elapsed = round(time.time() - batch_start, 2)
    results_df = pd.DataFrame(results)

    attempted = ok_count + miss_count
    acc = (ok_count / attempted) if attempted > 0 else 0.0

    print("\n" + "#" * 100)
    print(
        f"FINAL TRACKER | total={total} | ok={ok_count} | miss={miss_count} | "
        f"error={error_count} | accuracy(on non-error)={acc:.4f} | total_elapsed={total_elapsed}s"
    )
    print(f"Saved to: {output_path}")
    print("#" * 100)

    return results_df

In [ ]:
## Quick Validation: 11 problems (3 algebra regression + 2 easy combinatorics + 6 hard combinatorics)
# Estimated runtime: ~55 minutes
# Rows: 0,1,8 (algebra), 30,31 (easy combinatorics), 42-47 (hard combinatorics that all failed before)

batch_df = run_batch_experiment(
    csv_path="/kaggle/input/datasets/ritwikakancharla/aimo-3-benchmark/AIMO3_IMO_Bench_Eval.csv",
    start_idx=42,
    end_idx=48,
    output_path="/kaggle/working/aimo3_validation_results.csv",
)
display(batch_df)

In [ ]:
batch_df = run_batch_experiment(
    csv_path="/kaggle/input/datasets/ritwikakancharla/aimo-3-benchmark/AIMO3_IMO_Bench_Eval.csv",
    start_idx=3,
    end_idx=197,
    output_path="/kaggle/working/aimo3_batch_results.csv",
)
display(batch_df)

In [ ]:
# Accuracy patch: model selection, parser robustness, stateless tool sandboxes,
# route ensembles, and stronger tool verification nudges.

CFG.early_stop_votes = min(3, max(2, CFG.attempts))
CFG.attempt_temperatures = (0.0, 0.15, 0.35, 0.55, 0.75, 0.9)
CFG.tool_rounds = min(CFG.tool_rounds, 6)
CFG.rescue_attempts = int(os.getenv("AIMO3_RESCUE_ATTEMPTS", "3"))
CFG.rescue_close_ratio = float(os.getenv("AIMO3_RESCUE_CLOSE_RATIO", "0.72"))
CFG.rescue_min_time_left = int(os.getenv("AIMO3_RESCUE_MIN_TIME_LEFT", "45"))
CFG.rescue_temperatures = (0.55, 0.75, 0.9)

FORCE_PYTHON_ROUTES = {
    "combinatorics_counting",
    "number_theory_exact",
    "recurrence_sequence_structure",
    "polynomial_root_structure",
    "functional_equation_structure",
    "subset_sum_threshold",
    "optimization_extremal",
}

def _route_set(route):
    if isinstance(route, (list, tuple, set)):
        return {str(x) for x in route}
    return {str(route)}

def normalize_answer(value):
    iv = int(str(value).strip())
    if not (0 <= iv <= 99999):
        raise ValueError(f"answer_out_of_range:{iv}")
    return iv

def extract_json_object(text):
    text = str(text or "")
    if not text.strip():
        return None
    decoder = json.JSONDecoder()
    candidates = []
    for i, ch in enumerate(text):
        if ch != "{":
            continue
        try:
            obj, end = decoder.raw_decode(text[i:])
        except Exception:
            continue
        if isinstance(obj, dict):
            keys = set(obj.keys())
            answerish = len(keys & {"answer", "selected_answer", "final_answer", "answer_mod_100000"})
            schema = len(keys & {"status", "method", "proof_sketch", "certificate_type", "certificate_summary", "assumptions", "self_checks", "final_statement"})
            candidates.append(((answerish, schema, len(keys), i), obj))
    if not candidates:
        return None
    candidates.sort(key=lambda item: item[0], reverse=True)
    return candidates[0][1]

def extract_answer(text):
    text = str(text or "")
    obj = extract_json_object(text)
    if isinstance(obj, dict):
        for key in ("answer", "selected_answer", "final_answer", "answer_mod_100000"):
            if obj.get(key) is None:
                continue
            try:
                return normalize_answer(obj[key])
            except Exception:
                pass
    boxed = BOXED_RE.findall(text)
    if boxed:
        ints = INT_RE.findall(boxed[-1])
        if ints:
            try:
                return normalize_answer(int(ints[-1]))
            except Exception:
                pass
    stripped = text.strip()
    last_line = stripped.splitlines()[-1].strip() if stripped else ""
    for candidate_text in (last_line, stripped[-240:]):
        for pattern in (
            r"(?i)\bfinal answer\s*(?:is|=|:)?\s*(-?\d+)\b",
            r"(?i)\banswer\s*(?:is|=|:)\s*(-?\d+)\b",
            r"^\s*(-?\d+)\s*$",
        ):
            m = re.search(pattern, candidate_text)
            if not m:
                continue
            try:
                return normalize_answer(int(m.group(1)))
            except Exception:
                pass
    return None

class NoReuseSandboxPool:
    def acquire(self):
        # Fresh sandbox per attempt avoids cross-attempt / cross-problem state leakage.
        return SubprocessSandbox()
    def release(self, sandbox):
        try:
            sandbox.close()
        except Exception:
            pass
    def close(self):
        return None

SANDBOX_POOL = NoReuseSandboxPool()

CHAT_ROLE_SYSTEM_TEMPLATE = """# Instructions
You are a specialized olympiad worker.

# Primary role
{role_name}

# Role objective
{role_instruction}

# Global problem lens
{route_block}

# Planner subgoals
{subgoals}

# Verification focus
{verify_focus}

# Tool guidance
{tool_guidance}

# Attempt style
{attempt_block}

Solve the olympiad problem exactly.
Use Python only when it reduces risk.
If exact justification is missing, return answer=null instead of guessing.
If you need code execution next, return one ```python``` block only.
If you are finalizing, return only one JSON object with no markdown fences.
Use this JSON shape:
{{
  "answer": <int or null>,
  "status": "solved" | "inconclusive",
  "proof_sketch": "<short exact justification>",
  "certificate_type": "<none|python_exact|case_split|algebraic|construction|extremal>",
  "certificate_summary": "<short certificate summary>",
  "assumptions": ["<optional short items>"],
  "self_checks": ["<optional short items>"],
  "final_statement": "<one sentence conclusion>"
}}
"""


def select_worker_model_role(role_name, attempt_index, route):
    if CONTRARIAN_MODEL_PATH is None:
        return "primary"
    route_set = _route_set(route)
    if role_name in {"counterexample", "casework"}:
        return "contrarian"
    if (
        role_name == "algebra"
        and attempt_index >= max(2, CFG.attempts - 2)
        and route_set & {"functional_equation_structure", "number_theory_exact", "polynomial_root_structure", "recurrence_sequence_structure"}
    ):
        return "contrarian"
    return "primary"

def build_worker_schedule(plan, route):
    route_set = _route_set(route)
    roles = [r for r in plan.get("worker_roles", []) if r in WORKER_ROLE_LIBRARY]
    if not roles:
        roles = ["proof", "algebra", "counterexample"]
    roles = roles[: max(1, min(5, len(roles)))]
    schedule = [roles[i % len(roles)] for i in range(CFG.attempts)]
    if route_set & {"optimization_extremal", "geometry_inequality"} and "construction" not in schedule:
        schedule[0] = "construction"
    if route_set & {"functional_equation_structure", "polynomial_root_structure"} and "algebra" not in schedule and len(schedule) > 1:
        schedule[1] = "algebra"
    if route_set & {"recurrence_sequence_structure"} and "recurrence" not in schedule and len(schedule) > 1:
        schedule[0] = "recurrence"
    if route_set & {"combinatorics_counting", "subset_sum_threshold"} and "casework" not in schedule:
        schedule[0] = "casework"
    if "proof" not in schedule:
        schedule[-1] = "proof"
    if CFG.enable_aux_models and CONTRARIAN_MODEL_PATH is not None:
        if route_set & {"combinatorics_counting", "subset_sum_threshold"}:
            if "counterexample" not in schedule and len(schedule) > 1:
                schedule[-2] = "counterexample"
            if "casework" not in schedule and len(schedule) > 2:
                schedule[-3] = "casework"
        elif route_set & {"functional_equation_structure", "number_theory_exact", "polynomial_root_structure", "recurrence_sequence_structure", "optimization_extremal", "geometry_inequality"}:
            if "counterexample" not in schedule and len(schedule) > 1:
                schedule[-2] = "counterexample"
    return schedule

def build_candidate_briefs_v2(candidates, details, best_attempt_by_answer):
    briefs = []
    for answer in candidates:
        attempt = best_attempt_by_answer.get(int(answer))
        if attempt is None:
            continue
        artifacts = []
        for artifact in list(attempt.certificate_payload or ())[:2]:
            artifacts.append({
                "status": artifact.get("status"),
                "final_answer": artifact.get("final_answer"),
                "code": clip_text(artifact.get("code", ""), 360),
                "output": clip_text(artifact.get("output", ""), 420),
            })
        briefs.append({
            "answer": int(answer),
            "score": float(details["scores"].get(int(answer), 0.0)),
            "votes": int(details["counts"].get(int(answer), 0)),
            "worker_method": attempt.method,
            "certificate_type": attempt.certificate_type,
            "certificate_summary": attempt.certificate_summary,
            "tool_confused": bool(attempt.tool_confused),
            "approx_only": bool(attempt.approx_only),
            "random_search": bool(attempt.random_search),
            "weak_exactness": bool(attempt.weak_exactness),
            "proof_sketch": clip_text(attempt.final_text, 1400),
            "tool_artifacts": artifacts,
        })
    return briefs

def plan_problem(problem_text, route):
    client, model_spec = ensure_inference_ready(role="primary")
    route_block = route_prompt(route)
    route_label = route[0] if isinstance(route, (list, tuple)) and route else route
    prompt = (
        problem_text.strip()
        + "\n\nPrimary route:\n" + str(route_label)
        + "\n\nRoute context:\n" + route_block
        + "\n\nReturn JSON only."
    )
    try:
        response = client.chat.completions.create(
            model=model_spec["request_model_name"],
            messages=build_model_chat_messages(model_spec, "planner", PLANNER_SYSTEM_PROMPT, prompt),
            temperature=0.0,
            top_p=0.95,
            max_tokens=1200,
            seed=CFG.startup_seed + 6100,
        )
        if not response.choices:
            raise RuntimeError("planner_no_choices")
        text = sanitize_model_response_text(response.choices[0].message.content, model_spec["family"])
        obj = extract_json_object(text)
        if obj is None:
            raise RuntimeError("planner_bad_json")
        ok, err = validate_plan_payload(obj)
        if not ok:
            raise RuntimeError(err)
        obj["_raw"] = clip_text(text, 1200)
        return obj
    except Exception as exc:
        fallback_roles = ["proof", "algebra", "counterexample"]
        route_set = _route_set(route)
        if route_set & {"optimization_extremal", "geometry_inequality"}:
            fallback_roles = ["construction", "proof", "counterexample"]
        elif route_set & {"recurrence_sequence_structure"}:
            fallback_roles = ["recurrence", "proof", "algebra"]
        elif route_set & {"subset_sum_threshold", "combinatorics_counting"}:
            fallback_roles = ["casework", "construction", "counterexample"]
        return {
            "problem_type": str(route_label),
            "subgoals": [
                "Isolate the structural core of the problem.",
                "Derive or refute candidate patterns using exact reasoning.",
                "Verify the final candidate with an independent exact check.",
            ],
            "worker_roles": fallback_roles,
            "verification_focus": ["Check hidden edge cases and reject unsupported patterns."],
            "tool_guidance": ["Use python for exact enumeration, symbolic algebra, or deterministic verification before finalizing."],
            "_raw": f"planner_fallback:{type(exc).__name__}:{str(exc)[:160]}",
        }

def _needs_python_verification(route, role_name):
    route_set = _route_set(route)
    if role_name in {"algebra", "casework", "counterexample", "recurrence"}:
        return True
    return bool(route_set & FORCE_PYTHON_ROUTES)

def chat_role_attempt_with_tools(
    problem_text,
    attempt_index,
    route,
    role_name,
    plan,
    extra_user_note=None,
    temperature_override=None,
):
    client, model_spec = ensure_inference_ready(role=select_worker_model_role(role_name, attempt_index, route))
    sandbox = SANDBOX_POOL.acquire()
    py_calls = 0
    tool_artifacts = []
    all_texts = []
    seed = int((CFG.startup_seed + attempt_index + 1) ** 2)
    temp = max(0.10, attempt_temperature(attempt_index))
    if temperature_override is not None:
        temp = float(temperature_override)
    nudged_for_python = False

    try:
        role_cfg = WORKER_ROLE_LIBRARY.get(role_name, WORKER_ROLE_LIBRARY["proof"])
        route_block = route_prompt(route)
        attempt_block = route_attempt_stance(route[0] if isinstance(route, (list, tuple)) and route else route, attempt_index)
        subgoals = "\n".join(f"- {x}" for x in plan.get("subgoals", [])[:6])
        verify_focus = "\n".join(f"- {x}" for x in plan.get("verification_focus", [])[:6])
        tool_guidance = "\n".join(f"- {x}" for x in plan.get("tool_guidance", [])[:6])
        strategy_hint = domain_strategy_hint(problem_text)

        system_prompt = CHAT_ROLE_SYSTEM_TEMPLATE.format(
            role_name=role_name,
            role_instruction=role_cfg["instruction"],
            route_block=route_block,
            subgoals=subgoals,
            verify_focus=verify_focus,
            tool_guidance=tool_guidance,
            attempt_block=attempt_block,
        )

        user_content = problem_text.strip() + "\n\nDomain strategy hint:\n" + strategy_hint
        if extra_user_note:
            user_content += "\n\nSpecial instruction:\n" + str(extra_user_note).strip()
        messages = build_model_chat_messages(model_spec, "worker", system_prompt, user_content)

        for turn in range(CFG.tool_rounds):
            try:
                response = client.chat.completions.create(
                    model=model_spec["request_model_name"],
                    messages=messages,
                    temperature=temp,
                    top_p=0.95,
                    max_tokens=min(CFG.max_tokens, 8192),
                    seed=seed + turn,
                )
            except Exception:
                break

            if not response.choices:
                break

            text = sanitize_model_response_text(response.choices[0].message.content, model_spec["family"])
            all_texts.append(text)
            code_blocks = CODE_BLOCK_RE.findall(text)

            if code_blocks and turn < CFG.tool_rounds - 1:
                code = code_blocks[-1].strip()
                py_calls += 1
                timeout = CFG.python_timeout_seconds + (4 if role_name in {"algebra", "counterexample", "casework", "recurrence"} else 0)
                output = sandbox.execute(code, timeout=timeout)
                tool_artifacts.append(build_tool_artifact(code, output))
                messages.append({"role": "assistant", "content": text})
                messages.append({
                    "role": "user",
                    "content": (
                        f"Python execution result:\n{output}\n\n"
                        "Use this exact result. If it changes your candidate, say so. "
                        "If you are ready, return the final JSON only."
                    ),
                })
                continue

            payload = extract_json_object(text)
            certificate_type = payload.get("certificate_type") if isinstance(payload, dict) else None
            if (
                not nudged_for_python
                and py_calls == 0
                and turn == 0
                and _needs_python_verification(route, role_name)
                and certificate_type not in MACHINE_CHECKABLE_CERTIFICATES
                and turn < CFG.tool_rounds - 1
            ):
                nudged_for_python = True
                messages.append({"role": "assistant", "content": text})
                messages.append({
                    "role": "user",
                    "content": (
                        "Before finalizing, do one exact Python verification. "
                        "Use deterministic enumeration, a symbolic algebra check, or a contradiction search directly tied to your candidate. "
                        "Then return the final JSON only."
                    ),
                })
                continue
            break

        final_text = all_texts[-1] if all_texts else ""
        combined = "\n".join(all_texts)

        payload = extract_json_object(final_text) or extract_json_object(combined)
        if payload is not None:
            ok, err = validate_candidate_payload(payload)
            if ok:
                weight = 1.0
                if py_calls:
                    weight *= 1.20
                if payload.get("certificate_type") in MACHINE_CHECKABLE_CERTIFICATES:
                    weight *= 1.15
                elif _route_set(route) & FORCE_PYTHON_ROUTES and py_calls == 0:
                    weight *= 0.80
                result = make_attempt_result(
                    payload["answer"],
                    weight if payload["answer"] is not None else 0.0,
                    0.45,
                    attempt_index,
                    seed,
                    py_calls,
                    final_text or combined,
                    route[0] if isinstance(route, (list, tuple)) and route else route,
                    tool_artifacts=tool_artifacts,
                )
                result.method = role_name
                result.worker_role = role_name
                result.model_role = model_spec["role"]
                result.model_family = model_spec["family"]
                return result

        ans = extract_answer(final_text) if final_text else None
        if ans is None:
            ans = extract_answer(combined)
        weight = 0.75 if py_calls else 0.40
        if (_route_set(route) & FORCE_PYTHON_ROUTES) and py_calls == 0:
            weight *= 0.75
        result = make_attempt_result(
            ans,
            weight if ans is not None else 0.0,
            0.65,
            attempt_index,
            seed,
            py_calls,
            final_text or combined,
            route[0] if isinstance(route, (list, tuple)) and route else route,
            None if ans is not None else "no_answer",
            tool_artifacts=tool_artifacts,
        )
        result.method = role_name
        result.worker_role = role_name
        result.model_role = model_spec["role"]
        result.model_family = model_spec["family"]
        return result

    except Exception as e:
        return AttemptResult(
            None, 0.0, 1.0, attempt_index, seed, py_calls, "",
            f"{type(e).__name__}: {str(e)[:200]}",
            route=route[0] if isinstance(route, (list, tuple)) and route else route,
            method=role_name,
        )
    finally:
        SANDBOX_POOL.release(sandbox)

def verifier_rejected_all(verifier_result):
    if not verifier_result or verifier_result.get("answer") is not None or verifier_result.get("error"):
        return False
    text = str(verifier_result.get("text") or "")
    return '"status": "rejected_all"' in text or '"status": "inconclusive"' in text

def rescue_worker_roles(route):
    route_set = _route_set(route)
    if route_set & {"combinatorics_counting", "subset_sum_threshold"}:
        return ["counterexample", "casework", "construction"]
    if route_set & {"number_theory_exact", "polynomial_root_structure"}:
        return ["algebra", "counterexample", "proof"]
    if route_set & {"functional_equation_structure"}:
        return ["algebra", "proof", "counterexample"]
    if route_set & {"recurrence_sequence_structure"}:
        return ["recurrence", "counterexample", "algebra"]
    if route_set & {"optimization_extremal", "geometry_inequality"}:
        return ["construction", "counterexample", "proof"]
    return ["counterexample", "proof", "casework"]

def build_rescue_plan(route, base_plan, ranked, details, best_attempt_by_answer):
    candidates = []
    counts = details.get("counts") or {}
    tools = details.get("tools") or {}
    for answer, _score in ranked[:3]:
        answer = int(answer)
        attempt = best_attempt_by_answer.get(answer)
        cert = "none" if attempt is None else str(getattr(attempt, "certificate_type", "none") or "none")
        method = "unknown" if attempt is None else str(getattr(attempt, "method", "unknown") or "unknown")
        votes = int(counts.get(answer, counts.get(str(answer), 0)) or 0)
        tool_count = int(tools.get(answer, tools.get(str(answer), 0)) or 0)
        candidates.append(f"{answer} (votes={votes}, tools={tool_count}, role={method}, cert={cert})")
    base_subgoals = list(base_plan.get("subgoals", [])[:4])
    base_focus = list(base_plan.get("verification_focus", [])[:4])
    base_guidance = list(base_plan.get("tool_guidance", [])[:4])
    return {
        "problem_type": base_plan.get("problem_type") or (route[0] if isinstance(route, (list, tuple)) and route else route),
        "worker_roles": rescue_worker_roles(route),
        "subgoals": base_subgoals + [
            "Re-solve independently; do not anchor on the current pool.",
            "Try to falsify the current leader before accepting any answer.",
            "Check small cases and boundary transitions exactly before generalizing.",
        ],
        "verification_focus": base_focus + [
            "Current pool candidates: " + (", ".join(candidates) if candidates else "none"),
            "If a threshold, parity, recurrence, or power pattern appears, verify neighboring cases explicitly.",
        ],
        "tool_guidance": base_guidance + [
            "Use Python for exact enumeration, symbolic verification, or contradiction search tied to your final candidate.",
            "Avoid random search; prefer assert-heavy small-case and boundary-case checks.",
        ],
        "_raw": "rescue_round",
    }

def build_rescue_instruction(route, ranked, details):
    answers = [int(answer) for answer, _score in ranked[:3]]
    leader = answers[0] if answers else None
    route_set = _route_set(route)
    parts = [
        "Current candidate answers: " + (", ".join(str(x) for x in answers) if answers else "none") + ".",
        "Treat the current leader as a hypothesis to attack, not as evidence.",
    ]
    if leader is not None:
        parts.append(f"Try to disprove {leader} first; if it fails, produce the corrected exact answer.")
    if route_set & FORCE_PYTHON_ROUTES:
        parts.append("Use Python on exact small cases, transition cases, or neighboring values before finalizing.")
    parts.append("If you cannot justify a candidate exactly, return answer=null instead of guessing.")
    return " ".join(parts)

def should_run_rescue_round(route, ranked, details, valid_count, verifier_result=None):
    if not ranked:
        return False, []
    counts = details.get("counts") or {}
    machine = details.get("machine_checkable") or {}
    leader = int(ranked[0][0])
    leader_votes = int(counts.get(leader, counts.get(str(leader), 0)) or 0)
    leader_machine = bool(machine.get(leader, machine.get(str(leader), False)))
    candidate_count = len(ranked)
    top_score = float(ranked[0][1]) if ranked else 0.0
    second_score = float(ranked[1][1]) if len(ranked) > 1 else 0.0
    close_race = len(ranked) > 1 and second_score >= CFG.rescue_close_ratio * max(top_score, 1e-9)
    exact_route = bool(_route_set(route) & FORCE_PYTHON_ROUTES)
    reasons = []
    if verifier_rejected_all(verifier_result):
        reasons.append("verifier_rejected_all")
    if leader_machine and not reasons:
        return False, []
    if valid_count <= max(2, CFG.attempts // 2):
        reasons.append("thin_valid_pool")
    if candidate_count <= 2:
        reasons.append("candidate_pool_too_small")
    if leader_votes <= 2 and close_race:
        reasons.append("close_low_support")
    if exact_route and leader_votes <= 2:
        reasons.append("exact_route_weak_consensus")
    return bool(reasons), reasons

def run_rescue_round(problem_text, route, base_plan, ranked, details, best_attempt_by_answer, deadline):
    if time.time() >= deadline - CFG.rescue_min_time_left:
        return [], {"skipped": True, "reason": "insufficient_time"}
    rescue_plan = build_rescue_plan(route, base_plan, ranked, details, best_attempt_by_answer)
    roles = rescue_plan.get("worker_roles", [])[: max(1, CFG.rescue_attempts)]
    temps = list(CFG.rescue_temperatures)
    note = build_rescue_instruction(route, ranked, details)
    base_index = CFG.attempts + 20
    rescue_results = []
    with ThreadPoolExecutor(max_workers=min(CFG.workers, len(roles))) as ex:
        futures = {}
        for i, role_name in enumerate(roles):
            futures[ex.submit(
                chat_role_attempt_with_tools,
                problem_text,
                base_index + i,
                route,
                role_name,
                rescue_plan,
                extra_user_note=note,
                temperature_override=temps[i % len(temps)],
            )] = role_name
        for fut in as_completed(futures):
            if time.time() > deadline:
                break
            role_name = futures[fut]
            try:
                result = fut.result(timeout=max(5.0, deadline - time.time()))
            except Exception as exc:
                result = AttemptResult(
                    None, 0.0, 1.0, -1, 0, 0, "",
                    f"rescue_future:{type(exc).__name__}",
                    route=route[0] if isinstance(route, (list, tuple)) and route else route,
                    method=role_name,
                )
            rescue_results.append(result)
    meta = {
        "roles": roles,
        "note": note,
        "attempts": [summarize_attempt_result(r) for r in rescue_results],
    }
    return rescue_results, meta

LAST_SOLVE_DETAILS = {}

def solve_problem(problem_text, problem_id=None):
    global PROBLEMS_REMAINING, LAST_SOLVE_DETAILS
    started = time.time()
    elapsed = time.time() - NOTEBOOK_START
    time_left = max(0.0, CFG.notebook_budget_seconds - elapsed)

    if time_left < CFG.hard_deadline_reserve_seconds:
        PROBLEMS_REMAINING = max(0, PROBLEMS_REMAINING - 1)
        answer = coerce_submission_int(
            fallback_submission_answer(
                problem_text, problem_id=problem_id,
                route="deadline", reason="hard_deadline_reserve",
            )
        )
        LAST_SOLVE_DETAILS = {"route": "deadline", "problem_route": "deadline", "answer": answer, "failure_class": "deadline"}
        return answer

    problem_routes, route_scores = classify_problem_routes(problem_text)
    problem_route = problem_routes[0]
    route_context = tuple(problem_routes[:2]) if problem_routes else (problem_route,)
    force_full = problem_requires_full_attempts(problem_text, route=problem_route)
    deadline = time.time() + max(60.0, time_left / max(1, PROBLEMS_REMAINING))
    strategy_hint = domain_strategy_hint(problem_text)

    plan = plan_problem(problem_text, route_context)
    worker_schedule = build_worker_schedule(plan, route_context)

    write_debug_event({
        "event": "problem_start_v4",
        "problem_id": problem_id,
        "route": "chat_plan_workers_v4",
        "problem_route": problem_route,
        "problem_routes": list(route_context),
        "route_scores": dict(route_scores),
        "model_path": str(MODEL_PATH),
        "request_model": REQUEST_MODEL_NAME,
        "model_routing": summarize_model_routing(),
        "force_full": force_full,
        "time_left_s": round(time_left, 3),
        "strategy_hint": strategy_hint,
        "plan": {k: v for k, v in plan.items() if not str(k).startswith("_")},
        "worker_schedule": worker_schedule,
        "problem_excerpt": clip_text(problem_text, 500),
    })

    results = []
    futures = {}
    with ThreadPoolExecutor(max_workers=min(CFG.workers, max(1, CFG.attempts))) as ex:
        for i, role_name in enumerate(worker_schedule):
            futures[ex.submit(
                chat_role_attempt_with_tools,
                problem_text, i, route_context, role_name, plan,
            )] = (i, role_name)

        for fut in as_completed(futures):
            if time.time() > deadline:
                break
            idx, role_name = futures[fut]
            try:
                r = fut.result(timeout=max(5.0, deadline - time.time()))
            except Exception as e:
                r = AttemptResult(
                    None, 0.0, 1.0, idx, 0, 0, "",
                    f"future:{type(e).__name__}",
                    route=problem_route, method=role_name,
                )
            results.append(r)

            counts = Counter(x.answer for x in results if x.answer is not None)
            if (not force_full) and counts and counts.most_common(1)[0][1] >= CFG.early_stop_votes:
                break

    ranked, details, best_attempt_by_answer = aggregate_candidate_stats(results)
    valid = [r for r in results if r.answer is not None]

    if not valid and time.time() < deadline - 20:
        write_debug_event({
            "event": "fallback_to_chat_v4",
            "problem_id": problem_id,
            "reason": "no_valid_role_worker_answers",
            "attempts": [summarize_attempt_result(r) for r in results],
        })

        chat_results = []
        with ThreadPoolExecutor(max_workers=min(CFG.workers, max(1, CFG.attempts))) as ex:
            chat_futures = {
                ex.submit(chat_attempt, problem_text, i, problem_route): i
                for i in range(CFG.attempts)
            }
            for fut in as_completed(chat_futures):
                if time.time() > deadline:
                    break
                try:
                    r = fut.result(timeout=max(5.0, deadline - time.time()))
                except Exception as exc:
                    r = AttemptResult(
                        None, 0.0, 1.0, chat_futures[fut], 0, 0, "",
                        f"chat_future:{type(exc).__name__}",
                    )
                chat_results.append(r)

        results.extend(chat_results)
        ranked, details, best_attempt_by_answer = aggregate_candidate_stats(results)

    rescue_round = None
    rescue_needed, rescue_reasons = should_run_rescue_round(route_context, ranked, details, len(valid))
    if rescue_needed:
        rescue_round = {
            "scores_before": dict(details.get("scores") or {}),
            "counts_before": dict(details.get("counts") or {}),
        }
        rescue_results, rescue_meta = run_rescue_round(
            problem_text, route_context, plan, ranked, details, best_attempt_by_answer, deadline,
        )
        rescue_round.update(rescue_meta or {})
        if rescue_round is None:
            rescue_round = {}
        rescue_round["reasons"] = rescue_reasons
        if rescue_results:
            results.extend(rescue_results)
            ranked, details, best_attempt_by_answer = aggregate_candidate_stats(results)
            valid = [r for r in results if r.answer is not None]
            write_debug_event({
                "event": "rescue_round_v4",
                "problem_id": problem_id,
                "problem_route": problem_route,
                "reasons": rescue_reasons,
                "details_before": {
                    "scores": rescue_round.get("scores_before") or {},
                    "counts": rescue_round.get("counts_before") or {},
                },
                "rescue_meta": rescue_round,
            })

    answer = choose_weighted_winner(ranked, details, best_attempt_by_answer)

    verifier_result = None
    need_verifier = (
        should_run_verifier(problem_route, ranked, details)
        or (answer is None and len(ranked) >= 2)
    )
    if need_verifier and time.time() < deadline - 20:
        verifier_result = run_candidate_verifier_v2(
            problem_text, problem_route, strategy_hint, plan,
            ranked, details, best_attempt_by_answer,
        )
        if verifier_result and verifier_result.get("answer") is not None:
            answer = int(verifier_result["answer"])

    winner_attempt = best_attempt_by_answer.get(answer) if answer is not None else None
    if (
        answer is not None
        and winner_attempt is not None
        and (_route_set(problem_route) & FORCE_PYTHON_ROUTES)
        and winner_attempt.python_calls == 0
        and winner_attempt.certificate_type == "none"
        and len(ranked) >= 2
        and not bool((details.get("deterministic") or {}).get("locked"))
        and time.time() < deadline - 10
    ):
        verifier_result = verifier_result or run_candidate_verifier_v2(
            problem_text, problem_route, strategy_hint, plan,
            ranked, details, best_attempt_by_answer,
        )
        if verifier_result and verifier_result.get("answer") is not None:
            answer = int(verifier_result["answer"])

    PROBLEMS_REMAINING = max(0, PROBLEMS_REMAINING - 1)

    if verifier_result is not None:
        details["verifier"] = {
            "candidates": verifier_result.get("candidates"),
            "answer": verifier_result.get("answer"),
            "error": verifier_result.get("error"),
            "text": clip_text(verifier_result.get("text", ""), 1200),
        }
    if "rescue_round" in locals() and rescue_round is not None:
        details["rescue_round"] = rescue_round

    failure_class = None
    if answer is None:
        failure_class = "unresolved_v4"
        answer = fallback_submission_answer(
            problem_text, problem_id=problem_id,
            route=problem_route, reason=failure_class,
        )

    elapsed_s = time.time() - started
    valid_count = sum(1 for r in results if r.answer is not None)

    print(
        f"  -> answer={answer} | route=chat_plan_workers_v4 "
        f"| problem_route={problem_route} "
        f"| route_context={list(route_context)} "
        f"| workers={worker_schedule} "
        f"| scores={details.get('scores')} "
        f"| counts={details.get('counts')} "
        f"| tools={details.get('tools')} "
        f"| attempts={valid_count}/{len(results)} "
        f"| remaining={PROBLEMS_REMAINING} "
        f"| elapsed={elapsed_s:.1f}s"
    )

    LAST_SOLVE_DETAILS = {
        "route": "chat_plan_workers_v4",
        "problem_route": problem_route,
        "problem_routes": list(route_context),
        "answer": answer,
        "details": details,
        "worker_schedule": worker_schedule,
        "failure_class": failure_class,
        "valid_count": valid_count,
        "elapsed_s": round(elapsed_s, 3),
    }

    write_debug_event({
        "event": "problem_solved_v4",
        "problem_id": problem_id,
        "route": "chat_plan_workers_v4",
        "problem_route": problem_route,
        "problem_routes": list(route_context),
        "answer": answer,
        "details": details,
        "plan": {k: v for k, v in plan.items() if not str(k).startswith("_")},
        "worker_schedule": worker_schedule,
        "attempts": [summarize_attempt_result(r) for r in results],
        "valid_count": valid_count,
        "elapsed_s": round(elapsed_s, 3),
        "strategy_hint": strategy_hint,
        "problem_excerpt": clip_text(problem_text, 500),
    })

    return coerce_submission_int(answer)

solve_problem_v4_canonical = solve_problem

def run_single_experiment(problem_text=None, problem_id=None, csv_path=None, row_index=0, reset_budget=True):
    ensure_inference_ready()
    payload = None
    if problem_text is None:
        payload = load_problem_for_experiment(problem_id=problem_id, csv_path=csv_path, row_index=row_index)
        problem_text = payload["problem_text"]
        problem_id = payload["problem_id"]
    if reset_budget:
        reset_notebook_budget(1)
    solver_fn = globals().get("solve_problem_v4_canonical") or solve_problem
    answer = solver_fn(str(problem_text), problem_id=str(problem_id) if problem_id is not None else "adhoc")
    result = {
        "problem_id": None if problem_id is None else str(problem_id),
        "answer": int(answer),
        "expected_answer": None if payload is None else payload.get("expected_answer"),
        "route": LAST_SOLVE_DETAILS.get("route"),
        "problem_route": LAST_SOLVE_DETAILS.get("problem_route"),
        "problem_routes": LAST_SOLVE_DETAILS.get("problem_routes"),
        "failure_class": LAST_SOLVE_DETAILS.get("failure_class"),
        "worker_schedule": LAST_SOLVE_DETAILS.get("worker_schedule"),
        "details": LAST_SOLVE_DETAILS.get("details"),
        "elapsed_s": LAST_SOLVE_DETAILS.get("elapsed_s"),
    }
    print(result)
    return result

solve_problem = globals().get("solve_problem_v4_canonical", solve_problem)
print("Accuracy patch loaded (v4 canonical bound)")
